# V14ADfigures — the final figure set of paper3

Figure-only notebook. Every number is read from the saved outputs of the earlier notebooks, attached as inputs:
**V7ADmerged** (the merged benchmark), **ad-merged-geo-cohorts**, **V9ADexternal**, **V10ADgenevalidation**,
**V11ADrobustness** and **V15ADage**. Nothing is refitted.

The figure code is the paper's own (`scripts/merged_figures.py`, `validation_figures.py`, `age_figures.py`, the same files
`build.sh` runs), written out below and executed unchanged; each figure renders inline and is saved as vector PDF and
400-dpi PNG to `/kaggle/working/figures`.

Changes from V13 / V12: label-agnostic CVAE as the primary arm throughout; classification with per-split pairing and
per-cohort AUC as merged, age-corrected and age-matched (V15); latent geometry merged with the probability landscape,
learned on training arrays and judged on held-out arrays; paired separation effects; reconstruction and uncertainty with
exact medians and means; a gene-discovery atlas (permutation null, rank-vs-rank, fully named volcano with replication,
mirror chart); a sorted radar expression profile; a compact latent-to-gene map; the age-robustness figure; one darker
palette for the whole set.


In [ ]:
!pip install -q umap-learn 2>/dev/null; echo ready


In [ ]:
import os, glob
def find(marker, hint):
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True))
    hits = [h for h in hits if hint in h] or hits
    assert hits, f"attach the {hint} notebook output as an input"
    return os.path.dirname(hits[0])
V7 = find("summary_scalars.csv", "v7admerged")
for m, h in (("cohort_table.csv", "ad-merged-geo-cohorts"), ("loco_run_metrics.csv", "v9adexternal"),
             ("set_validation.csv", "v10adgenevalidation"), ("signature_beyond_composition.csv", "v11adrobustness"),
             ("comparison_per_cohort.csv", "v15adage")):
    print(h, "->", find(m, h))
os.environ.update({"PAPER_DATA": V7, "PAPER_FIGS": "/kaggle/working/figures", "PAPER_INLINE": "1",
                   "V12_INPUT": "/kaggle/input"})
os.makedirs("/kaggle/working/figures", exist_ok=True); os.makedirs("/kaggle/working/scripts", exist_ok=True)
print("V7 outputs:", V7)


## `scripts/merged_figures.py`


In [ ]:
SRC_MERGED_FIGURES = "\"\"\"Figures 1-11 of paper3, exported unchanged from the V13ADmergedfigures /\nV7ADmerged notebooks (paper2 figure code + the layout fixes for the merged\ngene lists). Reads data/ only.\"\"\"\n\n\"\"\"Shared visual language for every figure in the paper.\n\nOne module so that a panel letter, a tick, a bar gap or a significance bracket\nlooks the same wherever it appears -- the thing that separates a figure set from\na pile of plots. Everything here is layout and typography; the numbers come from\n../data and are never touched.\n\nColour is the data-viz reference palette used unchanged: categorical slots 1-3\n(blue / orange / aqua) for the three model arms, which is the documented\nall-pairs colourblind-safe subset, and the documented blue-red diverging pair\nwith a neutral grey midpoint for up/down regulation. Identity never rests on\ncolour alone -- every series also carries a marker shape, a direct label or a\nlegend entry.\n\"\"\"\nimport os\nimport warnings\nfrom pathlib import Path\n\nimport matplotlib as mpl\n# PAPER_INLINE is set when this same code runs inside the Kaggle notebook, where\n# the figures must render in the output rather than only be written to disk.\nINLINE = bool(os.environ.get(\"PAPER_INLINE\"))\nif not INLINE:\n    mpl.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom matplotlib.colors import LinearSegmentedColormap\nfrom matplotlib.lines import Line2D\n\nwarnings.filterwarnings(\"ignore\")\n\n# Paths are overridable so that the notebook can point them at /kaggle/working\n# and run this file unmodified -- one source of truth for every figure.\nif \"PAPER_DATA\" in os.environ:\n    DATA = Path(os.environ[\"PAPER_DATA\"])\n    FIG = Path(os.environ.get(\"PAPER_FIGS\", DATA / \"figures\"))\nelse:\n    ROOT = Path(__file__).resolve().parent.parent\n    DATA = ROOT / \"data\"\n    FIG = ROOT / \"figures\"\nFIG.mkdir(parents=True, exist_ok=True)\n\n# Target printed width. The LaTeX text block is 6.27 in wide, so a figure drawn\n# much wider than this is downscaled by \\includegraphics and every font in it\n# shrinks with the drawing -- the usual reason a figure set looks amateur on the\n# page. Everything here is laid out at TW and lands at ~0.9 scale.\nTW = 6.9\n\n# ---- palette -------------------------------------------------------------\n# A cool, low-chroma set in the register of a printed journal figure: navy,\n# wine and violet rather than the primary blue / orange / green of a screen\n# palette. The three model arms stay separable for the common forms of colour\n# blindness -- they differ in hue *and* in lightness (navy 30%, wine 42%,\n# violet 55%), so they also survive a greyscale print -- and identity never\n# rests on colour alone, since every series carries a marker or a direct label.\nPLS     = \"#173451\"   # categorical slot 1 -- navy\nCVAE_T  = \"#8f3a4f\"   # slot 2 -- wine\nCVAE_N  = \"#4f4478\"   # slot 3 -- muted violet\nAD      = \"#8f3a4f\"   # diverging warm pole -- wine\nCTRL    = \"#173451\"   # diverging cool pole -- navy\nUP, DOWN = AD, CTRL\nMID     = \"#8a8a92\"   # neutral midpoint ink, faintly cool\nACCENT  = \"#1f6372\"   # reserved for \"found by both\" -- petrol blue\nINK     = \"#17171c\"\nINK2    = \"#54545e\"\nINK3    = \"#8a8a92\"\nRULE    = \"#c9c9d1\"\nGRID    = \"#e9e9ef\"\nSURFACE = \"#ffffff\"\n\n# Diverging ramp for probability surfaces: cool pole -> neutral grey -> warm\n# pole. Deliberately light at both ends -- this ramp is only ever a background\n# under data marks, and a saturated pole swallows them.\nDIVERGING = LinearSegmentedColormap.from_list(\n    \"ctrl_ad\", [\"#7d92ad\", \"#a8b5c9\", \"#d3d9e3\", \"#eeeef1\",\n                \"#e3ccd3\", \"#c99aa8\", \"#a96d80\"])\n\n# Pale stepped bands of P(AD) for the latent-space maps (Figure 3): light enough\n# that held-out markers in the two pole colours stand out against them.\nfrom matplotlib.colors import ListedColormap, BoundaryNorm\nGEO_LEVELS = [0, 0.25, 0.5, 0.75, 1.0]\nGEO_BANDS = ListedColormap([\"#cdd6e3\", \"#e2e7ef\", \"#f3e7eb\", \"#e6cdd5\"])\nGEO_NORM = BoundaryNorm(GEO_LEVELS, 4)\n\n# Sequential ramps for the confusion matrices, built from the two arm colours so\n# the matrices sit inside the same muted palette as everything else instead of\n# borrowing matplotlib's saturated \"Blues\"/\"Oranges\".\nSEQ_PLS = LinearSegmentedColormap.from_list(\n    \"seq_pls\", [\"#ffffff\", \"#e7ecf2\", \"#c3d1e0\", \"#93aac6\", \"#5c7fa6\", PLS, \"#173252\"])\nSEQ_CVAE_N = LinearSegmentedColormap.from_list(\n    \"seq_cvae_n\", [\"#ffffff\", \"#efedf5\", \"#d9d4e8\", \"#b7aed2\", \"#9286b5\", CVAE_N, \"#463d66\"])\nSEQ_CVAE = LinearSegmentedColormap.from_list(\n    \"seq_cvae\", [\"#ffffff\", \"#f5e9ec\", \"#e6c9d1\", \"#d0a1ae\", \"#b06c81\", CVAE_T, \"#63242f\"])\n\nMODEL_COLOR = {\"pls\": PLS, \"cvae_true\": CVAE_T, \"cvae_neutral\": CVAE_N,\n               \"PLS-DA\": PLS, \"CVAE_true\": CVAE_T, \"CVAE_neutral\": CVAE_N}\nMODEL_LABEL = {\"pls\": \"PLS-DA\",\n               \"cvae_true\": \"CVAE (label-conditioned)\",\n               \"cvae_neutral\": \"CVAE (label-agnostic)\",\n               \"PLS-DA\": \"PLS-DA\",\n               \"CVAE_true\": \"CVAE (label-conditioned)\",\n               \"CVAE_neutral\": \"CVAE (label-agnostic)\"}\nGROUP_COLOR = {\"AD\": AD, \"Control\": CTRL}\nGROUP_MARKER = {\"AD\": \"^\", \"Control\": \"o\"}\n\n# database sources in the g:Profiler output\nSRC_COLOR = {\"GO:MF\": \"#24486b\", \"GO:BP\": \"#2f7d8c\", \"GO:CC\": \"#6f6396\",\n             \"CORUM\": \"#8f3a4f\", \"KEGG\": \"#4b7ca8\", \"REAC\": \"#57516e\",\n             \"WP\": \"#3f6d80\", \"MIRNA\": \"#a3798f\", \"TF\": \"#54545e\", \"HP\": \"#7a4a5e\"}\n\nplt.rcParams.update({\n    \"font.family\": \"serif\",\n    \"font.serif\": [\"STIXGeneral\", \"Latin Modern Roman\", \"DejaVu Serif\", \"Times New Roman\"],\n    \"mathtext.fontset\": \"stix\",\n    \"font.size\": 8,\n    \"axes.titlesize\": 9,\n    \"axes.titlelocation\": \"left\",\n    \"axes.titlepad\": 7,\n    \"axes.labelsize\": 8,\n    \"axes.labelpad\": 3.5,\n    \"axes.labelcolor\": INK,\n    \"axes.edgecolor\": RULE,\n    \"axes.linewidth\": 0.6,\n    \"axes.grid\": False,\n    \"grid.color\": GRID,\n    \"grid.linewidth\": 0.5,\n    \"axes.axisbelow\": True,\n    \"axes.spines.top\": False,\n    \"axes.spines.right\": False,\n    \"xtick.color\": INK2, \"ytick.color\": INK2,\n    \"xtick.labelsize\": 7, \"ytick.labelsize\": 7,\n    \"xtick.major.size\": 2.6, \"ytick.major.size\": 2.6,\n    \"xtick.major.width\": 0.6, \"ytick.major.width\": 0.6,\n    \"xtick.major.pad\": 2.5, \"ytick.major.pad\": 2.5,\n    \"legend.fontsize\": 7,\n    \"legend.frameon\": False,\n    \"legend.handlelength\": 1.1,\n    \"legend.handletextpad\": 0.5,\n    \"legend.columnspacing\": 1.2,\n    \"legend.borderaxespad\": 0.2,\n    \"lines.linewidth\": 1.4,\n    \"lines.solid_capstyle\": \"round\",\n    \"patch.linewidth\": 0,\n    \"figure.facecolor\": SURFACE,\n    \"axes.facecolor\": SURFACE,\n    \"savefig.facecolor\": SURFACE,\n    \"pdf.fonttype\": 42,\n})\n\n\n# ---- primitives ----------------------------------------------------------\ndef save(fig, name):\n    \"\"\"Vector PDF for pdflatex, 400-dpi PNG for the pandoc/docx build.\"\"\"\n    for ext, kw in ((\"pdf\", {}), (\"png\", {\"dpi\": 400})):\n        fig.savefig(FIG / f\"{name}.{ext}\", bbox_inches=\"tight\",\n                    pad_inches=0.03, **kw)\n    if INLINE:\n        try:\n            from IPython.display import display\n            display(fig)\n        except ImportError:      # running the notebook's cells outside a kernel\n            pass\n    plt.close(fig)\n    print(f\"  {FIG.name}/{name}.pdf + .png\")\n\n\ndef panel(ax, letter, title=None, subtitle=None, dx=-0.13, dy=1.0):\n    \"\"\"Panel letter in sans-serif, hung outside the axes at a fixed offset so\n    the letters line up down the figure regardless of tick-label width.\n\n    Title and subtitle are both annotations rather than `set_title`, because a\n    real title always sits at one fixed height and would land on top of the\n    subtitle whenever both are present.\n    \"\"\"\n    ax.annotate(letter, xy=(dx, dy), xycoords=\"axes fraction\",\n                xytext=(0, 2), textcoords=\"offset points\",\n                fontsize=9.5, fontweight=\"bold\", family=\"DejaVu Sans\",\n                color=INK, ha=\"left\", va=\"bottom\", annotation_clip=False)\n    if title:\n        ax.annotate(title, xy=(0, 1.0), xycoords=\"axes fraction\",\n                    xytext=(0, 13 if subtitle else 3), textcoords=\"offset points\",\n                    fontsize=8.6, color=INK, ha=\"left\", va=\"bottom\",\n                    annotation_clip=False)\n    if subtitle:\n        ax.annotate(subtitle, xy=(0, 1.0), xycoords=\"axes fraction\",\n                    xytext=(0, 3), textcoords=\"offset points\",\n                    fontsize=6.8, color=INK3, ha=\"left\", va=\"bottom\",\n                    annotation_clip=False)\n\n\ndef ygrid(ax):\n    ax.grid(axis=\"y\", visible=True)\n    ax.grid(axis=\"x\", visible=False)\n\n\ndef xgrid(ax):\n    ax.grid(axis=\"x\", visible=True)\n    ax.grid(axis=\"y\", visible=False)\n\n\ndef despine(ax, left=True, bottom=True):\n    ax.spines[\"left\"].set_visible(left)\n    ax.spines[\"bottom\"].set_visible(bottom)\n\n\ndef load(name):\n    return pd.read_csv(DATA / name)\n\n\ndef scal(df=None):\n    \"\"\"summary_scalars.csv -> dict, numeric where possible.\"\"\"\n    df = df if df is not None else load(\"summary_scalars.csv\")\n    out = {}\n    for _, r in df.iterrows():\n        try:\n            out[r[\"key\"]] = float(r[\"value\"])\n        except (TypeError, ValueError):\n            out[r[\"key\"]] = r[\"value\"]\n    return out\n\n\ndef stars(p):\n    return \"***\" if p < 1e-3 else \"**\" if p < 1e-2 else \"*\" if p < 0.05 else \"n.s.\"\n\n\ndef pmath(p, prefix=\"p\"):\n    \"\"\"`$p = ...$` with a real times-ten-to-the exponent, not `1e-05`.\"\"\"\n    if p >= 1e-3:\n        return f\"${prefix} = {p:.3f}$\"\n    if not p > 1e-300:          # V7: with thousands of pairs a p-value can underflow to 0\n        return f\"${prefix} < 10^{{-300}}$\"\n    e = int(np.floor(np.log10(p)))\n    return f\"${prefix} = {p/10**e:.1f}\\\\times 10^{{{e}}}$\"\n\n\ndef italic_ticks(ax, names, fontsize=6.0, axis=\"y\"):\n    \"\"\"Italic gene symbols as real text, so hyphens stay hyphens and digits do\n    not pick up math-italic spacing (`HLA-DQA2`, not `HLA - DQA2`).\"\"\"\n    ticks = np.arange(len(names))\n    if axis == \"y\":\n        ax.set_yticks(ticks)\n        ax.set_yticklabels(names, fontsize=fontsize, fontstyle=\"italic\")\n    else:\n        ax.set_xticks(ticks)\n        ax.set_xticklabels(names, fontsize=fontsize, fontstyle=\"italic\")\n\n\ndef bracket(ax, x0, x1, y, text, tick=None, fontsize=7.2, color=INK):\n    \"\"\"Significance bracket that never overlaps the marks it spans.\"\"\"\n    tick = tick if tick is not None else abs(y) * 0.02 + 1e-9\n    ax.plot([x0, x0, x1, x1], [y - tick, y, y, y - tick],\n            color=color, lw=0.7, clip_on=False, solid_capstyle=\"butt\")\n    ax.text((x0 + x1) / 2, y + tick * 0.5, text, ha=\"center\", va=\"bottom\",\n            fontsize=fontsize, color=color, clip_on=False)\n\n\ndef boxstrip(ax, values, pos, color, width=0.44, jitter=0.055, seed=0, size=4):\n    \"\"\"A box over its own points: the distribution and its summary together.\"\"\"\n    rs = np.random.RandomState(int(round(seed)))\n    ax.scatter(rs.normal(pos, jitter, len(values)), values, s=size, color=color,\n               alpha=0.28, linewidth=0, zorder=1, rasterized=True)\n    bp = ax.boxplot([values], positions=[pos], widths=width, showfliers=False,\n                    patch_artist=True, zorder=3, manage_ticks=False)\n    bp[\"boxes\"][0].set(facecolor=SURFACE, edgecolor=color, linewidth=1.2, alpha=0.95)\n    for part in (\"whiskers\", \"caps\", \"medians\"):\n        for a in bp[part]:\n            a.set(color=color, linewidth=1.2)\n    return bp\n\n\ndef swatches(labels_colors, marker=\"s\", size=5):\n    return [Line2D([], [], marker=marker, ls=\"\", color=c, markersize=size, label=l)\n            for l, c in labels_colors]\n\n\ndef repel_labels(ax, xs, ys, labels, fontsize=5.4, color=None, italic=True,\n                 clearance=1.6, leader_color=None):\n    \"\"\"Place a label for every point without letting any two labels touch.\n\n    A dependency-free replacement for adjustText: each label is tried at a ring\n    of candidate offsets, in decreasing order of preference, and the first\n    offset whose estimated box overlaps neither an already-placed label nor its\n    own point is taken. Everything is measured in typographic points, which is\n    exactly the unit `annotate` takes for `xytext`, so no round trip through\n    data coordinates is needed and the result is stable at any figure size.\n    \"\"\"\n    color = INK if color is None else color\n    leader_color = RULE if leader_color is None else leader_color\n    fig = ax.figure\n    ppd = fig.dpi / 72.0                     # points -> display pixels\n    pts = ax.transData.transform(np.column_stack([xs, ys]))\n\n    # candidate offsets: near ring first, then progressively further out\n    cands = []\n    for radius in (7, 11, 16, 22, 30, 40, 52, 66, 82, 100):   # V7: wider search\n        for ang in range(0, 360, 30):\n            a = np.deg2rad(ang)\n            cands.append((radius * np.cos(a), radius * np.sin(a)))\n\n    placed = []                              # display-space boxes already taken\n    leg = ax.get_legend()                    # a label under the key is unreadable\n    if leg is not None:\n        bb = leg.get_window_extent()\n        placed.append((bb.x0, bb.y0, bb.x1, bb.y1))\n    order = np.argsort(-np.asarray(ys))      # most significant genes get first pick\n\n    def overlaps(box):\n        x0, y0, x1, y1 = box\n        for p0, q0, p1, q1 in placed:\n            if x0 < p1 and p0 < x1 and y0 < q1 and q0 < y1:\n                return True\n        return False\n\n    axbb = ax.get_window_extent()            # V7: labels must stay inside the axes\n    inside = lambda b: (b[0] >= axbb.x0 and b[2] <= axbb.x1 and\n                        b[1] >= axbb.y0 and b[3] <= axbb.y1)\n    forced = 0\n    for i in order:\n        text = labels[i]\n        w = len(text) * fontsize * 0.56 * ppd + clearance * ppd\n        h = fontsize * 1.25 * ppd + clearance * ppd\n        px, py = pts[i]\n        boxes = []\n        for dx, dy in cands:\n            cx = px + dx * ppd + (w / 2 - clearance) * (1 if dx >= 0 else -1)\n            cy = py + dy * ppd\n            boxes.append(((cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2), dx, dy))\n        ok = [t for t in boxes if inside(t[0]) and not overlaps(t[0])]\n        if not ok:                           # never drop a gene: least-bad inside spot\n            forced += 1\n            ok = [t for t in boxes if inside(t[0])] or boxes\n        for box, dx, dy in ok[:1]:\n            if True:\n                placed.append(box)\n                ax.annotate(text, (xs[i], ys[i]), xytext=(dx, dy),\n                            textcoords=\"offset points\", fontsize=fontsize,\n                            color=color, fontstyle=\"italic\" if italic else \"normal\",\n                            ha=\"left\" if dx >= 0 else \"right\", va=\"center\",\n                            zorder=5, annotation_clip=False,\n                            arrowprops=dict(arrowstyle=\"-\", color=leader_color,\n                                            lw=0.35, shrinkA=0.5, shrinkB=1.5))\n    if forced:\n        print(f\"  repel_labels: {forced} label(s) had no collision-free spot\")\n\n\ndef auc_rank(y, s):\n    \"\"\"ROC-AUC from ranks (Mann-Whitney U); NaN when a group is absent.\"\"\"\n    y = np.asarray(y).astype(bool); s = np.asarray(s, float)\n    n1, n0 = y.sum(), (~y).sum()\n    if n1 == 0 or n0 == 0:\n        return np.nan\n    from scipy.stats import rankdata\n    r = rankdata(s)\n    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)\n\n\ndef column_labels(ax, xs, ys, labels, colors, x_text, y_lo, y_hi, side, fontsize=5.0, spacing=None):\n    \"\"\"Name every point from a column at x_text, labels spread evenly between\n    y_lo and y_hi in height order, each tied to its point by a thin leader. No\n    two labels can touch whatever the density of the points.\"\"\"\n    # each label starts level with its own point; overlapping neighbours are\n    # pushed apart by the minimum spacing only, so leaders stay short and flat\n    order = np.argsort(ys)\n    want = np.asarray(ys, float)[order]\n    gap = (y_hi - y_lo) / max(len(order), 1) * 0.98 if spacing is None else spacing\n    slots = np.maximum(want, y_lo)\n    for k in range(1, len(slots)):\n        slots[k] = max(slots[k], slots[k - 1] + gap)\n    if len(slots) and slots[-1] > y_hi:\n        slots[-1] = y_hi\n        for k in range(len(slots) - 2, -1, -1):\n            slots[k] = min(slots[k], slots[k + 1] - gap)\n    for yt, i in zip(slots, order):\n        ax.annotate(labels[i], xy=(xs[i], ys[i]), xytext=(x_text, yt), textcoords=\"data\",\n                    fontsize=fontsize, fontstyle=\"italic\", color=colors[i],\n                    ha=\"left\" if side == \"right\" else \"right\", va=\"center\", annotation_clip=False,\n                    arrowprops=dict(arrowstyle=\"-\", color=tint(colors[i], 0.45), lw=0.35,\n                                    shrinkA=1.0, shrinkB=1.5))\n\n\ndef kde2d(xy, grid_x, grid_y, bw_scale=1.0):\n    \"\"\"Gaussian KDE evaluated on a grid, without a scipy import at call sites.\"\"\"\n    from scipy.stats import gaussian_kde\n    k = gaussian_kde(xy.T)\n    k.set_bandwidth(k.factor * bw_scale)\n    XX, YY = np.meshgrid(grid_x, grid_y)\n    return k(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)\n\nimport numpy as np\nimport pandas as pd\nfrom matplotlib.lines import Line2D\nfrom matplotlib.patches import Circle, Patch\n\n# the two objects every figure function draws from\nS = scal()\nruns = load(\"run_metrics.csv\")\nprint(f\"{len(S)} scalars, {len(runs)} runs\")\n\n\"\"\"Figure 1 -- the study design, drawn rather than measured.\n\nThe only figure in the paper that carries no data: it exists so the reader can\nsee, before any result, that the two models are matched on the thing that\nmatters (an 8-dimensional bottleneck fitted on identical splits) and differ only\nin how they get there.\n\"\"\"\nimport numpy as np\nfrom matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle\n\n\nBOX_KW = dict(boxstyle=\"round,pad=0.30,rounding_size=0.09\", linewidth=0.9)\n\n\ndef box(ax, x, y, w, h, text, face, edge, fontsize=7.2, color=INK, weight=\"normal\"):\n    ax.add_patch(FancyBboxPatch((x, y), w, h, facecolor=face, edgecolor=edge, **BOX_KW))\n    ax.text(x + w / 2, y + h / 2, text, ha=\"center\", va=\"center\",\n            fontsize=fontsize, color=color, fontweight=weight, linespacing=1.35)\n\n\ndef arrow(ax, xy0, xy1, color=INK2, lw=0.9, style=\"-|>\", rad=0.0, ls=\"-\"):\n    ax.add_patch(FancyArrowPatch(xy0, xy1, arrowstyle=style, mutation_scale=8,\n                                 color=color, lw=lw, linestyle=ls,\n                                 shrinkA=1.5, shrinkB=1.5,\n                                 connectionstyle=f\"arc3,rad={rad}\"))\n\n\ndef tint(hex_color, alpha):\n    \"\"\"Flat tint over the white surface -- keeps the PDF free of transparency.\"\"\"\n    r, g, b = (int(hex_color[i:i + 2], 16) / 255 for i in (1, 3, 5))\n    return tuple(1 - alpha * (1 - c) for c in (r, g, b))\n\n\ndef fig_design():\n    coh = load(\"cohort_summary.csv\").iloc[0]\n    fig = plt.figure(figsize=(13.6, 7.2))\n    gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 2.9], hspace=0.10)\n\n    # ---------------- (a) cohort and resampling protocol -----------------\n    ax = fig.add_subplot(gs[0])\n    ax.set_xlim(0, 100); ax.set_ylim(0, 23); ax.axis(\"off\")\n    panel(ax, \"a\", dx=-0.012, dy=1.02)\n    ax.text(0, 21.6, \"Cohort, preprocessing and resampling protocol\",\n            fontsize=8.8, color=INK, ha=\"left\", va=\"bottom\")\n\n    box(ax, 0.5, 8, 15, 11,\n        f\"{int(coh.n_cohorts)} GEO cohorts\\n{int(coh.n_subjects)} arrays\\n{int(coh.n_donors)} donors, age $\\\\geq${int(coh.age_min)}\",\n        tint(MID, 0.22), INK3)\n    box(ax, 19, 8, 16, 11,\n        f\"Probe $\\\\rightarrow$ symbol\\n(NCBI Gene, all cohorts)\\n{int(coh.n_genes):,} genes\",\n        tint(MID, 0.22), INK3)\n    box(ax, 38.5, 8, 17, 11,\n        f\"Donor-grouped split\\n60 / 20 / 20\\ntrain / val / test\",\n        tint(MID, 0.22), INK3)\n    arrow(ax, (15.5, 13.5), (19, 13.5)); arrow(ax, (35, 13.5), (38.5, 13.5))\n\n    # the two resampling arms\n    box(ax, 60, 14, 18, 8, f\"$\\\\times\\\\,20$ real-label splits\\nrank statistics\",\n        tint(PLS, 0.30), PLS)\n    box(ax, 60, 3.5, 18, 8, \"$\\\\times\\\\,10$ label-permuted splits\\nempirical null\",\n        tint(MID, 0.30), INK3)\n    arrow(ax, (55.5, 14.5), (60, 18), rad=0.15)\n    arrow(ax, (55.5, 12.5), (60, 7.5), rad=-0.15)\n\n    box(ax, 82, 8, 17, 11,\n        \"One-sided Mann\u2013Whitney\\n$\\\\rightarrow$ BH FDR $q<0.05$\\n$+$ stability selection\",\n        CVAE_T, CVAE_T, color=\"white\", weight=\"bold\")\n    arrow(ax, (78, 18), (82, 14.6), rad=-0.15)\n    arrow(ax, (78, 7.5), (82, 12.4), rad=0.15)\n\n    ax.text(27, 5.4, \"z-scored with training-fold parameters only\",\n            fontsize=6.6, color=INK3, ha=\"center\", style=\"italic\")\n\n    # ---------------- (b) the two matched architectures ------------------\n    ax = fig.add_subplot(gs[1])\n    ax.set_xlim(0, 100); ax.set_ylim(-4.0, 57); ax.axis(\"off\")\n    panel(ax, \"b\", dx=-0.012, dy=1.01)\n    ax.text(0, 55.2, \"Two representations of the same 8-dimensional bottleneck\",\n            fontsize=8.8, color=INK, ha=\"left\", va=\"bottom\")\n\n    # ---- the two shared inputs, drawn once on the left -------------------\n    box(ax, 0.5, 21, 11.5, 10,\n        f\"$\\\\mathbf{{X}}$\\n$n\\\\times{int(coh.n_genes):,}$\\nexpression\",\n        tint(MID, 0.22), INK3, fontsize=7.0)\n    box(ax, 0.5, 35, 11.5, 7.5, \"$\\\\mathbf{Y}$\\none-hot labels\",\n        tint(CVAE_N, 0.30), RULE, fontsize=7.0)\n    ax.text(6.25, 19.6, \"same split for both lanes\", fontsize=6.2, color=INK3,\n            ha=\"center\", va=\"top\", style=\"italic\")\n\n    # ================= PLS-DA lane =======================================\n    lane = Rectangle((14.0, 29.5), 84.5, 25.0, facecolor=tint(PLS, 0.10),\n                     edgecolor=\"none\", zorder=0)\n    ax.add_patch(lane)\n    ax.text(15.0, 53.4, \"PLS-DA  \u00b7  linear, supervised projection\",\n            fontsize=7.6, color=PLS, ha=\"left\", va=\"top\", fontweight=\"bold\")\n\n    box(ax, 15.5, 38.0, 14.5, 9.0,\n        \"NIPALS\\n$\\\\mathbf{T}=\\\\mathbf{XW}$\\n8 components\", tint(PLS, 0.30), PLS)\n    box(ax, 33.5, 38.0, 12.5, 9.0, \"Scores\\n$\\\\mathbf{T}\\\\in\\\\mathbb{R}^{n\\\\times 8}$\",\n        PLS, PLS, fontsize=7.3, weight=\"bold\", color=\"white\")\n    box(ax, 50.5, 44.6, 15.5, 6.0, \"Logistic regression\", tint(PLS, 0.30), PLS,\n        fontsize=6.9)\n    box(ax, 50.5, 36.8, 15.5, 6.4, \"Loadings $\\\\mathbf{P}$\\n$\\\\mathbf{T}\\\\mathbf{P}^{\\\\top}$\",\n        tint(PLS, 0.30), PLS, fontsize=6.9)\n    box(ax, 70.0, 44.6, 14.0, 6.0, \"$\\\\hat{y}$  prediction\", tint(MID, 0.22), INK3,\n        fontsize=6.9)\n    box(ax, 70.0, 36.8, 14.0, 6.4, \"$\\\\hat{\\\\mathbf{X}}$  reconstruction\",\n        tint(MID, 0.22), INK3, fontsize=6.9)\n    box(ax, 33.5, 30.2, 12.5, 5.4, \"x-weights $\\\\mathbf{W}$\", tint(PLS, 0.22), PLS,\n        fontsize=6.7)\n    box(ax, 50.5, 30.2, 15.5, 5.4, \"VIP score per gene\", tint(PLS, 0.42), PLS,\n        fontsize=6.9)\n\n    arrow(ax, (12.0, 28.0), (15.5, 41.0), color=PLS, rad=0.14)\n    arrow(ax, (12.0, 38.5), (15.5, 43.5), color=CVAE_N, rad=-0.10)\n    arrow(ax, (30.0, 42.5), (33.5, 42.5), color=PLS)\n    arrow(ax, (46.0, 43.6), (50.5, 47.6), color=PLS, rad=0.10)\n    arrow(ax, (46.0, 41.5), (50.5, 40.0), color=PLS, rad=-0.10)\n    arrow(ax, (66.0, 47.6), (70.0, 47.6), color=PLS)\n    arrow(ax, (66.0, 40.0), (70.0, 40.0), color=PLS)\n    arrow(ax, (39.7, 38.0), (39.7, 35.6), color=PLS, ls=(0, (2.4, 1.6)))\n    arrow(ax, (46.0, 32.9), (50.5, 32.9), color=PLS)\n\n    # ================= beta-CVAE lane ====================================\n    lane = Rectangle((14.0, 1.0), 84.5, 25.0, facecolor=tint(CVAE_T, 0.10),\n                     edgecolor=\"none\", zorder=0)\n    ax.add_patch(lane)\n    ax.text(15.0, 24.9, \"$\\\\beta$-CVAE  \u00b7  non-linear, semi-supervised generative model\",\n            fontsize=7.6, color=CVAE_T, ha=\"left\", va=\"top\", fontweight=\"bold\")\n\n    box(ax, 15.5, 11.6, 14.0, 9.0,\n        \"Encoder $q_\\\\phi(\\\\mathbf{z}\\\\mid\\\\mathbf{x},\\\\mathbf{c})$\\n\"\n        \"$256\\\\rightarrow$ BN, ReLU\\ndropout 0.2\", tint(CVAE_T, 0.30), CVAE_T,\n        fontsize=6.8)\n    box(ax, 32.5, 16.4, 9.5, 4.6, \"$\\\\mu$\", tint(CVAE_T, 0.30), CVAE_T, fontsize=7.2)\n    box(ax, 32.5, 10.6, 9.5, 4.6, \"$\\\\log\\\\sigma^{2}$\", tint(CVAE_T, 0.30), CVAE_T,\n        fontsize=7.2)\n    box(ax, 45.5, 12.6, 13.5, 6.8,\n        \"$\\\\mathbf{z}=\\\\mu+\\\\sigma\\\\odot\\\\epsilon$\\n$\\\\in\\\\mathbb{R}^{n\\\\times 8}$\",\n        CVAE_T, CVAE_T, fontsize=7.2, weight=\"bold\", color=\"white\")\n    box(ax, 45.5, 5.4, 13.5, 4.6, \"$\\\\epsilon\\\\sim\\\\mathcal{N}(0,\\\\mathbf{I})$\",\n        tint(MID, 0.22), INK3, fontsize=6.7)\n    box(ax, 62.5, 17.4, 14.0, 6.2, \"Classifier head\\nFC-ReLU-FC\", tint(CVAE_T, 0.30),\n        CVAE_T, fontsize=6.7)\n    box(ax, 62.5, 9.6, 14.0, 6.2, \"Decoder $p_\\\\theta(\\\\mathbf{x}\\\\mid\\\\mathbf{z},\\\\mathbf{c})$\",\n        tint(CVAE_T, 0.30), CVAE_T, fontsize=6.7)\n    box(ax, 80.5, 17.4, 14.5, 6.2, \"$\\\\hat{y}$  prediction\", tint(MID, 0.22), INK3,\n        fontsize=6.9)\n    box(ax, 80.5, 9.6, 14.5, 6.2, \"$\\\\hat{\\\\mathbf{X}}$  reconstruction\",\n        tint(MID, 0.22), INK3, fontsize=6.9)\n    box(ax, 80.5, 2.0, 14.5, 5.6, \"Integrated Gradients\\nper gene\",\n        tint(CVAE_T, 0.42), CVAE_T, fontsize=6.7)\n\n    arrow(ax, (6.25, 21.0), (15.5, 17.5), color=CVAE_T, rad=-0.10)\n    arrow(ax, (29.5, 17.5), (32.5, 18.7), color=CVAE_T)\n    arrow(ax, (29.5, 14.5), (32.5, 12.9), color=CVAE_T)\n    arrow(ax, (42.0, 18.7), (45.5, 17.2), color=CVAE_T)\n    arrow(ax, (42.0, 12.9), (45.5, 14.6), color=CVAE_T)\n    arrow(ax, (52.2, 10.0), (52.2, 12.6), color=INK3, ls=(0, (2.4, 1.6)))\n    arrow(ax, (59.0, 17.4), (62.5, 20.0), color=CVAE_T, rad=0.10)\n    arrow(ax, (59.0, 14.6), (62.5, 12.7), color=CVAE_T, rad=-0.10)\n    arrow(ax, (76.5, 20.5), (80.5, 20.5), color=CVAE_T)\n    arrow(ax, (76.5, 12.7), (80.5, 12.7), color=CVAE_T)\n    # the attribution route runs outside the output boxes, down the lane edge\n    ax.plot([95.0, 97.2, 97.2], [20.5, 20.5, 4.8], color=CVAE_T, lw=0.9, ls=(0, (2.4, 1.6)))\n    arrow(ax, (97.2, 4.8), (95.0, 4.8), color=CVAE_T)\n\n    # the conditioning vector: the one input the two lanes do not share\n    box(ax, 15.5, 3.4, 14.0, 5.4, \"$\\\\mathbf{c}$  conditioning\", tint(CVAE_N, 0.30),\n        CVAE_N, fontsize=6.8)\n    arrow(ax, (22.5, 8.8), (22.5, 11.6), color=CVAE_N, ls=(0, (2.4, 1.6)))\n    # the conditioning route to the decoder runs beneath the epsilon box\n    arrow(ax, (29.5, 4.4), (69.5, 4.4), color=CVAE_N, ls=(0, (2.4, 1.6)), style=\"-\")\n    arrow(ax, (69.5, 4.4), (69.5, 9.6), color=CVAE_N, ls=(0, (2.4, 1.6)))\n    ax.text(31.0, 2.6, \"true one-hot label  |  neutral $1/K$\",\n            fontsize=6.2, color=CVAE_N, ha=\"left\", va=\"center\")\n\n    # the matched constraint, drawn once between the two bottleneck boxes\n    ax.text(15.5, 27.9, \"matched: $d=8$, identical folds\",\n            fontsize=6.8, color=INK2, ha=\"left\", va=\"center\", style=\"italic\")\n\n    # One rule, stated once: grey is data entering or leaving a lane, violet is\n    # label information, a lane-coloured box is an operation of that lane, and\n    # the one box per lane with a heavier fill is the matched bottleneck. Every\n    # box in the drawing follows it, which is what makes the key worth reading.\n    key = [(\"data in / out\", [(tint(MID, 0.22), INK3)]),\n           (\"label information\", [(tint(CVAE_N, 0.30), CVAE_N)]),\n           (\"operation\", [(tint(PLS, 0.30), PLS), (tint(CVAE_T, 0.30), CVAE_T)]),\n           (\"matched $d=8$ bottleneck\", [(PLS, PLS),\n                                         (CVAE_T, CVAE_T)])]\n    x0 = 15.0\n    for lab, sw in key:\n        for j, (fc, ec) in enumerate(sw):\n            ax.add_patch(Rectangle((x0 + j * 2.4, -0.9), 2.1, 1.6, facecolor=fc,\n                                   edgecolor=ec, linewidth=0.7, clip_on=False))\n        x0 += 2.4 * (len(sw) - 1) + 3.0\n        ax.text(x0, -0.1, lab, fontsize=6.4, color=INK2, ha=\"left\", va=\"center\")\n        x0 += len(lab) * 0.95 + 5.0\n    ax.text(15.0, -3.0, \"Lane colour marks which model an operation belongs to; \"\n            \"a heavier fill marks the bottleneck the two models share.\",\n            fontsize=6.2, color=INK3, ha=\"left\", va=\"center\", style=\"italic\")\n\n    save(fig, \"fig1_design\")\n\n\n\nfig_design()\n\n# =====================================================================\n# Figure 2 -- classification\n# =====================================================================\ndef fig_classification(runs, S):\n    roc, pr = load(\"roc_curves_raw.csv\"), load(\"pr_curves_raw.csv\")\n    cm = load(\"confusion_matrices.csv\").set_index(\"model\")\n    op = load(\"oof_predictions.csv\")\n    base = np.linspace(0, 1, 101)\n    arms = (\"pls\", \"cvae_neutral\", \"cvae_true\")\n    SHORT = {\"pls\": \"PLS-DA\", \"cvae_neutral\": \"CVAE agnostic\", \"cvae_true\": \"CVAE conditioned\"}\n\n    fig = plt.figure(figsize=(TW, 9.3))\n    gs = fig.add_gridspec(4, 3, hspace=0.86, wspace=0.42, height_ratios=[1.0, 1.10, 0.95, 0.72])\n    top = gs[0, :].subgridspec(1, 3, width_ratios=[1, 1, 1.05], wspace=0.48)\n\n    # ---- (a) ROC -----------------------------------------------------\n    ax = fig.add_subplot(top[0])\n    for m in arms:\n        v = roc[roc.model == m][[f\"tpr_{i}\" for i in range(101)]].values\n        mu, sd = v.mean(0), v.std(0)\n        key = \"auc_pls\" if m == \"pls\" else \"auc_cvae_lat_\" + m.split(\"_\", 1)[1]\n        ax.fill_between(base, np.clip(mu - sd, 0, 1), np.clip(mu + sd, 0, 1), color=MODEL_COLOR[m], alpha=0.12, lw=0)\n        ax.plot(base, mu, color=MODEL_COLOR[m], lw=1.3, ls=\"-\" if m != \"cvae_true\" else (0, (3, 1.5)),\n                label=f\"{SHORT[m]}  {S[key + '_mean']:.3f}\")\n    ax.plot([0, 1], [0, 1], color=INK3, lw=0.6, ls=(0, (3, 2)))\n    ax.set_xlabel(\"False positive rate\"); ax.set_ylabel(\"True positive rate\")\n    ax.set_xlim(0, 1); ax.set_ylim(0, 1.004); ax.set_xticks([0, .5, 1]); ax.set_yticks([0, .5, 1])\n    ax.legend(loc=\"lower right\", fontsize=5.6, title=\"mean AUC\", title_fontsize=5.8, alignment=\"left\",\n              frameon=True, framealpha=1.0, facecolor=SURFACE, edgecolor=\"none\", handlelength=1.4,\n              borderaxespad=0.1)\n    panel(ax, \"a\", \"ROC\", \"mean $\\\\pm$ SD, 20 splits\", dx=-0.30)\n\n    # ---- (b) precision-recall ---------------------------------------\n    ax = fig.add_subplot(top[1])\n    for m in arms:\n        v = pr[pr.model == m][[f\"prec_{i}\" for i in range(101)]].values\n        mu, sd = v.mean(0), v.std(0)\n        ax.fill_between(base, np.clip(mu - sd, 0, 1), np.clip(mu + sd, 0, 1), color=MODEL_COLOR[m], alpha=0.12, lw=0)\n        ax.plot(base, mu, color=MODEL_COLOR[m], lw=1.3, ls=\"-\" if m != \"cvae_true\" else (0, (3, 1.5)))\n    prev = S[\"n_AD\"] / S[\"n_subjects\"] if \"n_AD\" in S else (op.group == \"AD\").mean()\n    ax.axhline(prev, color=INK3, lw=0.6, ls=(0, (3, 2)))\n    ax.text(0.03, prev - 0.03, \"AD prevalence\", fontsize=5.8, color=INK3, va=\"top\")\n    ax.set_xlabel(\"Recall\"); ax.set_ylabel(\"Precision\")\n    ax.set_xlim(0, 1); ax.set_ylim(0, 1.004); ax.set_xticks([0, .5, 1]); ax.set_yticks([0, .5, 1])\n    panel(ax, \"b\", \"Precision-recall\", \"same splits\", dx=-0.30)\n\n    # ---- (c) slope chart over all three arms, baselines as bands ----------------\n    ax = fig.add_subplot(top[2])\n    A3 = [runs.auc_pls.values, runs.auc_cvae_lat_neutral.values, runs.auc_cvae_lat_true.values]\n    C3 = [PLS, CVAE_N, CVAE_T]\n    for col, lab in ((\"auc_age_only\", \"age alone\"), (\"auc_cohort_only\", \"cohort alone\")):\n        m_, s_ = runs[col].mean(), runs[col].std()\n        ax.axhline(m_, color=tint(MID, 0.8), lw=0.7, ls=(0, (1, 1.6)), zorder=1)\n        ax.text(2.62, m_ + 0.004, f\"{lab}  {m_:.2f} $\\\\pm$ {s_:.2f}\", ha=\"right\", va=\"bottom\", fontsize=5.6,\n                color=INK3)\n    for i in range(len(runs)):\n        up = A3[1][i] > A3[0][i]\n        ax.plot([0, 1], [A3[0][i], A3[1][i]], color=CVAE_N if up else PLS, lw=0.7, alpha=0.5 if up else 0.85,\n                zorder=2)\n        ax.plot([1, 2], [A3[1][i], A3[2][i]], color=CVAE_T, lw=0.55, alpha=0.55, zorder=2)\n    for x, v, c in zip((0, 1, 2), A3, C3):\n        m_, s_ = v.mean(), v.std()\n        ax.add_patch(Rectangle((x - 0.13, m_ - s_), 0.26, 2 * s_, facecolor=tint(c, 0.16), edgecolor=\"none\",\n                               zorder=1))\n        ax.scatter(np.full(len(v), x), v, s=6, color=c, lw=0, zorder=3)\n        ax.plot([x - 0.17, x + 0.17], [m_, m_], color=c, lw=2.0, solid_capstyle=\"butt\", zorder=4)\n        ax.text(x, 0.968, f\"{m_:.3f}\", ha=\"center\", va=\"bottom\", fontsize=6.8, color=c)\n        ax.text(x, 0.962, f\"$\\\\pm${s_:.3f}\", ha=\"center\", va=\"top\", fontsize=5.4, color=tint(c, 0.8))\n    wins = int((A3[1] > A3[0]).sum()); p_ = S[\"auc_cvae_neutral_vs_pls_p\"]\n    ax.set_xticks([0, 1, 2]); ax.set_xticklabels([\"PLS-DA\", \"CVAE\\nagnostic\", \"CVAE\\nconditioned\"], fontsize=6.2)\n    for t, c in zip(ax.get_xticklabels(), C3):\n        t.set_color(c)\n    ax.tick_params(axis=\"x\", length=0)\n    ax.set_xlim(-0.45, 2.65); ax.set_ylim(0.55, 1.0); ax.set_yticks([0.6, 0.7, 0.8, 0.9])\n    ax.set_ylabel(\"Test ROC-AUC\"); ax.spines[\"bottom\"].set_visible(False)\n    panel(ax, \"c\", \"Paired AUC across splits\",\n          f\"higher for CVAE agnostic in {wins}/20, paired {pmath(p_)}\", dx=-0.27)\n\n    # ---- (d) small multiples: one cohort per mini-panel, age removed left -> right ----\n    v15 = None\n    for f_ in [DATA / \"validation\" / \"v15adage\" / \"comparison_per_cohort.csv\"] + \\\n              [Path(h) for h in __import__(\"glob\").glob(\"/kaggle/input/**/comparison_per_cohort.csv\", recursive=True)]:\n        if Path(f_).exists():\n            v15 = pd.read_csv(f_); break\n    nd_ = op.drop_duplicates(\"donor\").groupby([\"cohort\", \"group\"]).size()\n    cohorts = sorted(op.cohort.unique(), key=lambda c: -op[op.cohort == c].donor.nunique())\n    ARMS_ = [\"V7, no age handling\", \"A, age-corrected\", \"B, age-matched\"]\n    LINES = [(\"auc_age_alone\", \"#a4a4ad\", \"D\", \"age alone\", (0, (3, 1.6))),\n             (\"auc_pls\", PLS, \"o\", \"PLS-DA\", \"-\"), (\"auc_cvae_agn\", CVAE_N, \"s\", \"CVAE agnostic\", \"-\")]\n    sub = gs[1, :].subgridspec(1, len(cohorts), wspace=0.10)\n    xs_ = np.array([0, 1, 2])\n    axd = []\n    for k, c in enumerate(cohorts):\n        ax = fig.add_subplot(sub[k], sharey=axd[0] if axd else None)\n        axd.append(ax)\n        ax.axhspan(0.35, 0.5, color=\"#f5f5f8\", lw=0, zorder=0)\n        ax.axhline(0.5, color=INK3, lw=0.5, ls=(0, (1, 2)), zorder=1)\n        rr = [v15[(v15.arm == a) & (v15.cohort == c)].iloc[0] for a in ARMS_]\n        for col, colr, mk, lab, ls in LINES:\n            v = np.array([r[col] for r in rr])\n            if col + \"_sd\" in v15.columns and col != \"auc_age_alone\":\n                s = np.array([r[col + \"_sd\"] for r in rr])\n                ax.fill_between(xs_, np.clip(v - s, 0.35, 1), np.clip(v + s, 0.35, 1), color=colr, alpha=0.13,\n                                lw=0, zorder=2)\n            ax.plot(xs_, v, color=colr, lw=1.5 if col != \"auc_age_alone\" else 1.1, ls=ls, zorder=3,\n                    solid_capstyle=\"round\")\n            ax.scatter(xs_, v, marker=mk, s=18 if mk != \"D\" else 14, color=colr, edgecolor=SURFACE,\n                       linewidth=0.6, zorder=4)\n        # the one number per line a reader needs: where it ends up once age is matched\n        ends = sorted([(rr[2][col], colr) for col, colr, *_ in LINES])\n        last = -1\n        for v, colr in ends:\n            yv = max(v, last + 0.035)\n            ax.text(2.16, yv, f\"{v:.2f}\", ha=\"left\", va=\"center\", fontsize=5.6, color=colr)\n            last = yv\n        ax.set_xlim(-0.25, 2.55); ax.set_ylim(0.38, 1.02)\n        ax.set_xticks(xs_); ax.set_xticklabels([\"merged\", \"age-\\ncorrected\", \"age-\\nmatched\"], fontsize=5.6,\n                                                 color=INK2)\n        ax.tick_params(axis=\"x\", length=0)\n        ax.grid(axis=\"y\", visible=True)\n        if k:\n            plt.setp(ax.get_yticklabels(), visible=False); ax.tick_params(axis=\"y\", length=0)\n            ax.spines[\"left\"].set_visible(False)\n        else:\n            ax.set_ylabel(\"ROC-AUC within cohort\"); ax.set_yticks([0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])\n        ax.text(0.0, 1.06, c, transform=ax.transAxes, ha=\"left\", va=\"bottom\", fontsize=6.9, color=INK)\n        ax.text(0.0, 1.0, f\"{nd_[c]['Control']} ctrl \u00b7 {nd_[c]['AD']} AD\", transform=ax.transAxes,\n                ha=\"left\", va=\"bottom\", fontsize=5.6, color=INK3)\n    axd[0].text(0.02, 0.405, \"chance\", fontsize=5.4, color=INK3, style=\"italic\")\n    hl = [Line2D([], [], color=PLS, lw=1.5, marker=\"o\", ms=4, label=\"PLS-DA\"),\n          Line2D([], [], color=CVAE_N, lw=1.5, marker=\"s\", ms=3.8, label=\"CVAE agnostic\"),\n          Line2D([], [], color=\"#a4a4ad\", lw=1.1, ls=(0, (3, 1.6)), marker=\"D\", ms=3.4, label=\"age alone\"),\n          Rectangle((0, 0), 1, 1, facecolor=tint(INK2, 0.15), edgecolor=\"none\", label=\"$\\\\pm$ SD over 20 splits\")]\n    axd[2].legend(handles=hl, loc=\"upper center\", bbox_to_anchor=(0.5, -0.22), ncol=4, fontsize=6.2,\n                  frameon=False, columnspacing=1.4)\n    v9_ = [DATA / \"validation\" / \"v9adexternal\" / \"loco_run_metrics.csv\"] + \\\n          [Path(h) for h in __import__(\"glob\").glob(\"/kaggle/input/**/loco_run_metrics.csv\", recursive=True)]\n    v9_ = [f for f in v9_ if Path(f).exists()]\n    if v9_ and \"GSE132903\" in cohorts:\n        r9 = pd.read_csv(v9_[0])\n        axk = axd[cohorts.index(\"GSE132903\")]\n        axk.text(1.0, 0.72, f\"cohort fully held out\\n(separate run):\\nPLS-DA {r9.auc_pls.mean():.3f}\\n\"\n                            f\"CVAE {r9.auc_cvae_neutral.mean():.3f}\", ha=\"center\", va=\"center\", fontsize=5.6,\n                 color=INK2, style=\"italic\", linespacing=1.35)\n    a0_ = axd[0]\n    a0_.annotate(\"d\", xy=(-0.42, 1.0), xycoords=\"axes fraction\", xytext=(0, 36), textcoords=\"offset points\",\n                 fontsize=9.5, fontweight=\"bold\", family=\"DejaVu Sans\", color=INK, ha=\"left\", va=\"bottom\",\n                 annotation_clip=False)\n    a0_.annotate(\"Removing age leaves the disease signal; only age alone falls to chance\", xy=(0, 1.0),\n                 xycoords=\"axes fraction\", xytext=(0, 34), textcoords=\"offset points\", fontsize=8.6, color=INK,\n                 ha=\"left\", va=\"bottom\", annotation_clip=False)\n    a0_.annotate(\"test AUC within each cohort: as merged, with ageing slopes removed, and with donors matched on \"\n                 \"age; donors unseen, cohorts seen in training\", xy=(0, 1.0), xycoords=\"axes fraction\",\n                 xytext=(0, 24), textcoords=\"offset points\", fontsize=6.8, color=INK3, ha=\"left\", va=\"bottom\",\n                 annotation_clip=False)\n\n    # ---- (e-g) confusion matrices, all three arms -------------------------\n    for i, m in enumerate(arms):\n        ax = fig.add_subplot(gs[2, i])\n        if m in cm.index:\n            r = cm.loc[m]\n            M = np.array([[r.tn, r.fp], [r.fn, r.tp]], float)\n        else:\n            yy = (op.group == \"AD\").values; pp = (op.p_cvae_lat_neutral >= 0.5).values\n            M = np.array([[(~yy & ~pp).sum(), (~yy & pp).sum()], [(yy & ~pp).sum(), (yy & pp).sum()]], float)\n        Mn = M / M.sum(1, keepdims=True)\n        ax.imshow(Mn, cmap={\"pls\": SEQ_PLS, \"cvae_neutral\": SEQ_CVAE_N, \"cvae_true\": SEQ_CVAE}[m], vmin=0, vmax=1)\n        for u in range(2):\n            for v in range(2):\n                dark = Mn[u, v] > 0.55\n                ax.text(v, u - 0.06, f\"{Mn[u, v]:.2f}\", ha=\"center\", va=\"center\", fontsize=10,\n                        color=SURFACE if dark else INK)\n                ax.text(v, u + 0.25, f\"n = {int(M[u, v]):,}\", ha=\"center\", va=\"center\", fontsize=5.6,\n                        color=SURFACE if dark else INK2)\n        ax.set_xticks([0, 1]); ax.set_xticklabels([\"Control\", \"AD\"], fontsize=6.4)\n        ax.set_yticks([0, 1]); ax.set_yticklabels([\"Control\", \"AD\"] if i == 0 else [\"\", \"\"], fontsize=6.4)\n        ax.set_xlabel(\"Predicted\", fontsize=6.8); ax.set_ylabel(\"True\" if i == 0 else \"\", fontsize=6.8)\n        for sp in ax.spines.values():\n            sp.set_visible(False)\n        ax.tick_params(length=0)\n        sens, spec = Mn[1, 1], Mn[0, 0]\n        sub = f\"sens. {sens:.2f} \u00b7 spec. {spec:.2f}\"\n        if m == \"cvae_true\":\n            sub = \"reference: sees the test label\"\n        panel(ax, \"efg\"[i], SHORT[m], sub, dx=-0.30 if i == 0 else -0.12)\n        ax.set_position(ax.get_position().translated(0, -0.012))\n\n    # ---- (h) predicted probability by diagnosis ----------------------\n    ax = fig.add_subplot(gs[3, :])\n    grid = np.linspace(0, 1, 300)\n    from scipy.stats import gaussian_kde\n    for j, (col, ls) in enumerate(((\"p_pls\", \"-\"), (\"p_cvae_lat_neutral\", (0, (4, 2))))):\n        for g in (\"Control\", \"AD\"):\n            v = op.loc[op.group == g, col].values\n            d = gaussian_kde(v)(grid)\n            ax.plot(grid, d, color=GROUP_COLOR[g], lw=1.3, ls=ls)\n            if j == 0:\n                ax.fill_between(grid, 0, d, color=GROUP_COLOR[g], alpha=0.10, lw=0)\n    ax.axvline(0.5, color=INK3, lw=0.6, ls=(0, (3, 2)))\n    ax.set_xlim(0, 1); ax.set_xlabel(\"Predicted $P(\\\\mathrm{AD})$ on held-out arrays\")\n    ax.set_ylabel(\"Density\"); ygrid(ax)\n    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)\n    ax.legend(handles=[*swatches([(\"Control\", CTRL), (\"AD\", AD)]),\n                       Line2D([], [], color=INK2, ls=\"-\", label=\"PLS-DA\"),\n                       Line2D([], [], color=INK2, ls=(0, (4, 2)), label=\"CVAE agnostic\")],\n              loc=\"upper center\", ncol=4, fontsize=6.2, frameon=True, framealpha=1.0, facecolor=SURFACE,\n              edgecolor=\"none\")\n    panel(ax, \"h\", \"Predicted probability by diagnosis\",\n          f\"all {len(op):,} held-out array\u2013split pairs\", dx=-0.075)\n    save(fig, \"fig2_classification\")\n\n\nfig_classification(runs, S)\n\n# =====================================================================\n# Figure 3 -- latent geometry\n# =====================================================================\ndef fig_geometry():\n    from sklearn.metrics import silhouette_score\n    from scipy.stats import gaussian_kde\n    emb = load(\"latent_embeddings.csv\")\n    Zc = load(\"latent_coords_representative.csv\")\n    tags = [(\"PLS-DA\", \"PLS-DA\", PLS), (\"CVAE_neutral\", \"CVAE agnostic\", CVAE_N),\n            (\"CVAE_true\", \"CVAE conditioned\", CVAE_T)]\n    projs = [((\"pca1\", \"pca2\"), \"PCA\"), ((\"tsne1\", \"tsne2\"), \"t-SNE\")]\n    if \"umap1\" in emb.columns and emb.umap1.notna().any():\n        projs.append(((\"umap1\", \"umap2\"), \"UMAP\"))\n    zc = [f\"z{k}\" for k in range(1, 9)]\n    sil = {(t, sp): silhouette_score(Zc[(Zc.model == t) & (Zc.split == sp)][zc],\n                                     Zc[(Zc.model == t) & (Zc.split == sp)].group)\n           for t, _, _ in tags for sp in (\"train\", \"test\")}\n\n    nr = len(projs)\n    fig = plt.figure(figsize=(TW, TW / 3 * nr + 1.05))\n    gs = fig.add_gridspec(nr, 3, left=0.075, right=0.995, top=0.875, bottom=0.075, wspace=0.05, hspace=0.05)\n    letters = iter(\"abcdefghi\")\n    for r, (proj, pname) in enumerate(projs):\n        for c, (tag, tname, tcol) in enumerate(tags):\n            ax = fig.add_subplot(gs[r, c])\n            d = emb[emb.model == tag]\n            X = d[list(proj)].values\n            lo_, hi_ = np.percentile(X, [1, 99], axis=0); pad_ = 0.10 * (hi_ - lo_)\n            lo_, hi_ = lo_ - pad_, hi_ + pad_\n            tr = d[d.split == \"train\"]; te = d[d.split == \"test\"]\n            gx = np.linspace(lo_[0], hi_[0], 200); gy = np.linspace(lo_[1], hi_[1], 200)\n            GX, GY = np.meshgrid(gx, gy); G = np.vstack([GX.ravel(), GY.ravel()])\n            # landscape learned from TRAINING arrays: P(AD | position), equal priors\n            dens = {g: gaussian_kde(tr[tr.group == g][list(proj)].values.T, bw_method=\"scott\")(G).reshape(GX.shape)\n                    for g in (\"AD\", \"Control\")}\n            post = dens[\"AD\"] / (dens[\"AD\"] + dens[\"Control\"] + 1e-300)\n            # pale stepped bands, so that the held-out markers -- coloured by diagnosis in\n            # the bands' own two hues -- carry the eye and show the separation directly\n            im = ax.contourf(gx, gy, post, levels=GEO_LEVELS, cmap=GEO_BANDS, norm=GEO_NORM, zorder=0)\n            ax.contour(gx, gy, post, levels=[0.5], colors=[INK], linewidths=1.0, linestyles=[(0, (4, 2.2))],\n                       zorder=2)\n            ax.scatter(tr[proj[0]], tr[proj[1]], s=1.4, color=INK3, alpha=0.18, linewidth=0, zorder=1,\n                       rasterized=True)\n            # held-out arrays, judged against the training landscape\n            kt = {g: gaussian_kde(tr[tr.group == g][list(proj)].values.T, bw_method=\"scott\")\n                  for g in (\"AD\", \"Control\")}\n            pt = kt[\"AD\"](te[list(proj)].values.T); pc = kt[\"Control\"](te[list(proj)].values.T)\n            isad = (te.group == \"AD\").values\n            ok = (pt / (pt + pc + 1e-300) > 0.5) == isad\n            for g, kw in ((\"Control\", dict(marker=\"o\", color=CTRL, s=13, edgecolor=SURFACE, linewidth=0.45)),\n                          (\"AD\", dict(marker=\"^\", color=AD, s=15, edgecolor=SURFACE, linewidth=0.45))):\n                s = te[te.group == g]\n                ax.scatter(s[proj[0]], s[proj[1]], zorder=4, **kw)\n            ax.text(0.965, 0.035, f\"correct side  {ok.mean():.0%}\\ncontrols {ok[~isad].mean():.0%} \u00b7 \"\n                                  f\"AD {ok[isad].mean():.0%}\", transform=ax.transAxes, ha=\"right\",\n                    va=\"bottom\", fontsize=5.8, color=INK, linespacing=1.35,\n                    bbox=dict(boxstyle=\"round,pad=0.3\", facecolor=SURFACE, edgecolor=\"none\", alpha=0.9), zorder=6)\n            ax.set_xlim(lo_[0], hi_[0]); ax.set_ylim(lo_[1], hi_[1])\n            ax.set_xticks([]); ax.set_yticks([])\n            for sp in ax.spines.values():\n                sp.set_visible(True); sp.set_color(RULE); sp.set_linewidth(0.6)\n            ax.text(0.035, 0.965, next(letters), transform=ax.transAxes, ha=\"left\", va=\"top\", fontsize=9,\n                    fontweight=\"bold\", family=\"DejaVu Sans\", color=INK,\n                    bbox=dict(boxstyle=\"square,pad=0.15\", facecolor=SURFACE, edgecolor=\"none\", alpha=0.75))\n            if r == 0:\n                ax.text(0.5, 1.10, tname, transform=ax.transAxes, ha=\"center\", va=\"bottom\", fontsize=8.6,\n                        color=tcol)\n                ax.text(0.5, 1.035, f\"silhouette  held-out {sil[(tag, 'test')]:.2f}  \u00b7  training \"\n                                    f\"{sil[(tag, 'train')]:.2f}\", transform=ax.transAxes, ha=\"center\",\n                        va=\"bottom\", fontsize=6.3, color=INK2)\n            if c == 0:\n                ax.text(-0.06, 0.5, pname, transform=ax.transAxes, ha=\"right\", va=\"center\", rotation=90,\n                        fontsize=8.4, color=INK2)\n    hl = [Line2D([], [], marker=\"o\", ls=\"\", color=CTRL, markeredgecolor=SURFACE, markersize=5,\n                 label=\"control, held out\"),\n          Line2D([], [], marker=\"^\", ls=\"\", color=AD, markeredgecolor=SURFACE, markersize=5.4, label=\"AD, held out\"),\n          Line2D([], [], marker=\"o\", ls=\"\", color=INK3, alpha=0.4, markersize=2.5, label=\"training arrays\"),\n          Line2D([], [], color=INK, lw=1.0, ls=(0, (4, 2.2)), label=\"$P = 0.5$ boundary\")]\n    fig.legend(handles=hl, loc=\"lower left\", bbox_to_anchor=(0.06, 0.0), ncol=2, fontsize=7.0, frameon=False,\n               columnspacing=1.6)\n    cax = fig.add_axes([0.62, 0.022, 0.30, 0.011])\n    cb = fig.colorbar(im, cax=cax, orientation=\"horizontal\")\n    cb.set_ticks([0, 0.25, 0.5, 0.75, 1]); cb.ax.tick_params(labelsize=6.0, length=2); cb.outline.set_visible(False)\n    cb.set_label(\"$P(\\\\mathrm{AD} \\\\mid$ position$)$, learned from training arrays\", fontsize=6.4, color=INK2,\n                 labelpad=2)\n    fig.text(0.075, 0.975, \"Latent space of one representative split: the landscape learned in training, \"\n                           \"and where unseen donors land\", fontsize=8.8, color=INK, ha=\"left\", va=\"top\")\n    fig.text(0.075, 0.952, \"bands: P(AD) learned from training arrays, in steps of 0.25 \u00b7 markers: held-out arrays, \"\n                           \"coloured by diagnosis\", fontsize=6.6, color=INK3, ha=\"left\", va=\"top\")\n    save(fig, \"fig3_latent_geometry\")\n\n\nfig_geometry()\n\n# =====================================================================\n# Figure 4 -- what the geometry is worth, and what conditioning buys\n# =====================================================================\ndef fig_separation(S):\n    from scipy.stats import ttest_rel, t as tdist\n    # with two classes Davies-Bouldin = 2 / (inter/intra), so the two are one measurement\n    specs = [(\"sil\", \"Silhouette\", +1), (\"ari\", \"Adjusted Rand\", +1),\n             (\"iir\", \"Inter / intra distance\\n(= 2 / Davies\u2013Bouldin)\", +1), (\"ch\", \"Calinski\u2013Harabasz\", +1)]\n    DPLS, DAGN, DCND = PLS, CVAE_N, CVAE_T\n    arms = [(\"pls\", DPLS, \"PLS-DA\"), (\"cvae_neutral\", DAGN, \"CVAE\\nagn.\"), (\"cvae_true\", DCND, \"CVAE\\ncond.\")]\n    fig = plt.figure(figsize=(TW, 6.3))\n    gs = fig.add_gridspec(2, 4, height_ratios=[1.0, 0.72], hspace=0.44, wspace=0.42)\n\n    # ---- (a) one paired slope chart per metric --------------------------------\n    for k, (m, name, sgn) in enumerate(specs):\n        ax = fig.add_subplot(gs[0, k])\n        V = [runs[f\"{m}_{a}\"].values for a, _, _ in arms]\n        for i in range(len(runs)):\n            up = (V[1][i] - V[0][i]) * sgn > 0\n            ax.plot([0, 1], [V[0][i], V[1][i]], color=DAGN if up else DPLS, lw=0.65,\n                    alpha=0.62 if up else 0.95, zorder=2)\n            ax.plot([1, 2], [V[1][i], V[2][i]], color=DCND, lw=0.5, alpha=0.55, zorder=1)\n        for x, (v, (a, c, _)) in enumerate(zip(V, arms)):\n            ax.scatter(np.full(len(v), x), v, s=6, color=c, lw=0, zorder=3)\n            mu, sd = v.mean(), v.std()\n            ax.add_patch(Rectangle((x - 0.17, mu - sd), 0.34, 2 * sd, facecolor=tint(c, 0.24), edgecolor=\"none\",\n                                   zorder=0))\n            ax.plot([x - 0.22, x + 0.22], [mu, mu], color=c, lw=1.8, solid_capstyle=\"butt\", zorder=4)\n        wins = int(((V[1] - V[0]) * sgn > 0).sum())\n        ax.set_xticks([0, 1, 2]); ax.set_xticklabels([a[2] for a in arms], fontsize=6.0)\n        for t_, (_, c, _) in zip(ax.get_xticklabels(), arms):\n            t_.set_color(c)\n        ax.tick_params(axis=\"x\", length=0); ax.tick_params(axis=\"y\", labelsize=5.8)\n        ax.set_xlim(-0.45, 2.45); ax.spines[\"bottom\"].set_visible(False); ygrid(ax)\n        if m == \"ari\":\n            ax.axhline(0, color=INK3, lw=0.5, ls=(0, (1, 1.6)), zorder=1)\n        if sgn < 0:\n            ax.invert_yaxis()\n        ax.text(0.5, 1.10, name.split(\"\\n\")[0], transform=ax.transAxes, ha=\"center\", va=\"bottom\", fontsize=7.3, color=INK)\n        ax.text(0.5, 1.03, f\"CVAE agn. better in {wins}/20\",\n                transform=ax.transAxes, ha=\"center\", va=\"bottom\", fontsize=5.6, color=INK3)\n    a0 = fig.axes[0]\n    a0.annotate(\"a\", xy=(-0.42, 1.10), xycoords=\"axes fraction\", fontsize=9.5, fontweight=\"bold\",\n                family=\"DejaVu Sans\", color=INK, ha=\"left\", va=\"bottom\", annotation_clip=False)\n    a0.annotate(\"Every split, every metric\", xy=(-0.42, 1.30), xycoords=\"axes fraction\", fontsize=8.6,\n                color=INK, ha=\"left\", va=\"bottom\", annotation_clip=False)\n    a0.annotate(\"held-out arrays; one line per split; bar = mean, band = $\\\\pm$SD; higher = better throughout\",\n                xy=(-0.42, 1.22), xycoords=\"axes fraction\",\n                fontsize=6.4, color=INK3, ha=\"left\", va=\"bottom\", annotation_clip=False)\n\n    # ---- (b) distribution of the per-split gain, two scales ----------------------\n    from scipy.stats import gaussian_kde\n    sub = gs[1, :].subgridspec(1, 2, width_ratios=[3.1, 1.0], wspace=0.10)\n    ari_slot = sub[1].subgridspec(3, 1, hspace=0.0)[1:2, 0]\n    def rows_for(metrics, ratio):\n        out = []\n        for m, name, sgn in metrics:\n            b_ = runs[f\"{m}_pls\"].values\n            per = {}\n            for arm in (\"cvae_neutral\", \"cvae_true\"):\n                a_ = runs[f\"{m}_{arm}\"].values\n                per[arm] = (np.log2(a_ / b_) * sgn) if ratio else (a_ - b_)\n            out.append((name, per, ttest_rel(runs[f\"{m}_cvae_neutral\"], b_).pvalue))\n        return out\n    blocks = [(sub[0], rows_for([s for s in specs if s[0] != \"ari\"], True), True),\n              (ari_slot, rows_for([s for s in specs if s[0] == \"ari\"], False), False)]\n    axes_b = []\n    for spec_, R_, ratio in blocks:\n        ax = fig.add_subplot(spec_); axes_b.append(ax)\n        allv = np.concatenate([np.r_[r[1][\"cvae_neutral\"], r[1][\"cvae_true\"]] for r in R_])\n        lo, hi = min(allv.min(), 0) - 0.08 * np.ptp(allv), allv.max() + 0.08 * np.ptp(allv)\n        for i, (name, per, p_) in enumerate(R_):\n            if i % 2 == 0:\n                ax.axhspan(i - 0.5, i + 0.5, color=\"#f3f3f7\", lw=0, zorder=0)\n            d = per[\"cvae_neutral\"]\n            g = np.linspace(d.min() - 0.05 * np.ptp(d), d.max() + 0.05 * np.ptp(d), 200)\n            k = gaussian_kde(d)(g); k = k / k.max() * 0.36\n            ax.fill_between(g, i - 0.04, i - 0.04 - k, color=tint(DAGN, 0.30), lw=0, zorder=1)\n            ax.plot(g, i - 0.04 - k, color=DAGN, lw=0.8, zorder=2)\n            ax.scatter(d, np.full(len(d), i + 0.10) + np.random.RandomState(i).uniform(-0.05, 0.05, len(d)),\n                       s=7, color=DAGN, alpha=0.85, lw=0, zorder=3)\n            mu = d.mean(); h = tdist.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d))\n            ax.plot([mu - h, mu + h], [i + 0.27, i + 0.27], color=INK, lw=1.4, solid_capstyle=\"butt\", zorder=4)\n            ax.scatter(mu, i + 0.27, s=26, marker=\"D\", color=INK, edgecolor=SURFACE, linewidth=0.6, zorder=5)\n            dc = per[\"cvae_true\"]; mc = dc.mean()\n            hc = tdist.ppf(0.975, len(dc) - 1) * dc.std(ddof=1) / np.sqrt(len(dc))\n            ax.plot([mc - hc, mc + hc], [i + 0.39, i + 0.39], color=DCND, lw=0.9, zorder=4)\n            ax.scatter(mc, i + 0.39, s=13, marker=\"D\", facecolor=SURFACE, edgecolor=DCND, linewidth=0.8, zorder=5)\n            lab = (f\"\u00d7{2 ** mu:.2f}  [{2 ** (mu - h):.2f}\u2013{2 ** (mu + h):.2f}]\" if ratio\n                   else f\"+{mu:.3f}  [{mu - h:.3f}\u2013{mu + h:.3f}]\")\n            if ratio:\n                ax.text(1.0, i - 0.30, lab, transform=ax.get_yaxis_transform(), ha=\"right\",\n                        va=\"center\", fontsize=6.1, color=INK, fontweight=\"bold\")\n                ax.text(1.0, i - 0.12, pmath(p_), transform=ax.get_yaxis_transform(), ha=\"right\",\n                        va=\"center\", fontsize=5.7, color=INK2)\n            else:\n                ax.text(0.0, 1.36, \"Adjusted Rand (k-means agreement)\", transform=ax.transAxes, ha=\"left\",\n                        va=\"bottom\", fontsize=6.5, color=INK)\n                ax.text(0.0, 1.08, f\"{lab}   {pmath(p_)}\", transform=ax.transAxes, ha=\"left\",\n                        va=\"bottom\", fontsize=5.9, color=INK, fontweight=\"bold\")\n        ax.axvline(0, color=INK, lw=0.8, zorder=1)\n        ax.set_xlim(lo, hi * 1.42 if ratio else hi)\n        ax.set_ylim(len(R_) - 0.45, -0.55)\n        ax.set_yticks(range(len(R_)))\n        if ratio:\n            ax.set_yticklabels([r[0] for r in R_], fontsize=6.5, linespacing=1.15)\n            tv = [1, 1.5, 2, 3, 4]\n            ax.set_xticks([np.log2(v) for v in tv]); ax.set_xticklabels([f\"\u00d7{v:g}\" for v in tv])\n            ax.set_xlabel(\"fold change, CVAE agnostic / PLS-DA on the same split (log scale)\", fontsize=6.6)\n        else:\n            ax.set_yticklabels([\"\"])\n            ax.set_xlabel(\"difference, CVAE agn. \u2212 PLS-DA\", fontsize=6.4)\n        ax.tick_params(axis=\"y\", length=0); xgrid(ax)\n        for s_ in (\"left\", \"right\"):\n            ax.spines[s_].set_visible(False)\n    hl = [Line2D([], [], marker=\"o\", ls=\"\", color=DAGN, markersize=3.5, label=\"one split\"),\n          Rectangle((0, 0), 1, 1, facecolor=tint(DAGN, 0.30), edgecolor=DAGN, lw=0.6, label=\"distribution\"),\n          Line2D([], [], marker=\"D\", color=INK, lw=1.4, markersize=4.2, label=\"mean, 95% CI\"),\n          Line2D([], [], marker=\"D\", color=DCND, markerfacecolor=SURFACE, markeredgecolor=DCND,\n                 lw=0.9, markersize=3.6, label=\"CVAE conditioned (reference)\")]\n    axes_b[0].legend(handles=hl, loc=\"upper center\", bbox_to_anchor=(0.66, -0.24), ncol=4, fontsize=6.2,\n                     frameon=False, columnspacing=1.4)\n    panel(axes_b[0], \"b\", \"How much better the CVAE separates the classes, split by split\",\n          \"fold change on three measures (left); adjusted Rand as a difference (right); right of the line = CVAE better\", dx=-0.16)\n    save(fig, \"fig4_latent_separation\")\n\n\nfig_separation(S)\n\n# =====================================================================\n# Figure 5 -- posterior probability landscapes over the latent spaces\n# =====================================================================\ndef fig_landscape():\n    emb = load(\"latent_embeddings.csv\")\n    tags = [\"PLS-DA\", \"CVAE_neutral\"]\n    projs = [((\"pca1\", \"pca2\"), \"PCA\"), ((\"tsne1\", \"tsne2\"), \"t-SNE\")]\n    if \"umap1\" in emb.columns and emb.umap1.notna().any():\n        projs.append(((\"umap1\", \"umap2\"), \"UMAP\"))\n\n    fig, axes = plt.subplots(2, len(projs), figsize=(TW, TW / len(projs) * 2 + 0.7))\n    axes = np.atleast_2d(axes)\n    im = None\n    for row, tag in enumerate(tags):\n        for col, (proj, name) in enumerate(projs):\n            ax = axes[row, col]\n            sub = emb[emb.model == tag]\n            XY = sub[list(proj)].values\n            pad = 0.09 * (XY.max(0) - XY.min(0))\n            gx = np.linspace(XY[:, 0].min() - pad[0], XY[:, 0].max() + pad[0], 220)\n            gy = np.linspace(XY[:, 1].min() - pad[1], XY[:, 1].max() + pad[1], 220)\n            # class-conditional KDEs -> posterior P(AD | position) at equal priors\n            dens = {g: kde2d(sub.loc[sub.group == g, list(proj)].values, gx, gy, 1.15)\n                    for g in (\"AD\", \"Control\")}\n            post = dens[\"AD\"] / (dens[\"AD\"] + dens[\"Control\"] + 1e-300)\n            im = ax.pcolormesh(gx, gy, post, cmap=DIVERGING, vmin=0, vmax=1,\n                               shading=\"gouraud\", rasterized=True)\n            ax.contour(gx, gy, post, levels=[0.5], colors=[INK2], linewidths=1.0,\n                       linestyles=[(0, (3.5, 2.0))])\n            # The surface already carries class by colour, so the marks must not:\n            # a blue dot on a blue field is invisible. Shape plus fill/no-fill in\n            # ink reads on every part of the ramp.\n            mark = {\"Control\": dict(marker=\"o\", facecolor=SURFACE, edgecolor=INK),\n                    \"AD\": dict(marker=\"^\", facecolor=INK, edgecolor=SURFACE)}\n            for g in (\"Control\", \"AD\"):\n                s = sub[sub.group == g]\n                ax.scatter(s[proj[0]], s[proj[1]], s=13, linewidth=0.6,\n                           label=g, zorder=3, **mark[g])\n            ax.set_xticks([]); ax.set_yticks([])\n            ax.set_xlim(gx[0], gx[-1]); ax.set_ylim(gy[0], gy[-1])\n            for sp in ax.spines.values():\n                sp.set_visible(False)\n            panel(ax, \"abcdef\"[row * len(projs) + col],\n                  name if row == 0 else None, dx=-0.11)\n            if col == 0:\n                ax.annotate(MODEL_LABEL[tag], xy=(-0.06, 0.5),\n                            xycoords=\"axes fraction\", rotation=90, ha=\"center\",\n                            va=\"center\", fontsize=7.6, color=INK2,\n                            annotation_clip=False)\n            if row == 0 and col == 0:\n                ax.legend(loc=\"upper left\", markerscale=1.25, labelcolor=INK,\n                          facecolor=SURFACE, framealpha=0.88, frameon=True,\n                          edgecolor=\"none\", borderpad=0.45)\n    cax = fig.add_axes([0.24, -0.022, 0.44, 0.012])\n    cb = fig.colorbar(im, cax=cax, orientation=\"horizontal\")\n    cb.set_label(\"Posterior $P(\\\\mathrm{AD}\\\\mid$ latent position$)$ from class-conditional KDE\",\n                 fontsize=7, color=INK2)\n    cb.set_ticks([0, 0.25, 0.5, 0.75, 1]); cb.ax.tick_params(labelsize=6.5, length=2)\n    cb.outline.set_visible(False)\n    fig.text(0.72, -0.020, \"dashed contour: $P=0.5$\", fontsize=6.6, color=INK2,\n             ha=\"left\", style=\"italic\")\n    fig.tight_layout(w_pad=1.0, h_pad=1.4)\n    save(fig, \"fig5_probability_landscape\")\n\n\n\n# fig_landscape(): merged into Figure 3 (latent geometry with the training-set landscape)\n\n# =====================================================================\n# Figure 6 -- reconstruction fidelity\n# =====================================================================\ndef fig_reconstruction(S):\n    sc = load(\"recon_scatter_representative.csv\")\n    oof = load(\"oof_reconstruction.csv\")\n\n    fig = plt.figure(figsize=(TW, 6.5))\n    gs = fig.add_gridspec(2, 2, wspace=0.30, hspace=0.50, bottom=0.10)\n    lim = (-4.5, 4.5)      # where 99.9% of the values are; the rest is noted on the panel\n    beyond = np.mean((sc.observed.abs() > 4.5) | (sc.recon_pls.abs() > 4.5) | (sc.recon_cvae.abs() > 4.5))\n\n    def clean_box(ax, v, x, c, w=0.34, label_y=None, side_label=True):\n        \"\"\"Slim box and 5-95% whisker over a faint strip; median and mean written above.\"\"\"\n        rs = np.random.RandomState(int(x * 10))\n        ax.scatter(x + rs.uniform(-w * 0.42, w * 0.42, len(v)), v, s=2.2, color=c, alpha=0.18,\n                   linewidth=0, zorder=1, rasterized=True)\n        q5, q1, q2, q3, q95 = np.percentile(v, [5, 25, 50, 75, 95])\n        ax.plot([x, x], [q5, q1], color=c, lw=1.0, zorder=2); ax.plot([x, x], [q3, q95], color=c, lw=1.0, zorder=2)\n        ax.add_patch(Rectangle((x - w / 2, q1), w, q3 - q1, facecolor=tint(c, 0.18), edgecolor=c, lw=1.1,\n                               zorder=3))\n        ax.plot([x - w / 2, x + w / 2], [q2, q2], color=c, lw=2.0, solid_capstyle=\"butt\", zorder=4)\n        ax.scatter([x], [v.mean()], s=16, marker=\"D\", color=SURFACE, edgecolor=c, linewidth=1.0, zorder=5)\n        # values written beside the box, level with it, clear of the points\n        if side_label: ax.text(x + w / 2 + 0.06, q2, f\"median {q2:.3f}\\nmean {v.mean():.3f}\", ha=\"left\", va=\"center\",\n                fontsize=5.9, color=c, linespacing=1.3, zorder=6,\n                bbox=dict(boxstyle=\"square,pad=0.12\", facecolor=SURFACE, edgecolor=\"none\", alpha=0.85))\n\n    for i, (col, name, cmap, key) in enumerate(\n            ((\"recon_pls\", \"PLS-DA\", SEQ_PLS, \"mse_pls_mean_mean\"),\n             (\"recon_cvae\", \"CVAE (label-conditioned)\", SEQ_CVAE, \"mse_cvae_true_mean_mean\"))):\n        ax = fig.add_subplot(gs[0, i]); ax.set_box_aspect(0.82)\n        x, y = sc.observed.values, sc[col].values\n        ax.hexbin(x, y, gridsize=52, bins=\"log\", cmap=cmap, mincnt=1, linewidths=0, rasterized=True,\n                  extent=(*lim, *lim))\n        ax.plot(lim, lim, color=INK2, lw=0.7, ls=(0, (3.5, 2.5)))\n        b, a0 = np.polyfit(x, y, 1)\n        ax.plot(np.array(lim), a0 + b * np.array(lim), color=MODEL_COLOR[\"pls\" if i == 0 else \"cvae_true\"], lw=1.1)\n        ax.set_xlim(lim); ax.set_ylim(lim); ax.set_xticks([-4, -2, 0, 2, 4]); ax.set_yticks([-4, -2, 0, 2, 4])\n        ax.set_xlabel(\"Observed expression ($z$)\")\n        ax.set_ylabel(\"Reconstructed ($z$)\" if i == 0 else \"\")\n        ax.text(0.04, 0.95, f\"mean MSE {S[key]:.3f}\\nfitted slope {b:.2f}\", transform=ax.transAxes,\n                ha=\"left\", va=\"top\", fontsize=6.8, color=INK, linespacing=1.35)\n        ax.text(0.96, 0.04, f\"dashed = identity \u00b7 {beyond:.1%} beyond \u00b14.5\", transform=ax.transAxes,\n                ha=\"right\", va=\"bottom\", fontsize=5.6, color=INK3)\n        panel(ax, \"ab\"[i], name, \"one representative held-out split\" if i == 0 else\n              \"decoder output saved for this arm only\", dx=-0.20)\n\n    # (c) paired per-array error\n    ax = fig.add_subplot(gs[1, 0])\n    d = {\"PLS-DA\": oof.mse_pls.values, \"CVAE (agn.)\": oof.mse_cvae_neutral.values}\n    for j, (nm, color) in enumerate(((\"PLS-DA\", PLS), (\"CVAE (agn.)\", CVAE_N))):\n        clean_box(ax, d[nm], j, color, w=0.34)\n    ax.set_xticks([0, 1]); ax.set_xticklabels(list(d)); ax.tick_params(axis=\"x\", length=0)\n    for t_, c in zip(ax.get_xticklabels(), (PLS, CVAE_N)):\n        t_.set_color(c)\n    ax.set_yscale(\"log\"); ax.set_ylim(0.07, 22); ax.set_xlim(-0.45, 1.85)\n    ax.set_yticks([0.1, 0.3, 1, 3, 10]); ax.set_yticklabels([\"0.1\", \"0.3\", \"1\", \"3\", \"10\"])\n    ax.set_ylabel(\"Per-array MSE (log)\"); ygrid(ax)\n    p, r = S[\"recon_cvae_neutral_vs_pls_wilcoxon_p\"], S[\"recon_cvae_neutral_vs_pls_rank_biserial\"]\n    bracket(ax, 0, 1, 11.5, f\"{stars(p)}  Wilcoxon {pmath(p)},  $r = {r:.2f}$\", tick=1.8, fontsize=6.4)\n    panel(ax, \"c\", \"Every held-out array\u2013split pair\",\n          f\"$n = {int(S['recon_n_oof_samples']):,}$, paired; box = IQR, whiskers 5\u201395%, \u25c7 = mean\", dx=-0.20)\n\n    # (d) does either model reconstruct one diagnosis better?\n    ax = fig.add_subplot(gs[1, 1])\n    pos = [0, 1.0, 2.6, 3.6]; k = 0; tl = []\n    for nm, col, color, tag in ((\"PLS-DA\", \"mse_pls\", PLS, \"pls\"),\n                                (\"CVAE (agn.)\", \"mse_cvae_neutral\", CVAE_N, \"cvaeneutral\")):\n        for g in (\"Control\", \"AD\"):\n            v = oof.loc[oof.group == g, col].values\n            clean_box(ax, v, pos[k], GROUP_COLOR[g], w=0.34, side_label=False)\n            tl.append((f\"{g}\\nmedian {np.median(v):.3f}\\nmean {v.mean():.3f}\", GROUP_COLOR[g]))\n            k += 1\n        pg = S[f\"recon_group_{tag}_mannwhitney_p\"]\n        a_, b_ = pos[k - 2], pos[k - 1]\n        bracket(ax, a_, b_, 6.0, f\"{stars(pg)}  {pmath(pg)}\", tick=0.6, fontsize=6.2)\n        ax.text((a_ + b_) / 2, -0.255, nm, transform=ax.get_xaxis_transform(), ha=\"center\", fontsize=7.4,\n                color=color)\n    ax.set_xticks(pos); ax.set_xticklabels([t[0] for t in tl], fontsize=6.0, linespacing=1.3)\n    for t_, (_, c) in zip(ax.get_xticklabels(), tl):\n        t_.set_color(c)\n    ax.tick_params(axis=\"x\", length=0)\n    ax.set_yscale(\"log\"); ax.set_ylim(0.07, 11); ax.set_xlim(-0.6, 4.2)\n    ax.set_yticks([0.1, 0.3, 1, 3]); ax.set_yticklabels([\"0.1\", \"0.3\", \"1\", \"3\"])\n    ax.set_ylabel(\"Per-array MSE (log)\"); ygrid(ax)\n    panel(ax, \"d\", \"Does error depend on diagnosis?\", \"Mann\u2013Whitney on held-out arrays, within each model\",\n          dx=-0.20)\n    save(fig, \"fig6_reconstruction\")\n\n\nfig_reconstruction(S)\n\n# =====================================================================\n# Figure 7 -- out-of-fold uncertainty\n# =====================================================================\ndef fig_uncertainty(S):\n    unc = load(\"oof_uncertainty.csv\")\n    panels = [(\"kl_neutral\", \"KL divergence\", \"label-agnostic CVAE\", True),\n              (\"kl_true\", \"KL divergence\", \"label-conditioned CVAE (reference)\", True),\n              (\"meanvar_neutral\", \"Mean latent variance\", \"label-agnostic CVAE\", False),\n              (\"meanvar_true\", \"Mean latent variance\", \"label-conditioned CVAE (reference)\", False)]\n\n    def clean_box(ax, v, x, c, w=0.36):\n        rs = np.random.RandomState(int(x * 10) + 3)\n        ax.scatter(x + rs.uniform(-w * 0.45, w * 0.45, len(v)), v, s=2.2, color=c, alpha=0.18,\n                   linewidth=0, zorder=1, rasterized=True)\n        q5, q1, q2, q3, q95 = np.percentile(v, [5, 25, 50, 75, 95])\n        ax.plot([x, x], [q5, q1], color=c, lw=1.0, zorder=2); ax.plot([x, x], [q3, q95], color=c, lw=1.0, zorder=2)\n        ax.add_patch(Rectangle((x - w / 2, q1), w, q3 - q1, facecolor=tint(c, 0.18), edgecolor=c, lw=1.1, zorder=3))\n        ax.plot([x - w / 2, x + w / 2], [q2, q2], color=c, lw=2.0, solid_capstyle=\"butt\", zorder=4)\n        ax.scatter([x], [v.mean()], s=16, marker=\"D\", color=SURFACE, edgecolor=c, linewidth=1.0, zorder=5)\n        return q2, v.mean()\n\n    fig, axes = plt.subplots(2, 2, figsize=(TW, 6.4))\n    fig.subplots_adjust(wspace=0.32, hspace=0.62, bottom=0.14, top=0.93)\n    for i, (col, ylab, title, logy) in enumerate(panels):\n        ax = axes.ravel()[i]\n        tl = []\n        for j, g in enumerate((\"Control\", \"AD\")):\n            v = unc.loc[unc.group == g, col].values\n            med, mean = clean_box(ax, v, j, GROUP_COLOR[g])\n            fmt = (lambda z: f\"{z:.3f}\") if not logy else (lambda z: f\"{z:.2f}\")\n            tl.append(f\"{g}\\nmedian {fmt(med)}\\nmean {fmt(mean)}\")\n        ax.set_xticks([0, 1]); ax.set_xticklabels(tl, fontsize=6.0, linespacing=1.3)\n        for t_, g in zip(ax.get_xticklabels(), (\"Control\", \"AD\")):\n            t_.set_color(GROUP_COLOR[g])\n        ax.tick_params(axis=\"x\", length=0); ax.set_xlim(-0.6, 1.6)\n        p = S[f\"{col}_mannwhitney_p\"]\n        if logy:\n            ax.set_yscale(\"log\"); ax.set_ylim(0.003, 150)\n            ax.set_yticks([0.01, 0.1, 1, 10, 100]); ax.set_yticklabels([\"0.01\", \"0.1\", \"1\", \"10\", \"100\"])\n            bracket(ax, 0, 1, 45, f\"{stars(p)}  Mann\u2013Whitney {pmath(p)}\", tick=12, fontsize=6.2)\n            ax.set_ylabel(f\"{ylab}, nats (log)\" if i % 2 == 0 else \"\")\n        else:\n            ax.set_ylim(0.38, 1.13)\n            bracket(ax, 0, 1, 1.065, f\"{stars(p) or 'n.s.'}  Mann\u2013Whitney {pmath(p)}\", tick=0.015, fontsize=6.2)\n            ax.set_ylabel(ylab if i % 2 == 0 else \"\")\n        ygrid(ax)\n        delta = S[f\"{col}_pct_increase_AD\"]\n        panel(ax, \"abcd\"[i], f\"{ylab} \u2014 {title.split(' CVAE')[0]}\",\n              f\"mean in AD {delta:+.1f}% vs control\" + (\" \u00b7 reference arm\" if \"reference\" in title else \"\"), dx=-0.20)\n    fig.text(0.5, 0.012, f\"Held-out arrays pooled over the 20 splits ({int(S['unc_n_AD']):,} AD, \"\n                         f\"{int(S['unc_n_control']):,} control). Box = IQR, whiskers 5\u201395%, open diamond = mean.\\n\"\n                         \"KL: the median is lower but the mean higher in AD, so the difference sits in the upper tail.\",\n             ha=\"center\", fontsize=6.4, color=INK2, style=\"italic\")\n    save(fig, \"fig7_uncertainty\")\n\n\nfig_uncertainty(S)\n\n# =====================================================================\n# Figure 8 -- gene discovery and its discordance\n# =====================================================================\ndef fig_genes(S):\n    import glob as _glob\n    from pathlib import Path as _P\n    pls = load(\"PLSDA_gene_significance.csv\").set_index(\"gene\")\n    cv = load(\"CVAE_gene_significance.csv\").set_index(\"gene\")\n    de = load(\"gene_de_stats.csv\").set_index(\"gene\")\n    ps, cs = set(pls.index[pls.significant]), set(cv.index[cv.significant])\n    both = ps & cs\n    prov = {g: (\"both\" if g in both else \"pls\" if g in ps else \"cvae\") for g in ps | cs}\n    PC = {\"pls\": PLS, \"both\": ACCENT, \"cvae\": CVAE_N}\n    PM = {\"pls\": \"o\", \"both\": \"D\", \"cvae\": \"s\"}\n    PN = {\"pls\": \"PLS-DA only\", \"both\": \"both models\", \"cvae\": \"CVAE only\"}\n\n    def _find(rel, marker):\n        f = DATA / \"validation\" / rel\n        if f.exists():\n            return f\n        hits = sorted(_glob.glob(f\"/kaggle/input/**/{marker}\", recursive=True))\n        return _P(hits[0]) if hits else None\n    # replication in the four cohorts V7 never used (meta-analysis, new cohorts only)\n    fm = _find(\"v10adgenevalidation/meta_analysis_new_cohorts_only.csv\", \"meta_analysis_new_cohorts_only.csv\")\n    M = pd.read_csv(fm, index_col=0) if fm is not None else pd.DataFrame(columns=[\"pooled\", \"q\", \"k\"])\n    bg = M.join(de, how=\"inner\")\n    rep_all = (bg.q < 0.05) & (np.sign(bg.pooled) == np.sign(bg.logFC))\n    rep = lambda g: bool(rep_all.get(g, False))\n    # V15 age analyses\n    age_sel = {}\n    for arm, tag in ((\"A_age_corrected\", \"corr\"), (\"B_age_matched\", \"match\")):\n        sets = []\n        for f_ in (\"PLSDA_gene_significance.csv\", \"CVAE_gene_significance.csv\"):\n            ff = _find(f\"v15adage/{arm}/{f_}\", f\"{arm}/{f_}\")\n            d_ = pd.read_csv(ff) if ff is not None else pd.DataFrame(columns=[\"gene\", \"significant\"])\n            sets.append(set(d_.gene[d_.significant]))\n        age_sel[tag] = sets\n\n    # landscape: a-d on the left, the mirror chart e down the full right-hand side\n    fig = plt.figure(figsize=(9.6, 7.4))\n    outer = fig.add_gridspec(1, 2, width_ratios=[1.45, 1.0], wspace=0.10, left=0.06, right=0.99, top=0.90,\n                             bottom=0.07)\n    gs = outer[0].subgridspec(2, 3, height_ratios=[0.62, 1.38], hspace=0.26, wspace=0.62)\n\n    # ---- (a, b) each gene against its own permutation null ---------------------------\n    allg = list(de.index)\n    for j, (real_f, null_f, sel_, mdl, c, lab) in enumerate(\n            ((\"vip_real.npy\", \"vip_null.npy\", ps, \"PLS-DA\", PLS, \"VIP\"),\n             (\"ig_real.npy\", \"ig_null.npy\", cs, \"CVAE\", CVAE_N, \"|IG|\"))):\n        ax = fig.add_subplot(gs[0, j]); ax.set_box_aspect(1)\n        mr, mn = np.load(DATA / real_f).mean(0), np.load(DATA / null_f).mean(0)\n        si = np.array([g in sel_ for g in allg])\n        lo, hi = np.percentile(np.r_[mr, mn], 0.5), max(mr.max(), mn.max()) * 1.2\n        GN = LinearSegmentedColormap.from_list(f\"gn{j}\", [\"#ffffff\", \"#ececf0\", \"#d2d2da\", \"#a9a9b5\"])\n        ax.hexbin(np.log10(mn[~si]), np.log10(mr[~si]), gridsize=36, bins=\"log\", cmap=GN, mincnt=1,\n                  linewidths=0, rasterized=True, extent=(np.log10(lo), np.log10(hi), np.log10(lo), np.log10(hi)))\n        ax.plot([np.log10(lo), np.log10(hi)], [np.log10(lo), np.log10(hi)], color=INK2, lw=0.6, ls=(0, (3, 2)))\n        ax.scatter(np.log10(mn[si]), np.log10(mr[si]), s=9, color=c, edgecolor=SURFACE, lw=0.35, zorder=3)\n        ax.set_xlim(np.log10(lo), np.log10(hi)); ax.set_ylim(np.log10(lo), np.log10(hi))\n        ax.set_xticks([]); ax.set_yticks([])\n        ax.set_xlabel(f\"{lab}, permuted labels (log)\", fontsize=6.2); ax.set_ylabel(f\"{lab}, real labels (log)\", fontsize=6.2)\n        fold = np.median(mr[si] / mn[si]); bgf = np.median(mr[~si] / mn[~si])\n        ax.text(0.96, 0.17, f\"{si.sum()} selected: {fold:.1f}\u00d7 null\", transform=ax.transAxes, ha=\"right\",\n                va=\"bottom\", fontsize=5.9, color=c)\n        ax.text(0.96, 0.05, f\"all others: {bgf:.1f}\u00d7 null\", transform=ax.transAxes, ha=\"right\", va=\"bottom\",\n                fontsize=5.6, color=INK3)\n        panel(ax, \"ab\"[j], f\"{mdl}: real vs null\", \"each gene vs itself, permuted labels\", dx=-0.14)\n\n    # ---- (c) two rankings of one genome ----------------------------------------------\n    ax = fig.add_subplot(gs[0, 2]); ax.set_box_aspect(1)\n    rp = np.log10(pls.mean_score.rank(ascending=False) / len(pls))\n    rc = np.log10(cv.mean_score.reindex(pls.index).rank(ascending=False) / len(pls))\n    GRE = LinearSegmentedColormap.from_list(\"gre\", [\"#ffffff\", \"#ececf0\", \"#d6d6dd\", \"#b8b8c2\"])\n    ax.hexbin(rp, rc, gridsize=38, bins=\"log\", cmap=GRE, mincnt=1, linewidths=0, rasterized=True,\n              extent=(-4.15, 0, -4.15, 0))\n    ax.plot([-4.15, 0], [-4.15, 0], color=INK3, lw=0.5, ls=(0, (3, 2)))\n    for k in (\"pls\", \"cvae\", \"both\"):\n        g_ = [g for g, v in prov.items() if v == k]\n        ax.scatter(rp[g_], rc[g_], s=18 if k == \"both\" else 10, marker=PM[k], color=PC[k], edgecolor=SURFACE,\n                   linewidth=0.4, zorder=3)\n    tk = [-4, -3, -2, -1, 0]; tl = [\"0.01%\", \"0.1%\", \"1%\", \"10%\", \"100%\"]\n    ax.set_xticks(tk); ax.set_xticklabels(tl); ax.set_yticks(tk); ax.set_yticklabels(tl)\n    ax.set_xlim(-4.15, 0); ax.set_ylim(-4.15, 0)\n    ax.set_xlabel(\"PLS-DA rank (top fraction)\", fontsize=6.2)\n    ax.set_ylabel(\"CVAE rank (top fraction)\", fontsize=6.2)\n    ax.tick_params(labelsize=5.6)\n    fig.canvas.draw()\n    bl = sorted(both)\n    # label only the shared genes that also replicate in unseen cohorts: the panel stays readable\n    bl = [g for g in bl if rep(g)]\n    # names in a column in the empty upper-left corner, each tied to its point by a hairline\n    column_labels(ax, rp[bl].values, rc[bl].values, bl, [ACCENT] * len(bl), -4.05, -2.75, -0.55, \"right\",\n                  fontsize=5.2, spacing=0.30)\n    for t_ in ax.texts:\n        ap = getattr(t_, \"arrow_patch\", None)\n        if ap is not None:\n            ap.set_alpha(0.6); ap.set_linewidth(0.35)\n    ax.text(0.97, 0.03, \"named: shared and replicated\", transform=ax.transAxes, ha=\"right\", va=\"bottom\",\n            fontsize=5.4, color=INK3, style=\"italic\")\n    panel(ax, \"c\", \"Two rankings, one genome\",\n          f\"genome-wide $\\\\rho$ = {S['spearman_rho_vip_ig']:.3f}\", dx=-0.22)\n\n    # ---- (c) volcano with every selected gene named -----------------------------------\n    ax = fig.add_subplot(gs[1, :])\n    d = de.copy(); d[\"nlp\"] = -np.log10(d.DE_pvalue.clip(lower=1e-300))\n    ns = d[~d.index.isin(prov)]\n    ax.scatter(ns.logFC, ns.nlp, s=1.4, color=\"#c9c9d1\", linewidth=0, rasterized=True, zorder=1)\n    sel = d[d.index.isin(prov)].copy(); sel[\"k\"] = [prov[g] for g in sel.index]\n    sel[\"rep\"] = [rep(g) for g in sel.index]\n    for k in (\"pls\", \"cvae\", \"both\"):\n        s_ = sel[sel.k == k]\n        ax.scatter(s_.logFC, s_.nlp, s=22 if k == \"both\" else 14, marker=PM[k], color=PC[k], edgecolor=SURFACE,\n                   linewidth=0.45, zorder=3)\n    r_ = sel[sel.rep]\n    ax.scatter(r_.logFC, r_.nlp, s=70, facecolor=\"none\", edgecolor=INK, linewidth=0.7, zorder=2)\n    ax.axvline(0, color=INK3, lw=0.6, ls=(0, (3, 2)))\n    xmin_, xmax_ = d.logFC.min(), d.logFC.max(); span_ = xmax_ - xmin_\n    ax.set_xlim(xmin_ - 0.42 * span_, xmax_ + 0.40 * span_)\n    top_ = sel.nlp.max() * 1.04\n    ax.set_ylim(-0.02 * top_, top_)\n    # two label columns per side, alternating by height, so each column is half as dense and\n    # every leader stays short; leaders are hairlines so the names, not the lines, carry the eye\n    for side, s_ in ((\"left\", sel[sel.logFC < 0]), (\"right\", sel[sel.logFC >= 0])):\n        s_ = s_.sort_values(\"nlp\")\n        for col_i in (1, 0):          # outer column first, so inner names mask its leaders\n            part = s_.iloc[col_i::2]\n            xt = (xmin_ - (0.06 + 0.17 * col_i) * span_) if side == \"left\" else (xmax_ + (0.06 + 0.17 * col_i) * span_)\n            column_labels(ax, part.logFC.values, part.nlp.values, list(part.index), [PC[k] for k in part.k],\n                          xt, -0.01 * top_, top_ * 0.985, side, fontsize=5.0, spacing=top_ * 0.026)\n    for ln in ax.texts:\n        ap = getattr(ln, \"arrow_patch\", None)\n        if ap is not None:\n            ap.set_alpha(0.35); ap.set_linewidth(0.3)\n            ln.set_bbox(dict(boxstyle=\"square,pad=0.08\", facecolor=SURFACE, edgecolor=\"none\"))\n    ax.set_xlabel(\"Standardised expression difference, AD \u2212 control (merged cohorts)\")\n    ax.set_ylabel(\"$-\\\\log_{10}$ $p$ (differential expression)\"); ygrid(ax)\n    hl = [Line2D([], [], marker=PM[k], ls=\"\", color=PC[k], markersize=4.5,\n                 label=f\"{PN[k]} ({sum(v == k for v in prov.values())})\") for k in (\"pls\", \"both\", \"cvae\")]\n    hl += [Line2D([], [], marker=\"o\", ls=\"\", markerfacecolor=\"none\", markeredgecolor=INK, markersize=7,\n                  label=\"replicates in independent cohorts\"),\n           Line2D([], [], marker=\"o\", ls=\"\", color=\"#c9c9d1\", markersize=3, label=\"not selected\")]\n    ax.legend(handles=hl, loc=\"upper center\", bbox_to_anchor=(0.5, 0.99), ncol=2, fontsize=6.2, frameon=True,\n              framealpha=1.0, facecolor=SURFACE, edgecolor=\"none\")\n    panel(ax, \"d\", \"Where every selected gene sits genome-wide\",\n          \"selection is not simply the strongest differential expression; names coloured by model\", dx=-0.075)\n\n    # ---- (e) mirror chart: the leading genes of each model, side by side ----------------\n    top_n = 25\n    tp = list(pls.mean_score.sort_values(ascending=False).index[:top_n])\n    tc = list(cv.mean_score.sort_values(ascending=False).index[:top_n])\n    union = list(dict.fromkeys(tp + tc))\n    rk_p = pls.mean_score.rank(ascending=False); rk_c = cv.mean_score.rank(ascending=False)\n    union.sort(key=lambda g: min(rk_p[g], rk_c[g]) + 0.001 * (rk_p[g] + rk_c[g]))\n    sub = outer[1].subgridspec(1, 3, width_ratios=[1, 0.62, 1], wspace=0.0)\n    axL, axM, axR = fig.add_subplot(sub[0]), fig.add_subplot(sub[1]), fig.add_subplot(sub[2])\n    y = np.arange(len(union))\n    pmax, cmax = pls.mean_score.max(), cv.mean_score.max()\n    for ax_, T, mx, c, sel_, sgn in ((axL, pls, pmax, PLS, ps, -1), (axR, cv, cmax, CVAE_N, cs, 1)):\n        v = T.loc[union, \"mean_score\"].values / mx; e = T.loc[union, \"std_score\"].values / mx\n        on = np.array([g in sel_ for g in union])\n        ax_.barh(y, sgn * v, 0.66, color=[c if o else tint(c, 0.25) for o in on], lw=0)\n        ax_.errorbar(sgn * v, y, xerr=e, fmt=\"none\", ecolor=[INK2 if o else RULE for o in on], elinewidth=0.6,\n                     capsize=0)\n        for yi, g in zip(y, union):\n            r_ = int((rk_p if T is pls else rk_c)[g])\n            ax_.text(sgn * (v[yi] + e[yi] + 0.03), yi, f\"#{r_}\", ha=\"left\" if sgn > 0 else \"right\", va=\"center\",\n                     fontsize=4.9, color=c if on[yi] else INK3)\n        ax_.set_ylim(len(union) - 0.4, -0.6); ax_.set_yticks([])\n        ax_.set_xlim((-1.35, 0) if sgn < 0 else (0, 1.35))\n        ax_.set_xticks([sgn * t for t in (0, 0.5, 1.0)]); ax_.set_xticklabels([\"0\", \"0.5\", \"1\"], fontsize=5.8)\n        ax_.grid(axis=\"x\", visible=True)\n        ax_.spines[\"left\" if sgn > 0 else \"right\"].set_visible(False)\n        if sgn < 0:\n            ax_.spines[\"left\"].set_visible(False); ax_.spines[\"right\"].set_visible(True)\n        ax_.set_xlabel((\"PLS-DA VIP\" if T is pls else \"CVAE |IG|\") + \" (relative)\", fontsize=6.4)\n    axM.set_ylim(len(union) - 0.4, -0.6); axM.axis(\"off\")\n    for yi, g in zip(y, union):\n        k = prov.get(g)\n        col = PC[k] if k else INK3\n        arrow = \"\u2191\" if de.loc[g, \"logFC\"] > 0 else \"\u2193\"\n        axM.text(0.5, yi, f\"{g}\", ha=\"center\", va=\"center\", fontsize=6.0, fontstyle=\"italic\", color=col,\n                 fontweight=\"bold\" if k == \"both\" else \"normal\")\n        axM.text(0.92, yi, arrow, ha=\"center\", va=\"center\", fontsize=6.4, color=AD if arrow == \"\u2191\" else CTRL)\n        if yi % 2 == 0:\n            for ax_ in (axL, axR):\n                ax_.axhspan(yi - 0.5, yi + 0.5, color=\"#f5f5f8\", lw=0, zorder=0)\n    axL.annotate(\"e\", xy=(-0.20, 1.0), xycoords=\"axes fraction\", xytext=(0, 21), textcoords=\"offset points\",\n                 fontsize=9.5, fontweight=\"bold\", family=\"DejaVu Sans\", color=INK, va=\"bottom\", annotation_clip=False)\n    axL.annotate(f\"Leading genes of each model (top {top_n} of each)\",\n                 xy=(0, 1.0), xycoords=\"axes fraction\", xytext=(0, 21), textcoords=\"offset points\", fontsize=8.6,\n                 color=INK, va=\"bottom\", annotation_clip=False)\n    axL.annotate(\"mean $\\\\pm$ SD over 20 splits, scaled to each model's top gene; #rank;\\npale = not selected \"\n                 \"by that model; bold name = both models; arrow = direction in AD\", xy=(0, 1.0), xycoords=\"axes fraction\", xytext=(0, 3),\n                 textcoords=\"offset points\", fontsize=6.0, color=INK3, va=\"bottom\", annotation_clip=False,\n                 linespacing=1.3)\n    save(fig, \"fig8_gene_discovery\")\n\n\nfig_genes(S)\n\n# =====================================================================\n# Figure 9 -- directional expression profile of every selected gene\n# =====================================================================\ndef fig_profile():\n    import glob as _glob\n    from pathlib import Path as _P\n    pls = load(\"PLSDA_gene_significance.csv\")\n    cv = load(\"CVAE_gene_significance.csv\")\n    de = load(\"gene_de_stats.csv\").set_index(\"gene\")\n    pls_set = set(pls.loc[pls.significant, \"gene\"]); cv_set = set(cv.loc[cv.significant, \"gene\"])\n    both = pls_set & cv_set\n    # within each block, genes run from most reduced to most increased in AD, so both traces\n    # sweep smoothly round the circle instead of zig-zagging gene to gene\n    dz = lambda g: de.loc[g, \"mean_z_AD\"] - de.loc[g, \"mean_z_control\"]\n    groups = [(\"PLS-DA only\", sorted(pls_set - cv_set, key=dz), PLS),\n              (\"Found by both\", sorted(both, key=dz), ACCENT),\n              (\"CVAE only\", sorted(cv_set - pls_set, key=dz), CVAE_N)]\n    # independent replication (4 cohorts never used), for a marker at each name\n    f_ = DATA / \"validation\" / \"v10adgenevalidation\" / \"meta_analysis_new_cohorts_only.csv\"\n    if not f_.exists():\n        hits = sorted(_glob.glob(\"/kaggle/input/**/meta_analysis_new_cohorts_only.csv\", recursive=True))\n        f_ = _P(hits[0]) if hits else None\n    M = pd.read_csv(f_, index_col=0) if f_ is not None else pd.DataFrame(columns=[\"pooled\", \"q\"])\n    rep = lambda g: g in M.index and M.loc[g, \"q\"] < 0.05 and np.sign(M.loc[g, \"pooled\"]) == np.sign(de.loc[g, \"logFC\"])\n\n    gap = 2.2                                   # empty spokes between blocks\n    slots, genes, gcol = [], [], []\n    pos = 0.0\n    for name, gl, c in groups:\n        for g in gl:\n            slots.append(pos); genes.append(g); gcol.append(c); pos += 1\n        pos += gap\n    n_slots = pos\n    ang = np.array(slots) / n_slots * 2 * np.pi\n    ad = np.array([de.loc[g, \"mean_z_AD\"] for g in genes]); ct = np.array([de.loc[g, \"mean_z_control\"] for g in genes])\n    r0, sc = 1.0, 1.15                          # zero ring and radial scale\n    rA, rC = r0 + sc * ad, r0 + sc * ct\n    r_out = r0 + sc * max(np.abs(np.r_[ad, ct])) + 0.12\n\n    fig = plt.figure(figsize=(TW, TW + 0.35))\n    ax = fig.add_axes([0.13, 0.11, 0.74, 0.74], projection=\"polar\")\n    ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)\n    ax.set_ylim(0, r_out + 0.02)\n\n    # provenance wedges and rim arcs\n    start = 0.0\n    for name, gl, c in groups:\n        a0 = (start - 0.5) / n_slots * 2 * np.pi; a1 = (start + len(gl) - 0.5) / n_slots * 2 * np.pi\n        arc = np.linspace(a0, a1, 160)\n        ax.fill_between(arc, 0.18, r_out, color=c, alpha=0.045, lw=0, zorder=0)\n        ax.plot(arc, np.full_like(arc, r_out), color=c, lw=2.6, solid_capstyle=\"butt\", zorder=5)\n        mid = (a0 + a1) / 2\n        ax.text(mid, r_out * 1.36, f\"{name}\\n{len(gl)} genes\", ha=\"center\", va=\"center\", fontsize=7.6,\n                color=c, linespacing=1.25, fontweight=\"bold\")\n        start += len(gl) + gap\n\n    # reference rings\n    ring = np.linspace(0, 2 * np.pi, 400)\n    for z, ls_, lw_ in ((-0.5, \"-\", 0.4), (0.0, (0, (3, 2)), 0.9), (0.5, \"-\", 0.4)):\n        ax.plot(ring, np.full_like(ring, r0 + sc * z), color=INK3 if z else INK2, lw=lw_, ls=ls_, zorder=1)\n    for z in (-0.5, 0.0, 0.5):\n        ax.text(np.deg2rad(-3.5), r0 + sc * z, f\"{z:+.1f}\" if z else \"0\", ha=\"center\", va=\"center\",\n                fontsize=5.4, color=INK3, bbox=dict(boxstyle=\"square,pad=0.05\", facecolor=SURFACE, edgecolor=\"none\"))\n\n    # one radial segment per gene: control to AD, coloured by direction\n    for a, r1, r2 in zip(ang, rC, rA):\n        ax.plot([a, a], [r1, r2], color=AD if r2 > r1 else CTRL, lw=1.6, alpha=0.38, solid_capstyle=\"butt\", zorder=2)\n    # smooth traces within each block\n    k = 0\n    for name, gl, c in groups:\n        sl = slice(k, k + len(gl))\n        ax.plot(ang[sl], rC[sl], color=CTRL, lw=1.0, zorder=3)\n        ax.plot(ang[sl], rA[sl], color=AD, lw=1.0, ls=(0, (3.5, 1.8)), zorder=3)\n        k += len(gl)\n    ax.scatter(ang, rC, s=9, color=CTRL, edgecolor=SURFACE, linewidth=0.3, zorder=4)\n    ax.scatter(ang, rA, s=11, marker=\"^\", color=AD, edgecolor=SURFACE, linewidth=0.3, zorder=4)\n\n    # names along their own spokes; a small dot marks independent replication\n    for a, g, c in zip(ang, genes, gcol):\n        phi = (90.0 - np.degrees(a)) % 360\n        flip = 90 < phi < 270\n        if rep(g):\n            ax.scatter([a], [r_out + 0.05], s=5, color=INK, zorder=6, clip_on=False)\n        ax.text(a, r_out + 0.10, g, rotation=phi - 180 if flip else phi, rotation_mode=\"anchor\",\n                ha=\"right\" if flip else \"left\", va=\"center\", fontsize=4.9, fontstyle=\"italic\", color=c,\n                fontweight=\"bold\" if g in both else \"normal\", clip_on=False)\n\n    ax.set_xticks([]); ax.set_yticks([])\n    ax.grid(False); ax.spines[\"polar\"].set_visible(False)\n    # centre key\n    n_up = int((ad > ct).sum())\n    ax.text(0, 0, f\"{len(genes)} genes\\n{n_up} higher in AD\\n{len(genes) - n_up} lower\", ha=\"center\", va=\"center\",\n            fontsize=6.8, color=INK, linespacing=1.35, zorder=6)\n    fig.legend(handles=[Line2D([], [], color=CTRL, lw=1.1, marker=\"o\", markersize=3.8, label=\"control mean\"),\n                        Line2D([], [], color=AD, lw=1.1, ls=(0, (3.5, 1.8)), marker=\"^\", markersize=4.2,\n                               label=\"AD mean\"),\n                        Line2D([], [], color=AD, lw=2.2, alpha=0.45, label=\"higher in AD\"),\n                        Line2D([], [], color=CTRL, lw=2.2, alpha=0.45, label=\"lower in AD\"),\n                        Line2D([], [], color=INK2, lw=0.9, ls=(0, (3, 2)), label=\"cohort mean ($z$ = 0)\"),\n                        Line2D([], [], marker=\"o\", ls=\"\", color=INK, markersize=2.6, label=\"replicates in unseen cohorts\")],\n               loc=\"lower center\", bbox_to_anchor=(0.5, 0.0), ncol=3, fontsize=6.4, frameon=False,\n               columnspacing=1.6)\n    fig.text(0.04, 0.985, \"Directional expression of every selected gene\", fontsize=8.8, color=INK, va=\"top\")\n    fig.text(0.04, 0.962, \"radius = mean $z$ in each diagnostic group; genes sorted within each block from most \"\n                          \"reduced to most increased in AD; shared genes in bold\", fontsize=6.4, color=INK3, va=\"top\")\n    save(fig, \"fig9_expression_profile\")\n\n\nfig_profile()\n\n# =====================================================================\n# Figure 10 -- how each latent dimension maps back onto genes\n# =====================================================================\ndef fig_latent_map():\n    pls = load(\"PLSDA_gene_significance.csv\")\n    cv = load(\"CVAE_gene_significance.csv\")\n    snr = load(\"latent_snr.csv\")\n    genes_all = list(load(\"gene_de_stats.csv\").gene)\n    gidx = {g: i for i, g in enumerate(genes_all)}\n    ps, cs = set(pls.loc[pls.significant, \"gene\"]), set(cv.loc[cv.significant, \"gene\"])\n    WP = np.abs(np.load(DATA / \"latent_gene_weights_PLS-DA.npy\")); WP = WP / WP.max()\n    WC = np.abs(np.load(DATA / \"latent_gene_weights_CVAE_neutral.npy\")); WC = WC / WC.max()\n    blocks = [(\"PLS-DA only\", ps - cs, PLS), (\"both\", ps & cs, ACCENT), (\"CVAE only\", cs - ps, CVAE_N)]\n    # within each block, genes grouped by the PLS-DA component that carries them most\n    rows = []\n    for name, gs_, c in blocks:\n        gl = sorted(gs_, key=lambda g: (int(np.argmax(WP[:, gidx[g]])), -WP[:, gidx[g]].max()))\n        rows += [(g, c, name) for g in gl]\n    half = int(np.ceil(len(rows) / 2))\n    parts = [rows[:half], rows[half:]]\n\n    row_in = 0.092\n    H = half * row_in + 2.75\n    fig = plt.figure(figsize=(TW, H))\n    top_frac = (half * row_in) / H\n    gs = fig.add_gridspec(2, 1, height_ratios=[half * row_in, 1.55], hspace=1.05 / ((half * row_in + 1.55) / 2),\n                          top=1 - 0.62 / H, bottom=0.40 / H)\n    upper = gs[0].subgridspec(1, 2, wspace=0.42)\n    cm_p, cm_c = SEQ_PLS, SEQ_CVAE_N\n    for k, part in enumerate(parts):\n        ax = fig.add_subplot(upper[k])\n        n = len(part)\n        cols = [gidx[g] for g, _, _ in part]\n        Mp, Mc = WP[:, cols].T, WC[:, cols].T          # rows = genes, cols = dimensions\n        ax.imshow(Mp, extent=(0.5, 8.5, n - 0.5, -0.5), cmap=cm_p, vmin=0, vmax=1, aspect=\"auto\",\n                  interpolation=\"nearest\")\n        ax.imshow(Mc, extent=(9.3, 17.3, n - 0.5, -0.5), cmap=cm_c, vmin=0, vmax=1, aspect=\"auto\",\n                  interpolation=\"nearest\")\n        for i, (g, c, name) in enumerate(part):\n            ax.add_patch(Rectangle((-0.35, i - 0.5), 0.6, 1.0, facecolor=c, edgecolor=SURFACE, lw=0.3))\n            if i and part[i - 1][2] != name:\n                ax.plot([-0.35, 17.3], [i - 0.5, i - 0.5], color=INK, lw=0.7)\n        ax.set_xlim(-0.4, 17.4); ax.set_ylim(n - 0.5, -0.5)\n        italic_ticks(ax, [g for g, _, _ in part], 5.2)\n        for t_, (_, c, _) in zip(ax.get_yticklabels(), part):\n            t_.set_color(c)\n        ax.set_xticks(list(range(1, 9)) + [x + 0.8 for x in range(9, 17)])\n        ax.set_xticklabels([f\"{i}\" for i in range(1, 9)] * 2, fontsize=5.4)\n        ax.xaxis.tick_top()\n        ax.tick_params(length=0, pad=1.5)\n        for s_ in ax.spines.values():\n            s_.set_visible(False)\n        ax.text(4.5, -2.6, \"PLS-DA LV\", ha=\"center\", va=\"bottom\", fontsize=6.6, color=PLS)\n        ax.text(13.3, -2.6, \"CVAE z (agnostic)\", ha=\"center\", va=\"bottom\", fontsize=6.6, color=CVAE_N)\n        if k == 0:\n            for x0f, cm_, lab, c in ((0.051, cm_p, \"PLS-DA |weight|\", PLS), (0.575, cm_c, \"CVAE |weight|\", CVAE_N)):\n                cax = ax.inset_axes([x0f, -0.035, 0.40, 0.010])\n                cb = fig.colorbar(plt.cm.ScalarMappable(cmap=cm_, norm=plt.Normalize(0, 1)), cax=cax,\n                                  orientation=\"horizontal\")\n                cb.set_ticks([0, 0.5, 1]); cb.ax.tick_params(labelsize=5.0, length=1.5, pad=1)\n                cb.outline.set_visible(False)\n                cax.set_xlabel(lab, fontsize=5.4, color=c, labelpad=1)\n            ax.annotate(\"a\", xy=(-0.30, 1.0), xycoords=\"axes fraction\", xytext=(0, 30), textcoords=\"offset points\",\n                        fontsize=9.5, fontweight=\"bold\", family=\"DejaVu Sans\", color=INK, va=\"bottom\",\n                        annotation_clip=False)\n            ax.annotate(\"How each latent dimension maps back onto the 111 selected genes\", xy=(0, 1.0),\n                        xycoords=\"axes fraction\", xytext=(0, 30), textcoords=\"offset points\", fontsize=8.6,\n                        color=INK, va=\"bottom\", annotation_clip=False)\n    fig.legend(handles=[Rectangle((0, 0), 1, 1, color=c, label=nm) for nm, _, c in blocks],\n               loc=\"upper right\", bbox_to_anchor=(0.995, 1 - 0.18 / H), ncol=1, fontsize=5.8, frameon=False,\n               title=\"selected by\", title_fontsize=5.8)\n\n    # ---- (b) signal-to-noise per dimension, all three arms ---------------------------\n    lower = gs[1].subgridspec(1, 1)\n    ax = fig.add_subplot(lower[0])\n    pos_ = ax.get_position(); ax.set_position([pos_.x0 + 0.04, pos_.y0, pos_.width - 0.04, pos_.height])\n    arms = [(\"PLS-DA\", PLS, \"PLS-DA (LV)\"), (\"CVAE_neutral\", CVAE_N, \"CVAE agnostic (z)\"),\n            (\"CVAE_true\", CVAE_T, \"CVAE conditioned (z)\")]\n    w = 0.26\n    for j, (tag, c, lab) in enumerate(arms):\n        d = snr[snr.model == tag].reset_index(drop=True)\n        x = np.arange(len(d)) + (j - 1) * w\n        ax.bar(x, d.snr, w * 0.9, color=c, lw=0, label=lab)\n        for xi, v in zip(x, d.snr):\n            ax.text(xi, v + 0.06, f\"{v:.1f}\", ha=\"center\", va=\"bottom\", fontsize=4.6, color=c)\n    ax.axhline(1.0, color=INK2, lw=0.7, ls=(0, (3, 2)))\n    ax.set_xticks(range(8)); ax.set_xticklabels([f\"dimension {i}\" for i in range(1, 9)], fontsize=6.2)\n    ax.set_ylabel(\"signal-to-noise\"); ax.set_ylim(0, snr.snr.max() * 1.22); ygrid(ax)\n    ax.legend(loc=\"upper left\", ncol=3, fontsize=6.2, frameon=False)\n    panel(ax, \"b\", \"Signal-to-noise of each latent dimension\",\n          \"mean |weight| of the 111 selected genes \u00f7 that of all other genes; dashed = parity; \"\n          \"heatmap weights scaled to each model's maximum\", dx=-0.075)\n    save(fig, \"fig10_latent_gene_map\")\n\n\nfig_latent_map()\n\n# =====================================================================\n# Figure 11 -- functional enrichment\n# =====================================================================\ndef fig_enrichment():\n    \"\"\"Enrichment as biology, every term shown. Terms are grouped into themes by the genes\n    that carry them; terms resting on one gene form gene-specific themes, drawn hollow and\n    marked as single-gene evidence rather than hidden.\"\"\"\n    SETS = [(\"plsda_down\", \"PLS-DA\\nlower\", PLS), (\"plsda_up\", \"PLS-DA\\nhigher\", PLS),\n            (\"cvae_down\", \"CVAE\\nlower\", CVAE_N), (\"cvae_up\", \"CVAE\\nhigher\", CVAE_N)]\n    pls = load(\"PLSDA_gene_significance.csv\"); cv = load(\"CVAE_gene_significance.csv\")\n    de = load(\"gene_de_stats.csv\").set_index(\"gene\")\n    ps, cs = set(pls.gene[pls.significant]), set(cv.gene[cv.significant])\n    prov = lambda g: ACCENT if (g in ps and g in cs) else PLS if g in ps else CVAE_N\n    D, stats_ = {}, {}\n    for key, _, _ in SETS:\n        d = pd.read_csv(DATA / \"enrichment\" / f\"{key}.csv\")\n        d = d.sort_values(\"adjusted_p_value\").drop_duplicates([\"source\", \"term_name\"]).copy()\n        d[\"genes\"] = d.intersections.fillna(\"\").apply(lambda s: frozenset(x for x in s.split(\",\") if x))\n        stats_[key] = (len(d), int((d.intersection_size >= 2).sum()))\n        D[key] = d\n\n    entries = [(k, r) for k in D for r in D[k].itertuples()]\n    multi_counts = {}\n    for k, r in entries:\n        if len(r.genes) >= 2:\n            multi_counts[r.genes] = multi_counts.get(r.genes, 0) + 1\n    cores = sorted(multi_counts, key=lambda s: (-multi_counts[s], len(s)))\n    HB = frozenset({\"HBB\", \"HBD\"})\n    MINOR = {\"TRMT5\", \"ELMO2\", \"POLR1D\", \"BDNF\"}           # one-off phenotype / enzyme terms\n\n    def core_of(k, r):\n        if len(r.genes) == 1:\n            g = next(iter(r.genes))\n            if g in HB:\n                return HB\n            return (\"ONE\", \"minor\" if g in MINOR else g)\n        if r.source == \"MIRNA\":\n            return \"MIRNA\"\n        if r.source == \"HP\":\n            return \"HP\"\n        best = None\n        for c in cores:\n            if c <= r.genes and len(r.genes) - len(c) <= 1 and multi_counts[c] >= 2:\n                if best is None or multi_counts[c] > multi_counts[best]:\n                    best = c\n        return best if best is not None else r.genes\n    themes = {}\n    for k, r in entries:\n        themes.setdefault(core_of(k, r), []).append((k, r))\n\n    NAMES = {HB: \"Hemoglobin and oxygen transport*\",\n             frozenset({\"NPTX2\", \"RPH3A\", \"BDNF\", \"TUBB2B\", \"CARTPT\", \"VGF\"}): \"Synaptic signalling\",\n             frozenset({\"CARTPT\", \"VGF\"}): \"Neuropeptide and neuroendocrine\",\n             frozenset({\"NPTX2\", \"VGF\"}): \"Anterior pituitary expression (HPA)\",\n             \"MIRNA\": \"microRNA target sets (miR-7, -892b, -4436b)\",\n             \"HP\": \"Human phenotype associations (HPO)\",\n             frozenset({\"CARTPT\", \"TNR\"}): \"Transmission of nerve impulse\",\n             frozenset({\"BDNF\", \"SLC2A3\"}): \"Neuronal glucose handling\",\n             frozenset({\"BDNF\", \"NKX2-5\"}): \"Myotube differentiation\",\n             frozenset({\"CHP1\", \"TOM1L1\"}): \"Protein autophosphorylation\",\n             (\"ONE\", \"RAF1\"): \"RAF1\u2013MAPK signalling complexes\",\n             (\"ONE\", \"TAB2\"): \"TAK1\u2013TAB2 inflammatory signalling\",\n             (\"ONE\", \"KIF5A\"): \"Kinesin KIF5A transport complexes\",\n             (\"ONE\", \"NEDD9\"): \"SMAD3\u2013HEF1 (NEDD9) complex\",\n             (\"ONE\", \"SRSF4\"): \"SR-protein splicing complex\",\n             (\"ONE\", \"TUBB2B\"): \"Tubulin and cortical malformation\",\n             (\"ONE\", \"DUSP6\"): \"MAP kinase phosphatase\",\n             (\"ONE\", \"NPAS2\"): \"Circadian ARNTL\u2013NPAS2 complex\",\n             (\"ONE\", \"ASF1B\"): \"ASF1 histone-chaperone complexes\",\n             (\"ONE\", \"CPSF6\"): \"CPSF6 mRNA 3'-end processing complexes\",\n             (\"ONE\", \"YEATS4\"): \"NuA4/Tip60 chromatin remodelling\",\n             (\"ONE\", \"MSL2\"): \"MSL histone-acetyltransferase complex\",\n             (\"ONE\", \"minor\"): \"Other terms (TRMT5, ELMO2, POLR1D, BDNF)\"}\n    RANK_MULTI = [\"Synaptic signalling\", \"Neuropeptide and neuroendocrine\", \"Anterior pituitary expression (HPA)\",\n                  \"Transmission of nerve impulse\", \"Neuronal glucose handling\", \"Myotube differentiation\",\n                  \"Protein autophosphorylation\", \"microRNA target sets (miR-7, -892b, -4436b)\",\n                  \"Human phenotype associations (HPO)\"]\n    T = []\n    for core, mem in themes.items():\n        single = isinstance(core, tuple)\n        name = NAMES.get(core, min(mem, key=lambda kr: kr[1].adjusted_p_value)[1].term_name[:38])\n        genes = sorted(set().union(*[r.genes for _, r in mem]))\n        per = {}\n        for k, r in mem:\n            n_, q_ = per.get(k, (0, 1.0)); per[k] = (n_ + 1, min(q_, r.adjusted_p_value))\n        rank = (RANK_MULTI.index(name) if name in RANK_MULTI else 50 if single else 40)\n        if core == HB:\n            rank = 100\n        T.append(dict(name=name, genes=genes, per=per, best=min(r.adjusted_p_value for _, r in mem),\n                      n=len(mem), mem=mem, single=single, rank=rank,\n                      home=min(per, key=lambda k: [s[0] for s in SETS].index(k))))\n    # single-gene themes ordered by the gene set they come from, then by strength\n    order_sets = [s[0] for s in SETS]\n    T.sort(key=lambda t: (t[\"rank\"] == 100, order_sets.index(t[\"home\"]), t[\"rank\"], t[\"best\"]))\n    nT = len(T)\n\n    fig = plt.figure(figsize=(12.4, 7.8))\n    Y0, Y1 = 0.115, 0.885\n\n    # ---- (a) theme x gene set ----------------------------------------------------------------\n    ax = fig.add_axes([0.155, Y0, 0.150, Y1 - Y0])\n    nmax = max(t[\"n\"] for t in T)\n    qmax = -np.log10(min(t[\"best\"] for t in T))\n    first_single = next(i for i, t in enumerate(T) if t[\"single\"])\n    for i, t in enumerate(T):\n        if i % 2 == 0:\n            ax.axhspan(i - 0.5, i + 0.5, color=\"#f4f4f7\", lw=0, zorder=0)\n        for j, (k, lab, c) in enumerate(SETS):\n            if k in t[\"per\"]:\n                n_, q_ = t[\"per\"][k]\n                s = 18 + 120 * np.sqrt(n_ / nmax)\n                lv = 0.35 + 0.65 * (-np.log10(q_)) / qmax\n                ax.scatter(j, i, s=s, color=tint(c, lv), edgecolor=c, linewidth=0.8, zorder=3)\n                ax.text(j + 0.30, i, f\"{n_}\", ha=\"left\", va=\"center\", fontsize=5.0, color=INK2)\n    ax.set_xlim(-0.5, 3.8); ax.set_ylim(nT - 0.5 + 0.9, -0.5)\n    ax.set_xticks(range(4)); ax.set_xticklabels([s[1] for s in SETS], fontsize=5.8)\n    for t_, s in zip(ax.get_xticklabels(), SETS):\n        t_.set_color(s[2])\n    ax.xaxis.tick_top(); ax.tick_params(length=0)\n    ax.set_yticks(range(nT)); ax.set_yticklabels([t[\"name\"] for t in T], fontsize=6.0)\n    for t_ in ax.get_yticklabels():\n        t_.set_color(INK)\n    for s_ in ax.spines.values():\n        s_.set_visible(False)\n    for j, (k, _, c) in enumerate(SETS):\n        ax.text(j, nT + 0.15, f\"{stats_[k][0]}\\nterms\", ha=\"center\", va=\"top\", fontsize=4.9, color=INK3,\n                linespacing=1.1)\n    fig.text(0.008, 0.965, \"a\", fontsize=9.5, fontweight=\"bold\", family=\"DejaVu Sans\", color=INK, va=\"bottom\")\n    fig.text(0.024, 0.965, \"Every enriched term, grouped into biological themes\", fontsize=8.8, color=INK,\n             va=\"bottom\")\n\n    # ---- (b) genes behind each theme ------------------------------------------------------------\n    axb = fig.add_axes([0.310, Y0, 0.225, Y1 - Y0], sharey=ax)\n    order = []\n    for t in T:\n        for g in t[\"genes\"]:\n            if g not in order:\n                order.append(g)\n    for i, t in enumerate(T):\n        if i % 2 == 0:\n            axb.axhspan(i - 0.5, i + 0.5, color=\"#f4f4f7\", lw=0, zorder=0)\n        xs = [order.index(g) for g in t[\"genes\"]]\n        axb.plot([min(xs), max(xs)], [i, i], color=RULE, lw=0.7, zorder=1)\n        for g in t[\"genes\"]:\n            both_ = g in ps and g in cs\n            axb.scatter(order.index(g), i, s=16, marker=\"D\" if both_ else \"o\", zorder=3,\n                        facecolor=prov(g), edgecolor=SURFACE, linewidth=0.5)\n    for x_, g in enumerate(order):\n        up_ = de.loc[g, \"logFC\"] > 0\n        axb.scatter(x_, nT - 0.5 + 0.45, marker=\"^\" if up_ else \"v\", s=8, color=AD if up_ else CTRL, lw=0,\n                    clip_on=False, zorder=4)\n    axb.set_xlim(-0.8, len(order) - 0.2)\n    axb.set_xticks(range(len(order)))\n    axb.set_xticklabels(order, rotation=90, fontsize=4.6, fontstyle=\"italic\")\n    for t_, g in zip(axb.get_xticklabels(), order):\n        t_.set_color(prov(g))\n    axb.xaxis.tick_top(); axb.tick_params(length=0, pad=1)\n    plt.setp(axb.get_yticklabels(), visible=False)\n    for s_ in axb.spines.values():\n        s_.set_visible(False)\n    fig.text(0.310, 0.965, \"b\", fontsize=9.5, fontweight=\"bold\", family=\"DejaVu Sans\", color=INK, va=\"bottom\")\n    fig.text(0.326, 0.965, \"The genes behind each theme\", fontsize=8.8, color=INK, va=\"bottom\")\n\n    # ---- (c) leading terms of each theme, with their genes ------------------------------------------\n    rows = []\n    for t in T:\n        mem = sorted(t[\"mem\"], key=lambda kr: kr[1].adjusted_p_value)\n        cap = 2 if (t[\"single\"] or t[\"n\"] <= 2) else 3\n        seen = []\n        for k, r in mem:\n            if r.term_name in seen:\n                continue\n            seen.append(r.term_name); rows.append((t, k, r))\n            if len(seen) == cap:\n                break\n    axc = fig.add_axes([0.735, Y0, 0.105, Y1 - Y0])\n    xmax = max(-np.log10(r.adjusted_p_value) for _, _, r in rows) * 1.08\n    prev = None\n    for yi, (t, k, r) in enumerate(rows):\n        c = PLS if k.startswith(\"plsda\") else CVAE_N\n        v = -np.log10(r.adjusted_p_value)\n        if t is not prev:\n            if prev is not None:\n                axc.axhline(yi - 0.5, color=RULE, lw=0.5, xmin=-1.55, xmax=2.5, clip_on=False)\n            prev = t\n        axc.hlines(yi, 0, v, color=c, lw=1.0, alpha=0.6, zorder=1)\n        mk = \"o\" if \"down\" in k else \"^\"\n        axc.scatter(v, yi, s=9 + 6 * min(r.intersection_size, 12), marker=mk, zorder=3,\n                    facecolor=c, edgecolor=SURFACE, linewidth=0.5)\n        nm = r.term_name if len(r.term_name) <= 48 else r.term_name[:47] + \"\u2026\"\n        axc.text(-0.03, yi, nm, transform=axc.get_yaxis_transform(), ha=\"right\", va=\"center\", fontsize=5.2,\n                 color=INK2)\n        gl = sorted(r.genes)\n        txt = \", \".join(gl if len(gl) <= 5 else gl[:5] + [f\"+{len(gl) - 5}\"])\n        axc.text(1.04, yi, f\"{txt}  \u00b7 {r.source}\", transform=axc.get_yaxis_transform(), ha=\"left\", va=\"center\",\n                 fontsize=4.9, fontstyle=\"italic\", color=c)\n    axc.axvline(-np.log10(0.05), color=INK3, lw=0.6, ls=(0, (3, 2)))\n    axc.set_xlim(0, xmax); axc.set_ylim(len(rows) - 0.5, -0.6)\n    axc.set_yticks([]); axc.set_xlabel(\"$-\\\\log_{10}$ FDR $q$\", fontsize=6.4); xgrid(axc)\n    axc.spines[\"left\"].set_visible(False); axc.tick_params(axis=\"x\", labelsize=5.8)\n    fig.text(0.555, 0.965, \"c\", fontsize=9.5, fontweight=\"bold\", family=\"DejaVu Sans\", color=INK, va=\"bottom\")\n    fig.text(0.571, 0.965, \"Leading terms of every theme, same order, with their genes\", fontsize=8.8,\n             color=INK, va=\"bottom\")\n\n    hl = [Line2D([], [], marker=\"o\", ls=\"\", color=PLS, markersize=4.2, label=\"PLS-DA set\"),\n          Line2D([], [], marker=\"o\", ls=\"\", color=CVAE_N, markersize=4.2, label=\"CVAE set\"),\n          Line2D([], [], marker=\"D\", ls=\"\", color=ACCENT, markersize=3.8, label=\"gene chosen by both\"),\n          Line2D([], [], marker=\"^\", ls=\"\", color=INK2, markersize=4.2, label=\"genes higher in AD\"),\n          Line2D([], [], marker=\"v\", ls=\"\", color=INK2, markersize=4.2, label=\"genes lower in AD\")]\n    fig.legend(handles=hl, loc=\"lower center\", bbox_to_anchor=(0.5, 0.022), ncol=6, fontsize=6.2, frameon=False)\n    fig.text(0.5, 0.004, ha=\"center\", s= \"Bubble area = number of terms in the theme (count beside it); fill depth = best FDR $q$. \"\n             \"Themes of one-gene terms are named by us from the terms' content. *Hemoglobin terms likely reflect \"\n             \"residual blood and are not interpreted.\", fontsize=5.8, color=INK3, style=\"italic\")\n    save(fig, \"fig11_enrichment\")\n\n\nfig_enrichment()"
open('/kaggle/working/scripts/merged_figures.py', 'w').write(SRC_MERGED_FIGURES)
print('wrote merged_figures.py', len(SRC_MERGED_FIGURES), 'chars')


## `scripts/validation_figures.py`


In [ ]:
SRC_VALIDATION_FIGURES = "\"\"\"Validation figures V1-V6 of paper3, exported unchanged from V12ADfigures.\nReads data/validation/ only.\"\"\"\n\n\"\"\"Shared visual language for every figure in the paper.\n\nOne module so that a panel letter, a tick, a bar gap or a significance bracket\nlooks the same wherever it appears -- the thing that separates a figure set from\na pile of plots. Everything here is layout and typography; the numbers come from\n../data and are never touched.\n\nColour is the data-viz reference palette used unchanged: categorical slots 1-3\n(blue / orange / aqua) for the three model arms, which is the documented\nall-pairs colourblind-safe subset, and the documented blue-red diverging pair\nwith a neutral grey midpoint for up/down regulation. Identity never rests on\ncolour alone -- every series also carries a marker shape, a direct label or a\nlegend entry.\n\"\"\"\nimport os\nimport warnings\nfrom pathlib import Path\n\nimport matplotlib as mpl\n# PAPER_INLINE is set when this same code runs inside the Kaggle notebook, where\n# the figures must render in the output rather than only be written to disk.\nINLINE = bool(os.environ.get(\"PAPER_INLINE\"))\nif not INLINE:\n    mpl.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom matplotlib.colors import LinearSegmentedColormap\nfrom matplotlib.lines import Line2D\n\nwarnings.filterwarnings(\"ignore\")\n\n# Paths are overridable so that the notebook can point them at /kaggle/working\n# and run this file unmodified -- one source of truth for every figure.\nif \"PAPER_DATA\" in os.environ:\n    DATA = Path(os.environ[\"PAPER_DATA\"])\n    FIG = Path(os.environ.get(\"PAPER_FIGS\", DATA / \"figures\"))\nelse:\n    ROOT = Path(__file__).resolve().parent.parent\n    DATA = ROOT / \"data\"\n    FIG = ROOT / \"figures\"\nFIG.mkdir(parents=True, exist_ok=True)\n\n# Target printed width. The LaTeX text block is 6.27 in wide, so a figure drawn\n# much wider than this is downscaled by \\includegraphics and every font in it\n# shrinks with the drawing -- the usual reason a figure set looks amateur on the\n# page. Everything here is laid out at TW and lands at ~0.9 scale.\nTW = 6.9\n\n# ---- palette -------------------------------------------------------------\n# A cool, low-chroma set in the register of a printed journal figure: navy,\n# wine and violet rather than the primary blue / orange / green of a screen\n# palette. The three model arms stay separable for the common forms of colour\n# blindness -- they differ in hue *and* in lightness (navy 30%, wine 42%,\n# violet 55%), so they also survive a greyscale print -- and identity never\n# rests on colour alone, since every series carries a marker or a direct label.\nPLS     = \"#173451\"   # categorical slot 1 -- navy\nCVAE_T  = \"#8f3a4f\"   # slot 2 -- wine\nCVAE_N  = \"#4f4478\"   # slot 3 -- muted violet\nAD      = \"#8f3a4f\"   # diverging warm pole -- wine\nCTRL    = \"#173451\"   # diverging cool pole -- navy\nUP, DOWN = AD, CTRL\nMID     = \"#8a8a92\"   # neutral midpoint ink, faintly cool\nACCENT  = \"#1f6372\"   # reserved for \"found by both\" -- petrol blue\nINK     = \"#17171c\"\nINK2    = \"#54545e\"\nINK3    = \"#8a8a92\"\nRULE    = \"#c9c9d1\"\nGRID    = \"#e9e9ef\"\nSURFACE = \"#ffffff\"\n\n# Diverging ramp for probability surfaces: cool pole -> neutral grey -> warm\n# pole. Deliberately light at both ends -- this ramp is only ever a background\n# under data marks, and a saturated pole swallows them.\nDIVERGING = LinearSegmentedColormap.from_list(\n    \"ctrl_ad\", [\"#7d92ad\", \"#a8b5c9\", \"#d3d9e3\", \"#eeeef1\",\n                \"#e3ccd3\", \"#c99aa8\", \"#a96d80\"])\n\n# Sequential ramps for the confusion matrices, built from the two arm colours so\n# the matrices sit inside the same muted palette as everything else instead of\n# borrowing matplotlib's saturated \"Blues\"/\"Oranges\".\nSEQ_PLS = LinearSegmentedColormap.from_list(\n    \"seq_pls\", [\"#ffffff\", \"#e7ecf2\", \"#c3d1e0\", \"#93aac6\", \"#5c7fa6\", PLS, \"#173252\"])\nSEQ_CVAE = LinearSegmentedColormap.from_list(\n    \"seq_cvae\", [\"#ffffff\", \"#f5e9ec\", \"#e6c9d1\", \"#d0a1ae\", \"#b06c81\", CVAE_T, \"#63242f\"])\n\nMODEL_COLOR = {\"pls\": PLS, \"cvae_true\": CVAE_T, \"cvae_neutral\": CVAE_N,\n               \"PLS-DA\": PLS, \"CVAE_true\": CVAE_T, \"CVAE_neutral\": CVAE_N}\nMODEL_LABEL = {\"pls\": \"PLS-DA\",\n               \"cvae_true\": \"CVAE (label-conditioned)\",\n               \"cvae_neutral\": \"CVAE (label-agnostic)\",\n               \"PLS-DA\": \"PLS-DA\",\n               \"CVAE_true\": \"CVAE (label-conditioned)\",\n               \"CVAE_neutral\": \"CVAE (label-agnostic)\"}\nGROUP_COLOR = {\"AD\": AD, \"Control\": CTRL}\nGROUP_MARKER = {\"AD\": \"^\", \"Control\": \"o\"}\n\n# database sources in the g:Profiler output\nSRC_COLOR = {\"GO:MF\": \"#24486b\", \"GO:BP\": \"#2f7d8c\", \"GO:CC\": \"#6f6396\",\n             \"CORUM\": \"#8f3a4f\", \"KEGG\": \"#4b7ca8\", \"REAC\": \"#57516e\",\n             \"WP\": \"#3f6d80\", \"MIRNA\": \"#a3798f\", \"TF\": \"#54545e\", \"HP\": \"#7a4a5e\"}\n\nplt.rcParams.update({\n    \"font.family\": \"serif\",\n    \"font.serif\": [\"STIXGeneral\", \"Latin Modern Roman\", \"DejaVu Serif\", \"Times New Roman\"],\n    \"mathtext.fontset\": \"stix\",\n    \"font.size\": 8,\n    \"axes.titlesize\": 9,\n    \"axes.titlelocation\": \"left\",\n    \"axes.titlepad\": 7,\n    \"axes.labelsize\": 8,\n    \"axes.labelpad\": 3.5,\n    \"axes.labelcolor\": INK,\n    \"axes.edgecolor\": RULE,\n    \"axes.linewidth\": 0.6,\n    \"axes.grid\": False,\n    \"grid.color\": GRID,\n    \"grid.linewidth\": 0.5,\n    \"axes.axisbelow\": True,\n    \"axes.spines.top\": False,\n    \"axes.spines.right\": False,\n    \"xtick.color\": INK2, \"ytick.color\": INK2,\n    \"xtick.labelsize\": 7, \"ytick.labelsize\": 7,\n    \"xtick.major.size\": 2.6, \"ytick.major.size\": 2.6,\n    \"xtick.major.width\": 0.6, \"ytick.major.width\": 0.6,\n    \"xtick.major.pad\": 2.5, \"ytick.major.pad\": 2.5,\n    \"legend.fontsize\": 7,\n    \"legend.frameon\": False,\n    \"legend.handlelength\": 1.1,\n    \"legend.handletextpad\": 0.5,\n    \"legend.columnspacing\": 1.2,\n    \"legend.borderaxespad\": 0.2,\n    \"lines.linewidth\": 1.4,\n    \"lines.solid_capstyle\": \"round\",\n    \"patch.linewidth\": 0,\n    \"figure.facecolor\": SURFACE,\n    \"axes.facecolor\": SURFACE,\n    \"savefig.facecolor\": SURFACE,\n    \"pdf.fonttype\": 42,\n})\n\n\n# ---- primitives ----------------------------------------------------------\ndef save(fig, name):\n    \"\"\"Vector PDF for pdflatex, 400-dpi PNG for the pandoc/docx build.\"\"\"\n    for ext, kw in ((\"pdf\", {}), (\"png\", {\"dpi\": 400})):\n        fig.savefig(FIG / f\"{name}.{ext}\", bbox_inches=\"tight\",\n                    pad_inches=0.03, **kw)\n    if INLINE:\n        try:\n            from IPython.display import display\n            display(fig)\n        except ImportError:      # running the notebook's cells outside a kernel\n            pass\n    plt.close(fig)\n    print(f\"  {FIG.name}/{name}.pdf + .png\")\n\n\ndef panel(ax, letter, title=None, subtitle=None, dx=-0.13, dy=1.0):\n    \"\"\"Panel letter in sans-serif, hung outside the axes at a fixed offset so\n    the letters line up down the figure regardless of tick-label width.\n\n    Title and subtitle are both annotations rather than `set_title`, because a\n    real title always sits at one fixed height and would land on top of the\n    subtitle whenever both are present.\n    \"\"\"\n    ax.annotate(letter, xy=(dx, dy), xycoords=\"axes fraction\",\n                xytext=(0, 2), textcoords=\"offset points\",\n                fontsize=9.5, fontweight=\"bold\", family=\"DejaVu Sans\",\n                color=INK, ha=\"left\", va=\"bottom\", annotation_clip=False)\n    if title:\n        ax.annotate(title, xy=(0, 1.0), xycoords=\"axes fraction\",\n                    xytext=(0, 13 if subtitle else 3), textcoords=\"offset points\",\n                    fontsize=8.6, color=INK, ha=\"left\", va=\"bottom\",\n                    annotation_clip=False)\n    if subtitle:\n        ax.annotate(subtitle, xy=(0, 1.0), xycoords=\"axes fraction\",\n                    xytext=(0, 3), textcoords=\"offset points\",\n                    fontsize=6.8, color=INK3, ha=\"left\", va=\"bottom\",\n                    annotation_clip=False)\n\n\ndef ygrid(ax):\n    ax.grid(axis=\"y\", visible=True)\n    ax.grid(axis=\"x\", visible=False)\n\n\ndef xgrid(ax):\n    ax.grid(axis=\"x\", visible=True)\n    ax.grid(axis=\"y\", visible=False)\n\n\ndef despine(ax, left=True, bottom=True):\n    ax.spines[\"left\"].set_visible(left)\n    ax.spines[\"bottom\"].set_visible(bottom)\n\n\ndef load(name):\n    return pd.read_csv(DATA / name)\n\n\ndef scal(df=None):\n    \"\"\"summary_scalars.csv -> dict, numeric where possible.\"\"\"\n    df = df if df is not None else load(\"summary_scalars.csv\")\n    out = {}\n    for _, r in df.iterrows():\n        try:\n            out[r[\"key\"]] = float(r[\"value\"])\n        except (TypeError, ValueError):\n            out[r[\"key\"]] = r[\"value\"]\n    return out\n\n\ndef stars(p):\n    return \"***\" if p < 1e-3 else \"**\" if p < 1e-2 else \"*\" if p < 0.05 else \"n.s.\"\n\n\ndef pmath(p, prefix=\"p\"):\n    \"\"\"`$p = ...$` with a real times-ten-to-the exponent, not `1e-05`.\"\"\"\n    if p >= 1e-3:\n        return f\"${prefix} = {p:.3f}$\"\n    if not p > 1e-300:          # V7: with thousands of pairs a p-value can underflow to 0\n        return f\"${prefix} < 10^{{-300}}$\"\n    e = int(np.floor(np.log10(p)))\n    return f\"${prefix} = {p/10**e:.1f}\\\\times 10^{{{e}}}$\"\n\n\ndef italic_ticks(ax, names, fontsize=6.0, axis=\"y\"):\n    \"\"\"Italic gene symbols as real text, so hyphens stay hyphens and digits do\n    not pick up math-italic spacing (`HLA-DQA2`, not `HLA - DQA2`).\"\"\"\n    ticks = np.arange(len(names))\n    if axis == \"y\":\n        ax.set_yticks(ticks)\n        ax.set_yticklabels(names, fontsize=fontsize, fontstyle=\"italic\")\n    else:\n        ax.set_xticks(ticks)\n        ax.set_xticklabels(names, fontsize=fontsize, fontstyle=\"italic\")\n\n\ndef bracket(ax, x0, x1, y, text, tick=None, fontsize=7.2, color=INK):\n    \"\"\"Significance bracket that never overlaps the marks it spans.\"\"\"\n    tick = tick if tick is not None else abs(y) * 0.02 + 1e-9\n    ax.plot([x0, x0, x1, x1], [y - tick, y, y, y - tick],\n            color=color, lw=0.7, clip_on=False, solid_capstyle=\"butt\")\n    ax.text((x0 + x1) / 2, y + tick * 0.5, text, ha=\"center\", va=\"bottom\",\n            fontsize=fontsize, color=color, clip_on=False)\n\n\ndef boxstrip(ax, values, pos, color, width=0.44, jitter=0.055, seed=0, size=4):\n    \"\"\"A box over its own points: the distribution and its summary together.\"\"\"\n    rs = np.random.RandomState(int(round(seed)))\n    ax.scatter(rs.normal(pos, jitter, len(values)), values, s=size, color=color,\n               alpha=0.28, linewidth=0, zorder=1, rasterized=True)\n    bp = ax.boxplot([values], positions=[pos], widths=width, showfliers=False,\n                    patch_artist=True, zorder=3, manage_ticks=False)\n    bp[\"boxes\"][0].set(facecolor=SURFACE, edgecolor=color, linewidth=1.2, alpha=0.95)\n    for part in (\"whiskers\", \"caps\", \"medians\"):\n        for a in bp[part]:\n            a.set(color=color, linewidth=1.2)\n    return bp\n\n\ndef swatches(labels_colors, marker=\"s\", size=5):\n    return [Line2D([], [], marker=marker, ls=\"\", color=c, markersize=size, label=l)\n            for l, c in labels_colors]\n\n\ndef repel_labels(ax, xs, ys, labels, fontsize=5.4, color=None, italic=True,\n                 clearance=1.6, leader_color=None):\n    \"\"\"Place a label for every point without letting any two labels touch.\n\n    A dependency-free replacement for adjustText: each label is tried at a ring\n    of candidate offsets, in decreasing order of preference, and the first\n    offset whose estimated box overlaps neither an already-placed label nor its\n    own point is taken. Everything is measured in typographic points, which is\n    exactly the unit `annotate` takes for `xytext`, so no round trip through\n    data coordinates is needed and the result is stable at any figure size.\n    \"\"\"\n    color = INK if color is None else color\n    leader_color = RULE if leader_color is None else leader_color\n    fig = ax.figure\n    ppd = fig.dpi / 72.0                     # points -> display pixels\n    pts = ax.transData.transform(np.column_stack([xs, ys]))\n\n    # candidate offsets: near ring first, then progressively further out\n    cands = []\n    for radius in (7, 11, 16, 22, 30, 40, 52, 66, 82, 100):   # V7: wider search\n        for ang in range(0, 360, 30):\n            a = np.deg2rad(ang)\n            cands.append((radius * np.cos(a), radius * np.sin(a)))\n\n    placed = []                              # display-space boxes already taken\n    leg = ax.get_legend()                    # a label under the key is unreadable\n    if leg is not None:\n        bb = leg.get_window_extent()\n        placed.append((bb.x0, bb.y0, bb.x1, bb.y1))\n    order = np.argsort(-np.asarray(ys))      # most significant genes get first pick\n\n    def overlaps(box):\n        x0, y0, x1, y1 = box\n        for p0, q0, p1, q1 in placed:\n            if x0 < p1 and p0 < x1 and y0 < q1 and q0 < y1:\n                return True\n        return False\n\n    axbb = ax.get_window_extent()            # V7: labels must stay inside the axes\n    inside = lambda b: (b[0] >= axbb.x0 and b[2] <= axbb.x1 and\n                        b[1] >= axbb.y0 and b[3] <= axbb.y1)\n    forced = 0\n    for i in order:\n        text = labels[i]\n        w = len(text) * fontsize * 0.56 * ppd + clearance * ppd\n        h = fontsize * 1.25 * ppd + clearance * ppd\n        px, py = pts[i]\n        boxes = []\n        for dx, dy in cands:\n            cx = px + dx * ppd + (w / 2 - clearance) * (1 if dx >= 0 else -1)\n            cy = py + dy * ppd\n            boxes.append(((cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2), dx, dy))\n        ok = [t for t in boxes if inside(t[0]) and not overlaps(t[0])]\n        if not ok:                           # never drop a gene: least-bad inside spot\n            forced += 1\n            ok = [t for t in boxes if inside(t[0])] or boxes\n        for box, dx, dy in ok[:1]:\n            if True:\n                placed.append(box)\n                ax.annotate(text, (xs[i], ys[i]), xytext=(dx, dy),\n                            textcoords=\"offset points\", fontsize=fontsize,\n                            color=color, fontstyle=\"italic\" if italic else \"normal\",\n                            ha=\"left\" if dx >= 0 else \"right\", va=\"center\",\n                            zorder=5, annotation_clip=False,\n                            arrowprops=dict(arrowstyle=\"-\", color=leader_color,\n                                            lw=0.35, shrinkA=0.5, shrinkB=1.5))\n    if forced:\n        print(f\"  repel_labels: {forced} label(s) had no collision-free spot\")\n\n\ndef kde2d(xy, grid_x, grid_y, bw_scale=1.0):\n    \"\"\"Gaussian KDE evaluated on a grid, without a scipy import at call sites.\"\"\"\n    from scipy.stats import gaussian_kde\n    k = gaussian_kde(xy.T)\n    k.set_bandwidth(k.factor * bw_scale)\n    XX, YY = np.meshgrid(grid_x, grid_y)\n    return k(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)\n\n# ======================================================================\n# V12 figure set -- reads the saved results of the merge, V9, V10 and V11\n# notebooks and draws the project's publication figures. Nothing is re-fitted.\n# ======================================================================\nimport glob, json, re, os\nimport numpy as np, pandas as pd\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import FancyBboxPatch, Rectangle\nfrom matplotlib.lines import Line2D\nfrom matplotlib.colors import LinearSegmentedColormap\n\nINPUT_ROOT = os.environ.get(\"V12_INPUT\", \"/kaggle/input\")\n\n\ndef src(slug, name):\n    \"\"\"Path of `name` written by the notebook `slug` (Kaggle mounts each input\n    notebook's output under /kaggle/input/...).\"\"\"\n    hits = sorted(glob.glob(f\"{INPUT_ROOT}/**/{name}\", recursive=True))\n    mine = [h for h in hits if f\"/{slug}/\" in h or f\"/{slug}\" in os.path.dirname(h)]\n    hits = mine or hits\n    if not hits:\n        raise FileNotFoundError(f\"{name} from {slug}: attach that notebook's output as an input\")\n    return hits[0]\n\n\nrd = lambda slug, name, **kw: pd.read_csv(src(slug, name), **kw)\nM, V9, V10, V11 = \"ad-merged-geo-cohorts\", \"v9adexternal\", \"v10adgenevalidation\", \"v11adrobustness\"\n\n# ---- shared vocabulary ------------------------------------------------------\nTRAIN = [\"GSE48350\", \"GSE33000\", \"GSE118553\", \"GSE84422\"]\nHOLDOUT = \"GSE132903\"\nVALID = [\"GSE15222\", \"GSE29378\", \"GSE122063\", \"GSE36980\"]\nRNASEQ = \"GSE125583\"\nSEVERITY = [\"GSE106241\", \"GSE1297\"]\nDXC = [HOLDOUT] + VALID + [RNASEQ]                 # every AD-vs-control test cohort\nREGION = {\"GSE48350\": \"4 regions (HC, EC, SFG, PCG)\", \"GSE33000\": \"prefrontal cortex\",\n          \"GSE118553\": \"temporal, entorhinal, frontal cx\", \"GSE84422\": \"15 cortical / hippocampal\",\n          \"GSE132903\": \"middle temporal gyrus\", \"GSE15222\": \"temporal cortex\",\n          \"GSE29378\": \"hippocampus (CA1, CA3)\", \"GSE122063\": \"frontal + temporal cortex\",\n          \"GSE36980\": \"frontal cortex\", \"GSE125583\": \"fusiform gyrus\",\n          \"GSE106241\": \"inferior temporal cortex\", \"GSE1297\": \"hippocampus\"}\nPLATFORM = {\"GPL570\": \"Affymetrix U133 Plus 2\", \"GPL4372\": \"Agilent/Rosetta 44K\",\n            \"GPL10558\": \"Illumina HT-12 v4\", \"GPL96\": \"Affymetrix U133A\",\n            \"GPL2700\": \"Illumina RefSeq-8\", \"GPL6947\": \"Illumina HT-12 v3\",\n            \"GPL16699\": \"Agilent 8x60K\", \"GPL6244\": \"Affymetrix HuGene 1.0 ST\",\n            \"GPL24170\": \"Agilent 8x60K custom\", \"GPL16791\": \"RNA-seq (HiSeq 2500)\"}\nROLE_COLOR = {\"training\": PLS, \"external test\": AD, \"validation\": ACCENT,\n              \"RNA-seq validation\": CVAE_N, \"severity\": MID}\nSET_COLOR = {\"PLS-DA\": PLS, \"CVAE\": CVAE_N, \"Core\": ACCENT}\nSET_MARK = {\"PLS-DA\": \"o\", \"CVAE\": \"s\", \"Core\": \"D\"}\nCELL_ORDER = [\"neurons\", \"astrocytes\", \"microglia\", \"oligodendrocytes\", \"OPCs\", \"endothelial\"]\n\n\ndef short(c):\n    return c.replace(\"GSE\", \"GSE\u2009\") if False else c\n\n\ndef tint(hex_color, alpha):\n    r, g, b = (int(hex_color[i:i + 2], 16) / 255 for i in (1, 3, 5))\n    return (1 - alpha + alpha * r, 1 - alpha + alpha * g, 1 - alpha + alpha * b)\n\n\ndef rnaseq_counts():\n    \"\"\"AD / control donors of GSE125583, read from V11's own log.\"\"\"\n    try:\n        L = json.load(open(src(V11, \"v11adrobustness.log\")))\n        t = \"\".join(x.get(\"data\", \"\") for x in L)\n        m = re.search(r\"GSE125583: \\d+ arrays -> (\\d+) donors.*?'AD': (\\d+), 'Control': (\\d+)\", t)\n        if m:\n            return int(m.group(1)), int(m.group(2)), int(m.group(3))\n    except Exception:\n        pass\n    # fallback: GEO's own sample records (same age >= 60 rule as V11)\n    import gzip, requests\n    url = \"https://ftp.ncbi.nlm.nih.gov/geo/series/GSE125nnn/GSE125583/matrix/GSE125583_series_matrix.txt.gz\"\n    r = requests.get(url, headers={\"User-Agent\": \"ad-merged-geo-cohorts (kaggle)\"}, timeout=180)\n    r.raise_for_status()\n    txt = gzip.decompress(r.content).decode(\"utf-8\", \"ignore\")\n    rows = {}\n    for line in txt.splitlines():\n        if line.startswith(\"!Sample_characteristics_ch1\"):\n            vals = [v.strip('\"') for v in line.split(\"\\t\")[1:]]\n            key = vals[0].split(\":\")[0].strip().lower()\n            rows[key] = [v.split(\":\", 1)[1].strip() if \":\" in v else \"\" for v in vals]\n    dx = np.array(rows[\"diagnosis\"]); age = pd.to_numeric(pd.Series(rows[\"age\"]).str.extract(r\"(\\d+)\")[0], errors=\"coerce\").values\n    keep = np.isnan(age) | (age >= 60)          # V11 keeps donors with no recorded age\n    nad = int(((dx == \"Alzheimer's disease\") & keep).sum()); nct = int(((dx == \"control\") & keep).sum())\n    return nad + nct, nad, nct\n\n\ndef star(p):\n    return \"***\" if p < 1e-3 else \"**\" if p < 1e-2 else \"*\" if p < 0.05 else \"\"\n\n\ndef box(ax, x, y, w, h, text, face, edge, fs=6.6, weight=\"normal\", ha=\"center\", lw=0.9):\n    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=\"round,pad=0.012,rounding_size=0.018\",\n                                facecolor=face, edgecolor=edge, linewidth=lw, zorder=2))\n    tx = x + w / 2 if ha == \"center\" else x + 0.012\n    ax.text(tx, y + h / 2, text, ha=ha, va=\"center\", fontsize=fs, color=INK, zorder=3,\n            fontweight=weight, linespacing=1.25)\n\n\ndef arrow(ax, p0, p1, color=INK3, rad=0.0, lw=0.9):\n    ax.annotate(\"\", xy=p1, xytext=p0, zorder=1,\n                arrowprops=dict(arrowstyle=\"-|>\", color=color, lw=lw, shrinkA=1, shrinkB=1,\n                                mutation_scale=9, connectionstyle=f\"arc3,rad={rad}\"))\n\n\ndef repel_avoiding(ax, xs, ys, labels, ox, oy, fontsize=5.8, clearance=3.2):\n    \"\"\"Like figstyle.repel_labels, but every plotted point is also an obstacle,\n    so a label never sits on a marker; leader lines join label and point.\"\"\"\n    fig = ax.figure; ppd = fig.dpi / 72.0\n    pts = ax.transData.transform(np.column_stack([xs, ys]))\n    obs = ax.transData.transform(np.column_stack([ox, oy]))\n    placed = [(x - 3.5 * ppd, y - 3.5 * ppd, x + 3.5 * ppd, y + 3.5 * ppd) for x, y in obs]\n    leg = ax.get_legend()\n    if leg is not None:\n        bb = leg.get_window_extent(); placed.append((bb.x0, bb.y0, bb.x1, bb.y1))\n    axbb = ax.get_window_extent()\n    cands = [(r * np.cos(np.deg2rad(a)), r * np.sin(np.deg2rad(a)))\n             for r in (8, 12, 17, 23, 30, 38, 48, 60, 74, 90) for a in range(0, 360, 20)]\n    for i in np.argsort(-np.asarray(ys)):\n        t = labels[i]; w = len(t) * fontsize * 0.56 * ppd + clearance * ppd; h = fontsize * 1.3 * ppd + clearance * ppd\n        px, py = pts[i]; best = None\n        for dx, dy in cands:\n            cx = px + dx * ppd + (w / 2) * (1 if dx >= 0 else -1); cy = py + dy * ppd\n            b = (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)\n            if not (b[0] >= axbb.x0 and b[2] <= axbb.x1 and b[1] >= axbb.y0 and b[3] <= axbb.y1):\n                continue\n            if not any(b[0] < q[2] and q[0] < b[2] and b[1] < q[3] and q[1] < b[3] for q in placed):\n                best = (b, dx, dy); break\n        if best is None:\n            continue\n        b, dx, dy = best; placed.append(b)\n        ax.annotate(t, (xs[i], ys[i]), xytext=(dx, dy), textcoords=\"offset points\", fontsize=fontsize,\n                    color=INK, fontstyle=\"italic\", ha=\"left\" if dx >= 0 else \"right\", va=\"center\", zorder=6,\n                    arrowprops=dict(arrowstyle=\"-\", color=INK3, lw=0.4, shrinkA=0.5, shrinkB=2.5))\n\n\n# =====================================================================\n# Figure 1 -- study design and cohorts\n# =====================================================================\ndef fig1_design():\n    ct = rd(M, \"cohort_table.csv\"); ct = ct[ct.cohort != \"TOTAL\"]\n    vc = rd(V10, \"validation_cohorts.csv\")\n    sel = rd(V9, \"loco_selected_genes.csv\")\n    n_pls = sel[(sel.method == \"PLS-DA\")].gene.nunique(); n_cvae = sel[(sel.method == \"CVAE\")].gene.nunique()\n    gv = rd(V10, \"gene_validation_table.csv\"); n_core = int(gv.sets.str.contains(\"Core\").sum())\n    rs = rnaseq_counts()\n\n    rows = []\n    for _, r in ct.iterrows():\n        role = \"external test\" if r.cohort == HOLDOUT else \"training\"\n        rows.append((r.cohort, role, int(r.donors), int(r.AD_donors), int(r.control_donors), r.platform))\n    for _, r in vc.iterrows():\n        if r.cohort == HOLDOUT:\n            continue\n        role = \"severity\" if r.cohort in SEVERITY else \"validation\"\n        rows.append((r.cohort, role, int(r.donors), int(r.AD), int(r.control), r.platform))\n    rows.append((RNASEQ, \"RNA-seq validation\", rs[0], rs[1], rs[2], \"GPL16791\"))\n    order = {\"training\": 0, \"external test\": 1, \"validation\": 2, \"RNA-seq validation\": 3, \"severity\": 4}\n    rows.sort(key=lambda r: (order[r[1]], -np.nan_to_num(r[2])))\n\n    fig = plt.figure(figsize=(TW, 10.2))\n    # ---- (a) flow, drawn across the full width ----------------------------\n    ax = fig.add_axes([0.0, 0.777, 1.0, 0.201]); ax.set_xlim(-0.02, 1.01); ax.set_ylim(0, 1); ax.axis(\"off\")\n    ax.text(0.0, 1.0, \"a\", fontsize=9.5, fontweight=\"bold\", family=\"DejaVu Sans\", va=\"top\")\n    ax.text(0.03, 1.0, \"Study design\", fontsize=8.6, color=INK, va=\"top\")\n    ax.text(0.03, 0.90, \"each step is its own Kaggle notebook; the test cohorts never touch training or gene selection\",\n            fontsize=6.6, color=INK3, va=\"top\")\n    ctd = ct.set_index(\"cohort\")\n    tr_txt = \"Training cohorts\\n\" + \"\\n\".join(f\"{c}  n={int(ctd.loc[c, 'donors'])}\" for c in TRAIN)\n    yc = 0.40\n    box(ax, 0.000, yc - 0.20, 0.160, 0.40, tr_txt, tint(PLS, 0.10), PLS, fs=6.0)\n    box(ax, 0.200, yc - 0.14, 0.155, 0.28, \"Preprocessing\\nprobe \u2192 gene\\nage \u2265 60, z per cohort\",\n        tint(MID, 0.10), RULE, fs=6.0)\n    box(ax, 0.395, yc - 0.15, 0.170, 0.30, \"PLS-DA  &  \u03b2-CVAE\\n8-d latent space\\n20 donor-grouped runs\\n10 permuted-label runs\",\n        tint(MID, 0.10), RULE, fs=6.0)\n    box(ax, 0.605, yc - 0.15, 0.130, 0.30, f\"Frozen gene lists\\nPLS-DA {n_pls} \u00b7 CVAE {n_cvae}\\nCore {n_core}\",\n        tint(ACCENT, 0.12), ACCENT, fs=6.0)\n    for x0, x1 in ((0.160, 0.200), (0.355, 0.395), (0.565, 0.605)):\n        arrow(ax, (x0 + 0.004, yc), (x1 - 0.004, yc), color=INK2, lw=1.0)\n    outs = [(0.66, f\"External test \u00b7 {HOLDOUT}\\nAUC and age checks\", AD),\n            (0.40, \"Gene validation\\n5 cohorts + 2 severity cohorts\", ACCENT),\n            (0.14, \"Robustness\\ncell mix + RNA-seq\", CVAE_N)]\n    for y0, t, c in outs:\n        box(ax, 0.785, y0 - 0.10, 0.20, 0.20, t, tint(c, 0.12), c, fs=6.0)\n    arrow(ax, (0.48, yc + 0.155), (0.785, 0.66), rad=-0.22, color=INK2, lw=1.0)\n    ax.text(0.62, 0.80, \"trained models\", fontsize=5.8, color=INK3, style=\"italic\", ha=\"center\")\n    arrow(ax, (0.739, yc), (0.781, 0.40), color=INK2, lw=1.0)\n    arrow(ax, (0.739, yc - 0.05), (0.781, 0.16), rad=-0.15, color=INK2, lw=1.0)\n    ax.text(0.0, -0.03, \"Notebooks:  ad-merged-geo-cohorts  \u203a  V9ADexternal  \u203a  V10ADgenevalidation  \u203a  \"\n                        \"V11ADrobustness  \u203a  V14ADfigures\", fontsize=5.8, color=INK3)\n\n    # ---- (b) cohorts ------------------------------------------------------\n    ax = fig.add_axes([0.16, 0.307, 0.52, 0.395])\n    y = np.arange(len(rows))[::-1]\n    maxn = max(np.nan_to_num(r[2]) for r in rows)\n    for yi, (c, role, n, nad, nct, plat) in zip(y, rows):\n        if role == \"severity\":\n            ax.barh(yi, n, 0.62, color=tint(MID, 0.45), edgecolor=SURFACE)\n        else:\n            ax.barh(yi, nad, 0.62, color=AD, edgecolor=SURFACE)\n            ax.barh(yi, nct, 0.62, left=nad, color=CTRL, edgecolor=SURFACE)\n        ax.text(n + maxn * 0.015, yi, f\"{int(n)}\", va=\"center\", fontsize=6.0, color=INK2)\n        ax.text(1.03, yi, f\"{PLATFORM.get(plat, plat)}\", va=\"center\", fontsize=5.9, color=INK2,\n                transform=ax.get_yaxis_transform())\n        ax.text(1.40, yi, REGION.get(c, \"\"), va=\"center\", fontsize=5.9, color=INK3,\n                transform=ax.get_yaxis_transform())\n    ax.set_yticks(y); ax.set_yticklabels([r[0] for r in rows], fontsize=6.6)\n    for lab, r in zip(ax.get_yticklabels(), rows):\n        lab.set_color(ROLE_COLOR[r[1]])\n    # role brackets\n    for role in ROLE_COLOR:\n        ys = [yi for yi, r in zip(y, rows) if r[1] == role]\n        if ys:\n            ax.annotate(\"\", xy=(-0.205, min(ys) - 0.35), xytext=(-0.205, max(ys) + 0.35),\n                        xycoords=(\"axes fraction\", \"data\"),\n                        arrowprops=dict(arrowstyle=\"-\", color=ROLE_COLOR[role], lw=2.0), annotation_clip=False)\n            pass\n    ax.text(1.03, len(rows) - 0.25, \"platform\", fontsize=6.0, color=INK, transform=ax.get_yaxis_transform())\n    ax.text(1.40, len(rows) - 0.25, \"region\", fontsize=6.0, color=INK, transform=ax.get_yaxis_transform())\n    ax.set_xlim(0, maxn * 1.10); ax.set_ylim(-0.6, len(rows) - 0.1)\n    ax.set_xlabel(\"donors aged \u2265 60\"); xgrid(ax); despine(ax, left=False)\n    ax.tick_params(axis=\"y\", length=0)\n    l1 = ax.legend(handles=swatches([(\"AD\", AD), (\"control\", CTRL), (\"severity cohort (Braak, MMSE)\", tint(MID, 0.45))]),\n                   loc=\"upper center\", bbox_to_anchor=(0.5, -0.10), ncol=3, fontsize=6.0, frameon=False)\n    ax.add_artist(l1)\n    rh = [Line2D([], [], color=v, lw=2.4, label=k) for k, v in ROLE_COLOR.items()]\n    ax.legend(handles=rh, loc=\"upper center\", bbox_to_anchor=(0.5, -0.155), ncol=5, fontsize=6.0, frameon=False,\n              handlelength=1.4, columnspacing=1.0)\n    panel(ax, \"b\", \"Cohorts\", \"names and left bars coloured by role\", dx=-0.31)\n\n    # ---- (c) donor age by diagnosis: the confound the age checks address ----\n    ax = fig.add_axes([0.09, 0.035, 0.38, 0.14])\n    meta = rd(M, \"merged_metadata.csv\").drop_duplicates(\"donor\")\n    order_ = TRAIN + [HOLDOUT]\n    for i, c in enumerate(order_):\n        d = meta[meta.cohort == c]\n        for g, o in ((\"Control\", -0.19), (\"AD\", 0.19)):\n            boxstrip(ax, d.loc[d.dx == g, \"age\"].dropna().values, i + o, GROUP_COLOR[g], width=0.32,\n                     jitter=0.04, seed=i, size=2.5)\n        dm = d.loc[d.dx == \"AD\", \"age\"].median() - d.loc[d.dx == \"Control\", \"age\"].median()\n        ax.text(i, 109, f\"$\\\\Delta$ {dm:+.0f} y\", ha=\"center\", fontsize=6.0,\n                color=AD if abs(dm) >= 5 else INK2, fontweight=\"bold\" if abs(dm) >= 5 else \"normal\")\n    ax.set_xticks(range(len(order_))); ax.set_xticklabels(order_, fontsize=6.0)\n    for t, c in zip(ax.get_xticklabels(), order_):\n        t.set_color(AD if c == HOLDOUT else PLS)\n    ax.set_ylim(58, 114); ax.set_yticks([60, 70, 80, 90, 100]); ax.set_ylabel(\"age at death\"); ygrid(ax)\n    ax.legend(handles=swatches([(\"control\", CTRL), (\"AD\", AD)]), loc=\"lower right\", ncol=2, fontsize=5.8,\n              frameon=True, framealpha=0.95, facecolor=SURFACE, edgecolor=\"none\")\n    panel(ax, \"c\", \"Donor age by cohort\", \"$\\\\Delta$ = AD $-$ control median; external cohort age-matched\", dx=-0.17)\n\n    # ---- (d) the two gene-list versions in the paper ------------------------\n    ax = fig.add_axes([0.60, 0.035, 0.37, 0.14])\n    def _v7(name):\n        here = Path(INPUT_ROOT).parent / name\n        if here.exists():\n            return pd.read_csv(here)\n        hits = sorted(glob.glob(f\"{INPUT_ROOT}/**/{name}\", recursive=True))\n        hits = [h for h in hits if \"v7admerged\" in h] or hits\n        return pd.read_csv(hits[0])\n    P7, C7 = _v7(\"PLSDA_gene_significance.csv\"), _v7(\"CVAE_gene_significance.csv\")\n    p7, c7 = set(P7.gene[P7.significant]), set(C7.gene[C7.significant])\n    p9 = set(sel.gene[sel.method == \"PLS-DA\"]); c9 = set(sel.gene[sel.method == \"CVAE\"])\n    core9 = set(gv.gene[gv.sets.str.contains(\"Core\")])\n    rows_ = [(\"PLS-DA\", p7, p9, PLS), (\"CVAE\", c7, c9, CVAE_N), (\"shared / core\", p7 & c7, core9, ACCENT)]\n    for i, (nm, a_, b_, c) in enumerate(rows_):\n        parts = [(len(a_ - b_), tint(c, 0.30)), (len(a_ & b_), c), (len(b_ - a_), tint(c, 0.62))]\n        left = 0\n        for n_, fc in parts:\n            ax.barh(i, n_, 0.62, left=left, color=fc, edgecolor=SURFACE, linewidth=0.8)\n            if n_ >= 6:\n                ax.text(left + n_ / 2, i, str(n_), ha=\"center\", va=\"center\", fontsize=6.0,\n                        color=SURFACE if fc == c else INK)\n            left += n_\n        ax.text(left + 2, i, f\"{len(a_ & b_)} of {len(a_)} kept\", va=\"center\", fontsize=5.8, color=INK2)\n    ax.set_yticks(range(3)); ax.set_yticklabels([r[0] for r in rows_], fontsize=6.4); ax.invert_yaxis()\n    for t, r in zip(ax.get_yticklabels(), rows_):\n        t.set_color(r[3])\n    ax.tick_params(axis=\"y\", length=0); ax.set_xlabel(\"genes\"); xgrid(ax)\n    ax.set_xlim(0, max(len(a_ | b_) for _, a_, b_, _ in rows_) * 1.32)\n    ax.legend(handles=swatches([(\"5-cohort run only\", tint(INK3, 0.30)), (\"both runs\", INK2),\n                                (\"held-out run only\", tint(INK3, 0.62))]),\n              loc=\"upper center\", bbox_to_anchor=(0.5, -0.30), ncol=3, fontsize=5.8, frameon=False)\n    panel(ax, \"d\", \"Two versions of the gene lists\",\n          \"5-cohort run (gene-discovery figures) vs GSE132903-held-out run (tested here)\", dx=-0.25)\n    save(fig, \"figV1_design_cohorts\")\n\n\n# =====================================================================\n# Figure 2 -- external test on GSE132903 and the age checks\n# =====================================================================\ndef fig2_external():\n    runs = rd(V9, \"loco_run_metrics.csv\"); roc = rd(V9, \"loco_roc_curves.csv\")\n    mp = rd(V9, \"external_mean_predictions.csv\"); bands = rd(V9, \"external_age_bands.csv\")\n    A = dict(rd(V9, \"external_age_validation.csv\").values)\n    arms = [(\"pls\", \"PLS-DA\", PLS), (\"cvae_neutral\", \"CVAE\", CVAE_N), (\"age\", \"age only\", MID)]\n    fig = plt.figure(figsize=(TW, 6.0))\n    gs = fig.add_gridspec(2, 3, height_ratios=[1, 1], width_ratios=[1.15, 0.85, 1.0],\n                          hspace=0.50, wspace=0.42)\n    base = np.linspace(0, 1, 101)\n\n    ax = fig.add_subplot(gs[0, 0])\n    for m, lab, col in arms:\n        v = roc[roc.model == m][[f\"tpr_{i}\" for i in range(101)]].values\n        mu, sd = v.mean(0), v.std(0)\n        ax.fill_between(base, np.clip(mu - sd, 0, 1), np.clip(mu + sd, 0, 1), color=col, alpha=0.13, lw=0)\n        ax.plot(base, mu, color=col, lw=1.5 if m != \"age\" else 1.1, ls=\"-\" if m != \"age\" else (0, (4, 2)),\n                label=f\"{lab}  {runs[f'auc_{m}'].mean():.3f}\")\n    ax.plot([0, 1], [0, 1], color=RULE, lw=0.6)\n    ax.set_xlim(0, 1); ax.set_ylim(0, 1.005); ax.set_xticks([0, .5, 1]); ax.set_yticks([0, .5, 1])\n    ax.set_xlabel(\"false positive rate\"); ax.set_ylabel(\"true positive rate\")\n    ax.legend(loc=\"lower right\", title=\"mean AUC\", title_fontsize=6.2, fontsize=6.2, alignment=\"left\",\n              frameon=True, framealpha=0.95, facecolor=SURFACE, edgecolor=\"none\")\n    panel(ax, \"a\", f\"Unseen cohort {HOLDOUT}\", f\"{int(A['n_AD'])} AD / {int(A['n_control'])} control donors\")\n\n    ax = fig.add_subplot(gs[0, 1])\n    for i, (m, lab, col) in enumerate(arms):\n        boxstrip(ax, runs[f\"auc_{m}\"].values, i, col, width=0.52, seed=i, size=6)\n    t_p = A.get(\"lr_pls_beyond_age_p\", np.nan)\n    ax.axhline(0.5, color=INK3, lw=0.6, ls=(0, (3, 2)))\n    ax.set_xticks(range(3)); ax.set_xticklabels([a[1] for a in arms], fontsize=6.6)\n    ax.set_ylim(0.4, 0.95); ax.set_ylabel(\"ROC-AUC\"); ygrid(ax)\n    panel(ax, \"b\", \"20 training runs\", \"one point per run\")\n\n    ax = fig.add_subplot(gs[0, 2])\n    groups = [(\"age\", [A[\"auc_age_only_raw\"]], MID),\n              (\"PLS-DA\", [A[\"auc_pls_mean_prob\"], A[\"auc_pls_plus_age_insample\"]], PLS),\n              (\"CVAE\", [A[\"auc_cvae_neutral_mean_prob\"], A[\"auc_cvae_neutral_plus_age_insample\"]], CVAE_N)]\n    for gi, (name, vals, c) in enumerate(groups):\n        offs = [0.0] if len(vals) == 1 else [-0.19, 0.19]\n        for k_, (v, o) in enumerate(zip(vals, offs)):\n            ax.bar(gi + o, v, 0.36, color=c if k_ == 0 else tint(c, 0.3), edgecolor=c, linewidth=0.9)\n            ax.text(gi + o, v + 0.012, f\"{v:.2f}\", ha=\"center\", fontsize=5.6, color=INK2)\n    ax.set_xticks(range(3)); ax.set_xticklabels([g[0] for g in groups], fontsize=6.6)\n    ax.set_ylim(0.4, 1.12); ax.set_ylabel(\"ROC-AUC\"); ygrid(ax)\n    ax.legend(handles=[Rectangle((0, 0), 1, 1, facecolor=PLS, edgecolor=PLS, label=\"model alone\"),\n                       Rectangle((0, 0), 1, 1, facecolor=tint(PLS, 0.3), edgecolor=PLS, label=\"model + age\")],\n              loc=\"upper center\", bbox_to_anchor=(0.5, -0.13), ncol=2, fontsize=5.8, frameon=False, handlelength=0.9)\n    ax.text(0.03, 0.97, f\"adds to age:\\nPLS-DA {pmath(A['lr_pls_beyond_age_p'])}\\nCVAE {pmath(A['lr_cvae_neutral_beyond_age_p'])}\",\n            transform=ax.transAxes, va=\"top\", fontsize=5.8, color=INK2)\n    panel(ax, \"c\", \"Beyond age\", \"likelihood-ratio test\")\n\n    ax = fig.add_subplot(gs[1, 0:2])\n    rng = np.random.RandomState(3)\n    for g in (\"Control\", \"AD\"):\n        s = mp[mp.group == g]\n        ax.scatter(s.age + rng.uniform(-0.35, 0.35, len(s)), s.p_pls, s=12, color=GROUP_COLOR[g],\n                   marker=GROUP_MARKER[g], alpha=0.78, edgecolor=SURFACE, linewidth=0.35, label=g)\n    ax.set_xlabel(\"age at death (years)\"); ax.set_ylabel(\"predicted P(AD), PLS-DA\"); ax.set_ylim(-0.03, 1.32)\n    ygrid(ax)\n    ax.text(0.015, 0.975, f\"risk vs age within group:  controls \u03c1 = {A['rho_age_pls_control']:+.2f} \"\n                          f\"({pmath(A['rho_age_pls_control_p'])}),  AD \u03c1 = {A['rho_age_pls_AD']:+.2f} \"\n                          f\"({pmath(A['rho_age_pls_AD_p'])})\",\n            transform=ax.transAxes, va=\"top\", fontsize=6.0, color=INK2)\n    ax.legend(loc=\"upper right\", bbox_to_anchor=(1.0, 0.90), fontsize=6.2, frameon=True, framealpha=0.95,\n              facecolor=SURFACE, edgecolor=\"none\", markerscale=1.2)\n    panel(ax, \"d\", \"Predicted risk does not follow age\", f\"AD and controls age-matched (median \"\n          f\"{A['age_median_AD']:.0f} vs {A['age_median_control']:.0f} years)\", dx=-0.075)\n\n    ax = fig.add_subplot(gs[1, 2])\n    x = np.arange(len(bands))\n    for j, (m, lab, col) in enumerate(arms):\n        ax.errorbar(x + (j - 1) * 0.2, bands[f\"auc_{m}_mean\"], yerr=bands[f\"auc_{m}_sd\"],\n                    fmt=[\"o\", \"s\", \"^\"][j], color=col, ms=4.6, lw=0.9, capsize=1.8, label=lab)\n    ax.axhline(0.5, color=INK3, lw=0.6, ls=(0, (3, 2)))\n    ax.set_xticks(x); ax.set_xticklabels([f\"{b.band}\\n{b.n_AD}/{b.n_control}\" for b in bands.itertuples()],\n                                         fontsize=6.0)\n    ax.set_xlim(-0.5, len(bands) - 0.5); ax.set_ylim(0.3, 1.08); ax.set_ylabel(\"ROC-AUC\"); ygrid(ax)\n    ax.legend(loc=\"lower left\", fontsize=5.8, ncol=3, frameon=True, framealpha=0.95, facecolor=SURFACE,\n              edgecolor=\"none\", columnspacing=0.6, handletextpad=0.2)\n    panel(ax, \"e\", \"Inside age bands\", \"years; AD / control donors\")\n    save(fig, \"figV2_external_validation\")\n\n\n# =====================================================================\n# Figure 3 -- the gene signatures in six independent cohorts\n# =====================================================================\ndef fig3_replication():\n    sv = rd(V10, \"set_validation.csv\"); rna = rd(V11, \"rnaseq_signature.csv\")\n    gv = rd(V10, \"gene_validation_table.csv\")\n    rows = []\n    for _, r in sv[sv.cohort != \"META\"].iterrows():\n        rows.append(dict(set=r.set, cohort=r.cohort, auc=r.sig_auc, lo=r.null_auc_lo, hi=r.null_auc_hi,\n                         p=r.p_auc, top=r.topDE_auc))\n    for _, r in rna.iterrows():\n        rows.append(dict(set=r.set, cohort=RNASEQ, auc=r.signature_auc, lo=r.null_lo, hi=r.null_hi,\n                         p=r.p_vs_matched, top=r.topDE_auc))\n    R = pd.DataFrame(rows)\n    fig = plt.figure(figsize=(TW, 7.4))\n    gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 1.15], hspace=0.50, wspace=0.16,\n                          width_ratios=[1, 1, 1])\n    cohorts = DXC\n    for k, s in enumerate([\"PLS-DA\", \"CVAE\", \"Core\"]):\n        ax = fig.add_subplot(gs[0, k])\n        d = R[R.set == s].set_index(\"cohort\").reindex(cohorts)\n        y = np.arange(len(cohorts))[::-1]\n        for yi, c in zip(y, cohorts):\n            if c not in d.index or np.isnan(d.loc[c, \"auc\"]):\n                ax.text(0.75, yi, \"not tested\" if c == HOLDOUT else \"\", ha=\"center\", va=\"center\",\n                        fontsize=5.6, color=INK3, style=\"italic\")\n                continue\n            r = d.loc[c]\n            ax.add_patch(Rectangle((r.lo, yi - 0.28), r.hi - r.lo, 0.56, color=GRID, lw=0, zorder=1))\n            ax.plot([r.top, r.top], [yi - 0.30, yi + 0.30], color=INK2, lw=1.3, zorder=2)\n            ax.scatter(r.auc, yi, marker=SET_MARK[s], s=34, color=SET_COLOR[s], zorder=3, edgecolor=SURFACE,\n                       linewidth=0.6)\n            ax.text(max(r.auc, r.hi, r.top) + 0.022, yi, f\"{r.auc:.2f}{star(r.p)}\", ha=\"left\", va=\"center\",\n                    fontsize=6.0, color=INK2)\n        ax.axvline(0.5, color=INK3, lw=0.6, ls=(0, (3, 2)))\n        ax.set_xlim(0.45, 1.06); ax.set_ylim(-0.6, len(cohorts) - 0.4); ax.set_xticks([0.5, 0.7, 0.9])\n        ax.set_yticks(y); ax.set_yticklabels(cohorts if k == 0 else [], fontsize=6.6)\n        ax.tick_params(axis=\"y\", length=0); xgrid(ax)\n        ax.set_xlabel(\"signature ROC-AUC\")\n        n = int(gv.sets.str.contains(s).sum())\n        panel(ax, \"abc\"[k], f\"{s} set ({n} genes)\",\n              \"grey = random 95%, bar = top-N DE\" if k == 0 else None, dx=-0.37 if k == 0 else -0.12)\n    fig.text(0.125, 0.515, \"* better than matched random genes (* p<0.05, ** p<0.01, *** p<0.001); \"\n                           \"signature = mean(up) \u2212 mean(down), no model fitted\", fontsize=5.8, color=INK3,\n             style=\"italic\")\n\n    # (d) meta-analysis volcano of every selected gene\n    gsb = gs[1, :].subgridspec(1, 2, width_ratios=[2.1, 1.0], wspace=0.32)\n    ax = fig.add_subplot(gsb[0])\n    g = gv.copy()\n    g[\"sign\"] = np.where(g.train_direction == \"up\", 1, -1)\n    g[\"eff\"] = g.meta_pooled * g.sign\n    g[\"nlq_true\"] = -np.log10(g.meta_q.clip(lower=1e-300))\n    YCAP = 12.0                                   # GFAP would otherwise compress every other gene\n    g[\"nlq\"] = g.nlq_true.clip(upper=YCAP)\n    cls = np.where(g.sets.str.contains(\"Core\"), \"Core\",\n                   np.where(g.sets.str.contains(\"PLS-DA\") & g.sets.str.contains(\"CVAE\"), \"both\",\n                            np.where(g.sets.str.contains(\"PLS-DA\"), \"PLS-DA\", \"CVAE\")))\n    g[\"cls\"] = cls\n    style = {\"CVAE\": (CVAE_N, \"s\", 14), \"PLS-DA\": (PLS, \"o\", 14), \"both\": (MID, \"o\", 16), \"Core\": (ACCENT, \"D\", 30)}\n    for k_ in (\"CVAE\", \"PLS-DA\", \"both\", \"Core\"):\n        s_ = g[g.cls == k_]; c, mk, sz = style[k_]\n        ax.scatter(s_.eff, s_.nlq, s=sz, marker=mk, color=c, alpha=0.85 if k_ != \"Core\" else 1,\n                   edgecolor=SURFACE, linewidth=0.4, zorder=3 if k_ == \"Core\" else 2,\n                   label={\"both\": \"both models\", \"Core\": \"Core genes\"}.get(k_, f\"{k_} only\"))\n    ax.axhline(-np.log10(0.05), color=INK3, lw=0.6, ls=(0, (3, 2)))\n    ax.axvline(0, color=INK3, lw=0.6)\n    ax.set_xlabel(\"pooled effect in the trained direction (SD, 5 cohorts)\")\n    ax.set_ylabel(\"$-\\\\log_{10}$ meta q (capped at 12)\")\n    ax.set_ylim(0, YCAP * 1.12); ygrid(ax)\n    capped = g[g.nlq_true > YCAP]\n    done = []\n    for _, r in capped.sort_values(\"nlq_true\", ascending=False).iterrows():\n        if any(abs(r.eff - x) < 0.08 for x in done):\n            continue\n        done.append(r.eff)\n        ax.annotate(f\"\\u2191{r.nlq_true:.0f}\", (r.eff, YCAP), xytext=(0, 5), textcoords=\"offset points\",\n                    ha=\"center\", fontsize=5.4, color=INK3)\n    core = g[g.cls == \"Core\"]\n    ax.set_xlim(g.eff.min() - 0.1, g.eff.max() + 0.25)\n    fig.canvas.draw()\n    repel_avoiding(ax, core.eff.values, core.nlq.values, list(core.gene), g.eff.values, g.nlq.values, fontsize=5.8)\n    ax.text(0.01, -np.log10(0.05) + 0.25, \"q = 0.05\", fontsize=5.6, color=INK3,\n            transform=ax.get_yaxis_transform())\n    ax.legend(loc=\"upper left\", fontsize=6.0, frameon=True, framealpha=0.95, facecolor=SURFACE,\n              edgecolor=\"none\", ncol=2)\n    ok = (g.eff > 0) & (g.meta_q < 0.05)\n    panel(ax, \"d\", \"Every selected gene, random-effects meta-analysis\",\n          f\"{int(ok.sum())} of {len(g)} replicate (right of 0, above line); age/sex-adjusted, donor level\", dx=-0.075)\n\n    # (e) gene-level replication vs baselines (honest: not above baseline)\n    ax = fig.add_subplot(gsb[1])\n    mt = sv[sv.cohort == \"META\"].set_index(\"set\")\n    for j, s in enumerate([\"PLS-DA\", \"CVAE\", \"Core\"]):\n        if s not in mt.index:\n            continue\n        r = mt.loc[s]\n        ax.add_patch(Rectangle((j - 0.22, r.null_rep_lo), 0.44, r.null_rep_hi - r.null_rep_lo, color=GRID, lw=0))\n        ax.plot([j - 0.25, j + 0.25], [r.topDE_replication] * 2, color=INK2, lw=1.3)\n        ax.scatter(j, r.replication, marker=SET_MARK[s], s=40, color=SET_COLOR[s], zorder=3, edgecolor=SURFACE)\n        ax.text(j, r.replication - 0.075, f\"{r.replication:.0%}\", ha=\"center\", fontsize=5.8, color=INK2)\n    ax.set_xticks(range(3)); ax.set_xticklabels([\"PLS-DA\", \"CVAE\", \"Core\"], fontsize=6.6)\n    ax.set_ylim(0, 1.02); ax.set_xlim(-0.6, 2.6); ax.set_ylabel(\"genes replicated (meta)\"); ygrid(ax)\n    panel(ax, \"e\", \"Gene by gene\", \"grey: matched random; bar: top-N DE\", dx=-0.36)\n    save(fig, \"figV3_signature_replication\")\n\n\n# =====================================================================\n# Figure 4 -- what cell composition explains, and what survives it\n# =====================================================================\ndef fig4_composition():\n    Q = rd(V11, \"cell_composition_qc.csv\"); B = rd(V11, \"signature_beyond_composition.csv\")\n    G = rd(V11, \"gene_effects_composition_adjusted.csv\")\n    piv = Q.dropna(subset=[\"AD_minus_control_d\"]).pivot(index=\"cohort\", columns=\"cell\", values=\"AD_minus_control_d\")\n    piv = piv.reindex([c for c in DXC if c in piv.index])[[c for c in CELL_ORDER if c in piv.columns]]\n    fig = plt.figure(figsize=(TW, 7.0))\n    gs = fig.add_gridspec(2, 2, height_ratios=[0.95, 1.25], width_ratios=[1.0, 1.0], hspace=0.62, wspace=0.55)\n\n    ax = fig.add_subplot(gs[0, 0])\n    lim = np.nanmax(np.abs(piv.values))\n    ax.imshow(piv.values, cmap=DIVERGING, vmin=-lim, vmax=lim, aspect=\"auto\")\n    for i in range(piv.shape[0]):\n        for j in range(piv.shape[1]):\n            v = piv.values[i, j]\n            ax.text(j, i, f\"{v:+.1f}\", ha=\"center\", va=\"center\", fontsize=5.8,\n                    color=SURFACE if abs(v) > 0.85 * lim else INK)\n    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns, rotation=35, ha=\"right\", fontsize=6.2)\n    ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index, fontsize=6.2)\n    ax.tick_params(length=0)\n    for sp in ax.spines.values():\n        sp.set_visible(False)\n    panel(ax, \"a\", \"Cell mix, AD \u2212 control\", \"standardised difference; markers: McKenzie 2018\", dx=-0.42)\n\n    ax = fig.add_subplot(gs[0, 1])\n    cohorts = [c for c in DXC if c in set(B.cohort)]\n    y = np.arange(len(cohorts))[::-1]\n    for yi, c in zip(y, cohorts):\n        d = B[B.cohort == c].set_index(\"set\")\n        b0 = d.cv_auc_age_sex_cells.iloc[0]\n        ax.scatter(b0, yi, s=26, facecolor=SURFACE, edgecolor=MID, linewidth=1.1, zorder=3)\n        for k_, s in enumerate((\"PLS-DA\", \"CVAE\")):\n            if s not in d.index:\n                continue\n            r = d.loc[s]; yy = yi + (0.17 if s == \"PLS-DA\" else -0.17)\n            ax.plot([b0, r.cv_auc_plus_signature], [yy, yy], color=tint(SET_COLOR[s], 0.55), lw=1.4, zorder=2)\n            ax.scatter(r.cv_auc_plus_signature, yy, marker=SET_MARK[s], s=26, color=SET_COLOR[s], zorder=4,\n                       edgecolor=SURFACE, linewidth=0.5)\n            if r.p_vs_matched_dauc < 0.05:\n                ax.text(r.cv_auc_plus_signature + 0.008, yy, \"*\", va=\"center\", fontsize=8, color=INK)\n    ax.set_yticks(y); ax.set_yticklabels(cohorts, fontsize=6.2); ax.tick_params(axis=\"y\", length=0)\n    ax.set_xlim(0.66, 0.97); ax.set_ylim(-0.6, len(cohorts) - 0.4)\n    ax.set_xlabel(\"cross-validated ROC-AUC\"); xgrid(ax)\n    ax.legend(handles=[Line2D([], [], marker=\"o\", ls=\"\", markerfacecolor=SURFACE, markeredgecolor=MID,\n                              label=\"age + sex + 6 cell types\"),\n                       Line2D([], [], marker=\"o\", ls=\"\", color=PLS, label=\"+ PLS-DA signature\"),\n                       Line2D([], [], marker=\"s\", ls=\"\", color=CVAE_N, label=\"+ CVAE signature\")],\n              loc=\"upper left\", bbox_to_anchor=(-0.05, -0.17), fontsize=5.8, ncol=3, frameon=False,\n              columnspacing=1.0, handletextpad=0.5)\n    panel(ax, \"b\", \"Signature beyond cell mix\", \"* gain beats matched random genes\", dx=-0.36)\n\n    ax = fig.add_subplot(gs[1, :])\n    C = G[G.set == \"Core\"].sort_values(\"pooled_raw\", ascending=True)\n    yy = np.arange(len(C))\n    ax.hlines(yy, C.pooled_adj, C.pooled_raw, color=RULE, lw=1.0, zorder=1)\n    ax.hlines(yy, C.lo_adj, C.hi_adj, color=tint(ACCENT, 0.6), lw=2.2, zorder=2)\n    ax.scatter(C.pooled_raw, yy, s=28, facecolor=SURFACE, edgecolor=INK3, linewidth=1.0, zorder=3,\n               label=\"age + sex adjusted\")\n    sig = C.q_adj < 0.05\n    ax.scatter(C.pooled_adj[sig], yy[sig.values], s=34, color=ACCENT, edgecolor=SURFACE, linewidth=0.5, zorder=4,\n               label=\"+ 6 cell types, q < 0.05\")\n    ax.scatter(C.pooled_adj[~sig], yy[~sig.values], s=30, facecolor=tint(ACCENT, 0.25), edgecolor=ACCENT,\n               linewidth=0.8, zorder=4, label=\"+ 6 cell types, n.s.\")\n    for i, (q, fr) in enumerate(zip(C.q_adj, C.retained_frac)):\n        ax.text(1.005, i, (f\"{fr:.0%} kept\" if fr <= 1 else f\"grew {fr:.1f}$\\\\times$\") + f\"   q {q:.1g}\",\n                transform=ax.get_yaxis_transform(), va=\"center\",\n                fontsize=5.8, color=INK if q < 0.05 else INK3)\n    ax.axvline(0, color=INK3, lw=0.6, ls=(0, (3, 2)))\n    italic_ticks(ax, [f\"{g} ({d})\" for g, d in zip(C.gene, C.direction)], 6.4)\n    ax.set_ylim(-0.7, len(C) - 0.3); ax.set_xlabel(\"pooled effect in the trained direction (SD)\")\n    xgrid(ax); ax.tick_params(axis=\"y\", length=0)\n    ax.legend(loc=\"lower right\", fontsize=6.0, frameon=True, framealpha=0.95, facecolor=SURFACE, edgecolor=\"none\")\n    panel(ax, \"c\", \"Core genes before and after cell-mix adjustment\",\n          \"meta over the 5 cohorts not used to define them (incl. RNA-seq)\", dx=-0.13)\n    save(fig, \"figV4_cell_composition\")\n\n\n# =====================================================================\n# Figure 5 -- dose-response with neuropathology and cognition\n# =====================================================================\ndef fig5_dose():\n    P10 = rd(V10, \"dose_response_points.csv\"); P11 = rd(V11, \"dose_response_points_v11.csv\")\n    D = rd(V11, \"dose_response_composition_adjusted.csv\")\n    pts = pd.concat([P10[P10.set == \"PLS-DA\"][[\"cohort\", \"measure\", \"score\", \"severity\"]],\n                     P11[P11.cohort == RNASEQ][[\"cohort\", \"measure\", \"score\", \"severity\"]]])\n    show = [(\"GSE106241\", \"braak\", \"Braak stage\"), (RNASEQ, \"braak\", \"Braak stage\"),\n            (\"GSE29378\", \"plaque\", \"plaque score (0\u20133)\"), (\"GSE1297\", \"mmse\", \"MMSE\")]\n    fig = plt.figure(figsize=(TW, 6.2))\n    gs = fig.add_gridspec(2, 4, height_ratios=[1, 1.05], hspace=0.52, wspace=0.55)\n    for k, (c, m, lab) in enumerate(show):\n        ax = fig.add_subplot(gs[0, k])\n        p = pts[(pts.cohort == c) & (pts.measure == m)].dropna()\n        lv = sorted(p.severity.unique())\n        if len(lv) <= 7:\n            for v in lv:\n                boxstrip(ax, p[p.severity == v].score.values, v, PLS, width=0.55, seed=int(v * 10), size=4)\n            ax.set_xticks(lv); ax.set_xticklabels([str(int(v)) for v in lv], fontsize=6.2)\n        else:\n            ax.scatter(p.severity, p.score, s=11, color=PLS, alpha=0.8, edgecolor=SURFACE, linewidth=0.3)\n            z = np.polyfit(p.severity, p.score, 1); xx = np.linspace(p.severity.min(), p.severity.max(), 5)\n            ax.plot(xx, np.polyval(z, xx), color=PLS, lw=1.0)\n        d = D[(D.cohort == c) & (D.measure == m) & (D.set == \"PLS-DA\")].iloc[0]\n        lo, hi = ax.get_ylim(); ax.set_ylim(lo, hi + 0.42 * (hi - lo))\n        ax.text(0.03, 0.97, f\"\u03c1 = {d.rho_age_adjusted:+.2f}\\n+cells {d.rho_age_cells_adjusted:+.2f}\"\n                            f\"{star(d.p_vs_matched_cells_adjusted)}\",\n                transform=ax.transAxes, va=\"top\", fontsize=5.8, color=INK2)\n        ax.set_xlabel(lab, fontsize=6.6); ax.set_ylabel(\"PLS-DA score\" if k == 0 else \"\"); ygrid(ax)\n        panel(ax, \"abcd\"[k], c, f\"n = {int(d.n)}\", dx=-0.42 if k == 0 else -0.30)\n\n    ax = fig.add_subplot(gs[1, :])\n    rows = D[[\"cohort\", \"measure\"]].drop_duplicates().values.tolist()\n    lab = {\"braak\": \"Braak\", \"mmse\": \"MMSE\", \"nft\": \"tangles\", \"plaque\": \"plaques\"}\n    x = np.arange(len(rows))\n    for j, s in enumerate([\"PLS-DA\", \"CVAE\", \"Core\"]):\n        for i, (c, m) in enumerate(rows):\n            d = D[(D.cohort == c) & (D.measure == m) & (D.set == s)]\n            if not len(d):\n                continue\n            d = d.iloc[0]; e = d.expected_sign; xx = i + (j - 1) * 0.24\n            ax.add_patch(Rectangle((xx - 0.07, d.null_lo), 0.14, d.null_hi - d.null_lo, color=GRID, lw=0))\n            ax.plot([xx, xx], [d.rho_age_adjusted * e, d.rho_age_cells_adjusted * e], color=tint(SET_COLOR[s], 0.5),\n                    lw=1.0)\n            ax.scatter(xx, d.rho_age_adjusted * e, marker=SET_MARK[s], s=18, facecolor=SURFACE,\n                       edgecolor=SET_COLOR[s], linewidth=0.9, zorder=3)\n            ax.scatter(xx, d.rho_age_cells_adjusted * e, marker=SET_MARK[s], s=24, color=SET_COLOR[s], zorder=4,\n                       edgecolor=SURFACE, linewidth=0.5, label=s if i == 0 else None)\n            if d.p_vs_matched_cells_adjusted < 0.05:\n                ax.text(xx + 0.075, d.rho_age_cells_adjusted * e, \"*\", ha=\"left\", va=\"center\", fontsize=8, color=INK)\n    ax.axhline(0, color=INK3, lw=0.6)\n    ax.set_xticks(x); ax.set_xticklabels([f\"{c}\\n{lab.get(m, m)}\" for c, m in rows], fontsize=6.0)\n    ax.set_xlim(-0.6, len(rows) - 0.4); ax.set_ylim(-0.35, 0.95)\n    ax.set_ylabel(\"partial \u03c1 (expected direction)\"); ygrid(ax)\n    ax.legend(loc=\"upper right\", ncol=3, fontsize=6.0, frameon=True, framealpha=0.95, facecolor=SURFACE,\n              edgecolor=\"none\")\n    panel(ax, \"e\", \"Severity, all cohorts and sets\",\n          \"hollow: age removed \u00b7 filled: age + cell mix removed \u00b7 grey: matched random \u00b7 \"\n          \"* beats matched random\", dx=-0.075)\n    save(fig, \"figV5_dose_response\")\n\n\n# =====================================================================\n# Figure 6 -- evidence scorecard\n# =====================================================================\ndef fig6_scorecard():\n    sv = rd(V10, \"set_validation.csv\"); rna = rd(V11, \"rnaseq_signature.csv\")\n    B = rd(V11, \"signature_beyond_composition.csv\"); D = rd(V11, \"dose_response_composition_adjusted.csv\")\n    A = dict(rd(V9, \"external_age_validation.csv\").values); runs = rd(V9, \"loco_run_metrics.csv\")\n    cols = DXC + SEVERITY\n    tests = []\n    for s in (\"PLS-DA\", \"CVAE\"):\n        key = \"pls\" if s == \"PLS-DA\" else \"cvae_neutral\"\n        auc = runs[f\"auc_{key}\"].mean()\n        tests.append((s, \"trained-model AUC, unseen cohort\",\n                      {HOLDOUT: (auc, 0.0 if auc > runs.auc_age.mean() else 1, \"auc\")}))\n        tests.append((s, \"model adds to age (LR test)\", {HOLDOUT: (np.nan, A[f\"lr_{key}_beyond_age_p\"], \"p\")}))\n        rr = rna[rna.set == s]\n        sub = sv[(sv.set == s) & (sv.cohort != \"META\")]\n        cells = {r.cohort: (r.sig_auc, r.p_auc, \"auc\") for _, r in sub.iterrows()}\n        if len(rr):\n            cells[RNASEQ] = (rr.signature_auc.iloc[0], rr.p_vs_matched.iloc[0], \"auc\")\n        tests.append((s, \"signature AUC > matched random\", cells))\n        cells = {r.cohort: (r.sig_auc - r.topDE_auc, 0.01 if r.sig_auc > r.topDE_auc else 1, \"diff\")\n                 for _, r in sub.iterrows()}\n        if len(rr):\n            dlt = rr.signature_auc.iloc[0] - rr.topDE_auc.iloc[0]\n            cells[RNASEQ] = (dlt, 0.01 if dlt > 0 else 1, \"diff\")\n        tests.append((s, \"signature AUC \\u2212 top-N DE AUC\", cells))\n        cells = {r.cohort: (r.delta_auc, r.p_vs_matched_dauc, \"diff\") for _, r in B[B.set == s].iterrows()}\n        tests.append((s, \"AUC gain beyond age, sex, cell mix\", cells))\n        cells = {r.cohort: (r.rho_age_cells_adjusted * r.expected_sign, r.p_vs_matched_cells_adjusted, \"rho\")\n                 for _, r in D[(D.set == s) & (D.measure == \"braak\")].iterrows()}\n        tests.append((s, \"tracks Braak beyond cell mix\", cells))\n    fig, ax = plt.subplots(figsize=(TW, 0.30 * len(tests) + 1.5))\n    for i, (s, name, cells) in enumerate(tests):\n        for j, c in enumerate(cols):\n            if c not in cells:\n                ax.add_patch(Rectangle((j - 0.47, i - 0.42), 0.94, 0.84, facecolor=SURFACE, edgecolor=GRID, lw=0.6))\n                continue\n            v, p, tag = cells[c]\n            ok = p < 0.05\n            face = SET_COLOR[s] if ok else tint(SET_COLOR[s], 0.12)\n            ax.add_patch(Rectangle((j - 0.47, i - 0.42), 0.94, 0.84, facecolor=face, edgecolor=SURFACE, lw=1.0))\n            if tag == \"p\":\n                txt = f\"p={p:.0e}\" if p < 1e-3 else f\"p={p:.2f}\"\n            elif tag == \"auc\":\n                txt = f\"{v:.2f}\"\n            else:\n                txt = f\"{v:+.2f}\"\n            ax.text(j, i, txt, ha=\"center\", va=\"center\", fontsize=5.6, color=SURFACE if ok else INK2)\n    ax.set_xlim(-0.5, len(cols) - 0.5); ax.set_ylim(len(tests) - 0.5, -0.5)\n    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=30, ha=\"right\", fontsize=6.4)\n    ax.set_yticks(range(len(tests))); ax.set_yticklabels([t[1] for t in tests], fontsize=6.4)\n    ax.tick_params(length=0)\n    for sp in ax.spines.values():\n        sp.set_visible(False)\n    half = len(tests) // 2\n    for k, s in enumerate((\"PLS-DA\", \"CVAE\")):\n        ax.annotate(s, xy=(-0.43, k * half + (half - 1) / 2), xycoords=(\"axes fraction\", \"data\"),\n                    ha=\"center\", va=\"center\", rotation=90, fontsize=8, color=SET_COLOR[s], fontweight=\"bold\",\n                    annotation_clip=False)\n        ax.annotate(\"\", xy=(-0.40, k * half - 0.4), xytext=(-0.40, k * half + half - 0.6),\n                    xycoords=(\"axes fraction\", \"data\"), arrowprops=dict(arrowstyle=\"-\", color=SET_COLOR[s], lw=1.6),\n                    annotation_clip=False)\n    ax.axhline(half - 0.5, color=INK3, lw=0.8)\n    ax.axvline(len(DXC) - 0.5, color=RULE, lw=0.8)\n    ax.text(len(DXC) + 0.5, len(tests) + 0.9, \"severity cohorts\", ha=\"center\", fontsize=6.0, color=INK3, style=\"italic\")\n    panel(ax, \"\", \"Evidence at a glance\",\n          \"filled = passes (p < 0.05 or above baseline) \\u00b7 pale = does not \\u00b7 blank = not tested\",\n          dx=-0.45, dy=1.04)\n    save(fig, \"figV6_evidence_scorecard\")\n\n\nfor _f in (fig1_design, fig2_external, fig3_replication, fig4_composition, fig5_dose, fig6_scorecard):\n    try:\n        _f()\n    except Exception as _e:\n        import traceback; traceback.print_exc()\n        print(f\"!! {_f.__name__} failed: {_e}\")\n"
open('/kaggle/working/scripts/validation_figures.py', 'w').write(SRC_VALIDATION_FIGURES)
print('wrote validation_figures.py', len(SRC_VALIDATION_FIGURES), 'chars')


## `scripts/age_figures.py`


In [ ]:
SRC_AGE_FIGURES = "\"\"\"Figure of paper3: the benchmark repeated with age taken out (V15ADage).\n\n(a) classification under the three ways of handling age, (b, c) how far each model's\ngenome-wide gene ranking moves, (d) the shared genes ranked under every analysis,\n(e) how much of each gene list survives, (f) whether the selected genes are age genes.\nReads data/ (V7ADmerged) and data/validation/v15adage only; reuses the style of\nvalidation_figures.py (its header is executed, not its figures).\"\"\"\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nexec((HERE / \"validation_figures.py\").read_text().split(\"\\ndef fig1_design\")[0])\nfrom scipy.stats import spearmanr\nfrom matplotlib.colors import to_rgb\nfrom matplotlib.patches import FancyBboxPatch, Rectangle\n\nV15 = \"v15adage\"\nARMS = [(\"V7\", \"as merged\"), (\"A_age_corrected\", \"age-corrected\"), (\"B_age_matched\", \"age-matched\")]\nMODELS = [(\"PLSDA\", \"PLS-DA\", PLS, \"o\"), (\"CVAE\", \"CVAE\", CVAE_N, \"s\")]\nBG = \"#f5f5f8\"\nTERM_LONG = {\"V7, no age handling\": \"V7\", \"A, age-corrected\": \"A_age_corrected\", \"B, age-matched\": \"B_age_matched\"}\n\n\ndef _load(arm, what):\n    \"\"\"V7 files live in data/, the age arms under data/validation/v15adage/<arm>/.\"\"\"\n    if arm == \"V7\":\n        base = Path(os.environ.get(\"PAPER_DATA\", HERE.parent / \"data\"))\n        return pd.read_csv(base / what)\n    return pd.read_csv(_arm_path(arm, what))\n\n\ndef _arm_path(arm, what):\n    hits = sorted(glob.glob(f\"{INPUT_ROOT}/**/{arm}/{what}\", recursive=True))\n    hits = [h for h in hits if V15 in h] or hits\n    if not hits:\n        raise FileNotFoundError(f\"{arm}/{what}\")\n    return hits[0]\n\n\ndef fig7_age():\n    runs = {a: _load(a, \"run_metrics.csv\") for a, _ in ARMS}\n    sig = {(a, m): _load(a, f\"{m}_gene_significance.csv\").set_index(\"gene\") for a, _ in ARMS for m, *_ in MODELS}\n    AG = rd(V15, \"comparison_age_genes.csv\")\n\n    fig = plt.figure(figsize=(TW, 9.0))\n    outer = fig.add_gridspec(3, 1, height_ratios=[1.0, 0.92, 0.62], hspace=0.38,\n                             left=0.085, right=0.985, top=0.885, bottom=0.03)\n    r1 = outer[0].subgridspec(1, 3, width_ratios=[1.35, 1.0, 1.0], wspace=0.42)\n    r3 = outer[2].subgridspec(1, 2, width_ratios=[1.45, 1.0], wspace=0.40)\n\n    fig.text(0.085, 0.985, \"Taking age out moves the edges of the gene lists, not the classification or the core\",\n             fontsize=10.2, color=INK, va=\"top\", fontweight=\"bold\")\n    fig.text(0.085, 0.962, \"the full benchmark repeated with each gene's ageing slope removed (age-corrected) and with \"\n             \"donors matched on age within cohort (age-matched)\", fontsize=6.8, color=INK3, va=\"top\", style=\"italic\")\n\n    # ---- (a) classification under the three age handlings ----------------------------------\n    ax = fig.add_subplot(r1[0])\n    xs = {}\n    for i, (arm, lab) in enumerate(ARMS):\n        x0 = i * 1.0\n        r = runs[arm]\n        ax.axvspan(x0 - 0.45, x0 + 0.45, color=BG if i % 2 == 0 else SURFACE, lw=0, zorder=0)\n        a_, b_ = r.auc_pls.values, r.auc_cvae_lat_neutral.values\n        for k in range(len(r)):\n            up = b_[k] > a_[k]\n            ax.plot([x0 - 0.17, x0 + 0.17], [a_[k], b_[k]], color=CVAE_N if up else PLS, lw=0.55,\n                    alpha=0.45 if up else 0.85, zorder=2)\n        for dx, v, col, mk in ((-0.17, a_, PLS, \"o\"), (0.17, b_, CVAE_N, \"s\")):\n            ax.scatter(np.full(len(v), x0 + dx), v, s=7, marker=mk, color=col, lw=0, zorder=3, alpha=0.9)\n            ax.plot([x0 + dx - 0.11, x0 + dx + 0.11], [v.mean()] * 2, color=col, lw=2.2, zorder=4,\n                    solid_capstyle=\"butt\")\n        ag = r.auc_age_only.mean()\n        ax.plot([x0 - 0.38, x0 + 0.38], [ag, ag], color=MID, lw=1.6, ls=(0, (2, 1.4)), zorder=2)\n        ax.text(x0, ag - 0.022, f\"age {ag:.2f}\", ha=\"center\", va=\"top\", fontsize=5.6, color=INK3)\n        xs[arm] = int((b_ > a_).sum())\n    ax.set_xticks(range(3))\n    ax.set_xticklabels([l for _, l in ARMS], fontsize=6.4)\n    for i, (arm, _) in enumerate(ARMS):\n        r = runs[arm]\n        ax.text(i - 0.03, 0.948, f\"{r.auc_pls.mean():.3f}\", ha=\"right\", va=\"bottom\", fontsize=6.0,\n                color=PLS, fontweight=\"bold\")\n        ax.text(i + 0.03, 0.948, f\"{r.auc_cvae_lat_neutral.mean():.3f}\", ha=\"left\", va=\"bottom\", fontsize=6.0,\n                color=CVAE_N, fontweight=\"bold\")\n    ax.tick_params(axis=\"x\", length=0)\n    ax.set_xlim(-0.5, 2.5); ax.set_ylim(0.38, 0.975); ax.set_yticks([0.4, 0.5, 0.6, 0.7, 0.8, 0.9])\n    ax.set_ylabel(\"test ROC-AUC, 20 splits\"); ax.grid(axis=\"y\", visible=True)\n    ax.spines[\"bottom\"].set_visible(False)\n    ax.legend(handles=[Line2D([], [], marker=\"o\", ls=\"\", color=PLS, markersize=4.2, label=\"PLS-DA\"),\n                       Line2D([], [], marker=\"s\", ls=\"\", color=CVAE_N, markersize=4.0, label=\"CVAE agnostic\"),\n                       Line2D([], [], color=MID, lw=1.6, ls=(0, (2, 1.4)), label=\"age alone\")],\n              loc=\"lower left\", ncol=1, fontsize=5.8, frameon=False, bbox_to_anchor=(0.0, 0.10))\n    panel(ax, \"a\", \"Classification does not move\",\n          f\"means on top; CVAE ahead in {xs['V7']}/{xs['A_age_corrected']}/{xs['B_age_matched']} of 20\", dx=-0.17)\n    ax.annotate(\"\", xy=(0, 1), xycoords=\"axes fraction\")\n    for t_ in ax.texts[-2:]:\n        pass\n\n    # ---- (b, c) genome-wide ranking: as merged vs age-corrected ------------------------------\n    GRE = LinearSegmentedColormap.from_list(\"gre\", [\"#ffffff\", \"#ececf1\", \"#d4d4dc\", \"#a9a9b5\", \"#7c7c88\"])\n    for j, (m, name, col, mk) in enumerate(MODELS):\n        ax = fig.add_subplot(r1[1 + j]); ax.set_box_aspect(1)\n        s0 = sig[(\"V7\", m)].mean_score\n        n = len(s0)\n        r0 = np.log10(s0.rank(ascending=False) / n)\n        rows = []\n        for arm, lab in ARMS[1:]:\n            s1 = sig[(arm, m)].mean_score.reindex(s0.index)\n            top = s0.rank(ascending=False) <= 200\n            rows.append((lab, spearmanr(s0, s1).correlation, spearmanr(s0[top], s1[top]).correlation))\n        s1 = sig[(\"A_age_corrected\", m)].mean_score.reindex(s0.index)\n        r1_ = np.log10(s1.rank(ascending=False) / n)\n        ax.hexbin(r0, r1_, gridsize=34, bins=\"log\", cmap=GRE, mincnt=1, linewidths=0, rasterized=True,\n                  extent=(-4.1, 0, -4.1, 0))\n        sel0 = sig[(\"V7\", m)].significant\n        kept = sel0 & sig[(\"A_age_corrected\", m)].significant.reindex(s0.index).fillna(False)\n        ax.scatter(r0[sel0 & ~kept], r1_[sel0 & ~kept], s=11, marker=mk, facecolor=SURFACE, edgecolor=col, lw=0.8,\n                   zorder=3)\n        ax.scatter(r0[kept], r1_[kept], s=13, marker=mk, color=col, edgecolor=SURFACE, lw=0.4, zorder=4)\n        ax.plot([-4.1, 0], [-4.1, 0], color=INK2, lw=0.6, ls=(0, (3, 2)))\n        ax.axvline(np.log10(sel0.sum() / n), color=col, lw=0.5, alpha=0.5)\n        tk = [-4, -3, -2, -1, 0]; tl = [\"0.01%\", \"0.1%\", \"1%\", \"10%\", \"100%\"]\n        ax.set_xticks(tk); ax.set_xticklabels(tl, fontsize=5.4); ax.set_yticks(tk); ax.set_yticklabels(tl, fontsize=5.4)\n        ax.set_xlim(-4.1, 0); ax.set_ylim(-4.1, 0)\n        ax.set_xlabel(\"rank, as merged\", fontsize=6.2); ax.set_ylabel(\"rank, age-corrected\" if j == 0 else \"\",\n                                                                       fontsize=6.2)\n        (l1, ra1, rt1), (l2, ra2, rt2) = rows\n        panel(ax, \"bc\"[j], f\"{name}: genome-wide \u03c1 {ra1:.2f} / {ra2:.2f}\",\n              f\"corrected / matched; top 200: {rt1:.2f} / {rt2:.2f}\", dx=-0.32)\n\n    # ---- (d) the 18 shared genes, ranked under every analysis ---------------------------------\n    ax = fig.add_subplot(outer[1])\n    p7, c7 = sig[(\"V7\", \"PLSDA\")], sig[(\"V7\", \"CVAE\")]\n    shared = sorted(set(p7.index[p7.significant]) & set(c7.index[c7.significant]))\n    rows = [(arm, lab, m, name, col) for arm, lab in ARMS for m, name, col, _ in MODELS]\n    robust = [g for g in shared if all(sig[(a, m)].loc[g, \"significant\"] for a, _ in ARMS for m, *_ in MODELS)]\n    score = {g: sum(sig[(a, m)].loc[g, \"significant\"] for a, _ in ARMS for m, *_ in MODELS) for g in shared}\n    shared.sort(key=lambda g: (-score[g], p7.loc[g, \"mean_score\"] * -1))\n    for i, (arm, lab, m, name, col) in enumerate(rows):\n        S = sig[(arm, m)]\n        rk = S.mean_score.rank(ascending=False)\n        cm_ = LinearSegmentedColormap.from_list(f\"rk{i}\", [\"#ffffff\", tint(col, 0.15), tint(col, 0.45), col,\n                                                            tuple(v * 0.75 for v in to_rgb(col))])\n        yy = i + (i // 2) * 0.35\n        for j, g in enumerate(shared):\n            r = rk[g]; sel = bool(S.loc[g, \"significant\"])\n            f = np.clip((np.log10(2000) - np.log10(r)) / np.log10(2000), 0, 1)\n            ax.add_patch(FancyBboxPatch((j - 0.42, yy - 0.40), 0.84, 0.80, boxstyle=\"round,pad=0,rounding_size=0.12\",\n                                        facecolor=cm_(f) if sel else SURFACE, edgecolor=col if sel else RULE,\n                                        linewidth=0.0 if sel else 0.6, zorder=2))\n            ax.text(j, yy, f\"{int(r)}\" if r < 1000 else f\"{r / 1000:.1f}k\", ha=\"center\", va=\"center\",\n                    fontsize=4.9, color=SURFACE if (sel and f > 0.55) else (INK2 if sel else INK3), zorder=3)\n        ax.text(-0.75, yy, f\"{name} \u00b7 {lab}\", ha=\"right\", va=\"center\", fontsize=6.2, color=col)\n    nrow_y = (len(rows) - 1) + ((len(rows) - 1) // 2) * 0.35\n    for j, g in enumerate(shared):\n        isr = g in robust\n        ax.text(j, nrow_y + 0.75, g, ha=\"center\", va=\"top\", rotation=90, fontsize=6.4, fontstyle=\"italic\",\n                color=ACCENT if isr else INK2, fontweight=\"bold\" if isr else \"normal\")\n        if isr:\n            ax.add_patch(Rectangle((j - 0.5, -0.62), 1.0, nrow_y + 1.24, facecolor=\"none\", edgecolor=ACCENT,\n                                   linewidth=1.1, zorder=4))\n    ax.set_xlim(-0.6, len(shared) - 0.4); ax.set_ylim(nrow_y + 2.75, -0.8)\n    ax.axis(\"off\")\n    ax.annotate(\"d\", xy=(-0.20, 1.0), xycoords=\"axes fraction\", xytext=(0, 4), textcoords=\"offset points\",\n                fontsize=9.5, fontweight=\"bold\", family=\"DejaVu Sans\", color=INK, va=\"bottom\", annotation_clip=False)\n    ax.annotate(f\"The {len(shared)} genes both models chose, ranked under every analysis: \"\n                f\"{len(robust)} stay selected by both models throughout\",\n                xy=(-0.17, 1.0), xycoords=\"axes fraction\", xytext=(0, 4), textcoords=\"offset points\",\n                fontsize=8.4, color=INK, va=\"bottom\", annotation_clip=False)\n    ax.text(-0.6, nrow_y + 2.72, \"number = genome-wide rank of 11,214 genes \u00b7 filled = selected \u00b7 teal frame = \"\n            \"selected by both models in all three analyses\", ha=\"left\", va=\"bottom\", fontsize=5.6, color=INK3,\n            style=\"italic\")\n\n    # ---- (e) how much of each list survives ------------------------------------------------------\n    ax = fig.add_subplot(r3[0])\n    y = 0\n    for m, name, col, mk in MODELS:\n        s0 = set(sig[(\"V7\", m)].index[sig[(\"V7\", m)].significant])\n        sa = set(sig[(\"A_age_corrected\", m)].index[sig[(\"A_age_corrected\", m)].significant])\n        sb = set(sig[(\"B_age_matched\", m)].index[sig[(\"B_age_matched\", m)].significant])\n        parts = [(len(s0 & sa & sb), col, \"kept in both\"), (len(s0 & (sa ^ sb)), tint(col, 0.50), \"kept in one\"),\n                 (len(s0 - sa - sb), \"#e3e3ea\", \"dropped in both\")]\n        left = 0\n        for n_, c_, lab in parts:\n            ax.barh(y, n_, 0.55, left=left, color=c_, lw=0)\n            if n_ >= 4:\n                ax.text(left + n_ / 2, y, str(n_), ha=\"center\", va=\"center\", fontsize=6.6,\n                        color=SURFACE if c_ == col else INK, fontweight=\"bold\" if c_ == col else \"normal\")\n            left += n_\n        new_a, new_b = len(sa - s0), len(sb - s0)\n        ax.text(left + 1.5, y, f\"of {len(s0)}   \u00b7   new: {new_a} corrected, {new_b} matched\", va=\"center\",\n                fontsize=5.8, color=INK2)\n        ax.text(-1.5, y, name, ha=\"right\", va=\"center\", fontsize=7.0, color=col, fontweight=\"bold\")\n        y += 1\n    ax.set_xlim(0, 125); ax.set_ylim(1.55, -0.6); ax.axis(\"off\")\n    ax.legend(handles=[Rectangle((0, 0), 1, 1, color=INK2, label=\"kept under both age handlings\"),\n                       Rectangle((0, 0), 1, 1, color=tint(INK2, 0.5), label=\"kept under one\"),\n                       Rectangle((0, 0), 1, 1, color=\"#e3e3ea\", label=\"dropped under both\")],\n              loc=\"upper center\", bbox_to_anchor=(0.42, 0.06), ncol=3, fontsize=5.8, frameon=False)\n    ax.annotate(\"e\", xy=(-0.17, 1.0), xycoords=\"axes fraction\", xytext=(0, 4), textcoords=\"offset points\",\n                fontsize=9.5, fontweight=\"bold\", family=\"DejaVu Sans\", color=INK, va=\"bottom\", annotation_clip=False)\n    ax.annotate(\"How much of each gene list survives\", xy=(-0.10, 1.0), xycoords=\"axes fraction\",\n                xytext=(0, 4), textcoords=\"offset points\", fontsize=8.4, color=INK, va=\"bottom\", annotation_clip=False)\n\n    # ---- (f) are the selected genes age genes? -----------------------------------------------------\n    ax = fig.add_subplot(r3[1])\n    bg = 100 * AG.frac_age_genes[AG.list == \"background\"].iloc[0]\n    for i, (arm, lab) in enumerate(ARMS):\n        key = [k for k, v in TERM_LONG.items() if v == arm][0]\n        for dx, (m, name, col, mk) in zip((-0.17, 0.17), MODELS):\n            v = 100 * AG.frac_age_genes[(AG.arm == key) & (AG.list == name)].iloc[0]\n            ax.plot([i + dx, i + dx], [bg, v], color=col, lw=2.4, solid_capstyle=\"butt\", zorder=2)\n            ax.scatter(i + dx, v, s=30, marker=mk, color=col, edgecolor=SURFACE, lw=0.6, zorder=3)\n            ax.text(i + dx, v + (2.2 if v > bg else -2.2), f\"{v:.0f}\", ha=\"center\", va=\"bottom\" if v > bg else \"top\",\n                    fontsize=6.0, color=col)\n    ax.axhline(bg, color=INK, lw=0.8)\n    ax.text(2.48, bg - 0.8, f\"all genes {bg:.0f}%\", ha=\"right\", va=\"top\", fontsize=5.8, color=INK)\n    ax.set_xticks(range(3)); ax.set_xticklabels([l for _, l in ARMS], fontsize=6.4); ax.tick_params(axis=\"x\", length=0)\n    ax.set_xlim(-0.5, 2.5); ax.set_ylim(30, 72); ax.set_ylabel(\"% age-associated in controls\", fontsize=6.6)\n    ax.grid(axis=\"y\", visible=True); ax.spines[\"bottom\"].set_visible(False)\n    panel(ax, \"f\", \"Selected genes are not age genes\", \"% of each list age-associated in controls\", dx=-0.22)\n    save(fig, \"figV7_age_robustness\")\n\n\nfig7_age()\n"
open('/kaggle/working/scripts/age_figures.py', 'w').write(SRC_AGE_FIGURES)
print('wrote age_figures.py', len(SRC_AGE_FIGURES), 'chars')


## `scripts/core_validation_figure.py`


In [ ]:
SRC_CORE_VALIDATION_FIGURE = "\"\"\"The single core figure of the external validation in paper3.\n\n(a) the held-out cohort, (b) its age bands, (c) the genes both models chose before\nand after cell-composition adjustment, and (d) an evidence map: every validation\ncohort against every test, PLS-DA and CVAE side by side as the two halves of one\ndisc. Reads V9ADexternal, V10ADgenevalidation and V11ADrobustness outputs only and\nreuses the style of validation_figures.py (its header is executed, not its figures).\"\"\"\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nexec((HERE / \"validation_figures.py\").read_text().split(\"\\ndef fig1_design\")[0])\nfrom matplotlib.patches import Wedge\n\nV9, V10, V11 = \"v9adexternal\", \"v10adgenevalidation\", \"v11adrobustness\"\nSETS = [(\"PLS-DA\", PLS), (\"CVAE\", CVAE_N)]\nPAPER_BG = \"#f6f6f9\"\n\n\ndef half_disc(ax, x, y, r, passes):\n    \"\"\"Left half PLS-DA, right half CVAE; solid when the test passes, pale ring when not.\"\"\"\n    for (s, col), (t1, t2), ok in zip(SETS, ((90, 270), (-90, 90)), passes):\n        if ok is None:\n            continue\n        ax.add_patch(Wedge((x, y), r, t1, t2, facecolor=col if ok else tint(col, 0.13),\n                           edgecolor=col, linewidth=0.0 if ok else 0.9, zorder=3))\n    ax.plot([x, x], [y - r, y + r], color=SURFACE, lw=0.9, zorder=4)\n\n\ndef fig_core():\n    sv = rd(V10, \"set_validation.csv\"); sv = sv[sv.cohort != \"META\"]\n    rs = rd(V11, \"rnaseq_signature.csv\"); bc = rd(V11, \"signature_beyond_composition.csv\")\n    D = rd(V11, \"dose_response_composition_adjusted.csv\")\n    G = rd(V11, \"gene_effects_composition_adjusted.csv\"); G = G[G.set == \"Core\"]\n    vc = rd(V10, \"validation_cohorts.csv\").set_index(\"cohort\")\n    runs = rd(V9, \"loco_run_metrics.csv\"); A = dict(rd(V9, \"external_age_validation.csv\").values)\n    bands = rd(V9, \"external_age_bands.csv\"); roc = rd(V9, \"loco_roc_curves.csv\")\n    rna = rnaseq_counts()\n\n    fig = plt.figure(figsize=(TW, 8.7))\n    outer = fig.add_gridspec(2, 1, height_ratios=[1.0, 2.2], hspace=0.20, left=0.07, right=0.985,\n                             top=0.955, bottom=0.015)\n    top = outer[0].subgridspec(1, 3, width_ratios=[1.0, 0.82, 1.35], wspace=0.62)\n\n    # ---- (a) the held-out cohort ------------------------------------------------------------\n    ax = fig.add_subplot(top[0]); ax.set_box_aspect(1); ax.set_anchor(\"N\")\n    base = np.linspace(0, 1, 101)\n    for m, c, ls, lw in ((\"age\", MID, (0, (3, 2)), 1.0), (\"cvae_neutral\", CVAE_N, \"-\", 1.5), (\"pls\", PLS, \"-\", 1.7)):\n        v = roc[roc.model == m][[f\"tpr_{i}\" for i in range(101)]].values\n        mu, sd = v.mean(0), v.std(0)\n        ax.fill_between(base, np.clip(mu - sd, 0, 1), np.clip(mu + sd, 0, 1), color=c, alpha=0.12, lw=0)\n        ax.plot(base, mu, color=c, lw=lw, ls=ls)\n    ax.plot([0, 1], [0, 1], color=RULE, lw=0.5)\n    for k, (m, lab, c) in enumerate(((\"pls\", \"PLS-DA\", PLS), (\"cvae_neutral\", \"CVAE\", CVAE_N), (\"age\", \"age\", MID))):\n        ax.text(0.97, 0.05 + 0.13 * (2 - k), f\"{runs[f'auc_{m}'].mean():.2f}\", transform=ax.transAxes, ha=\"right\",\n                va=\"bottom\", fontsize=9.5 if m != \"age\" else 7.5, color=c, fontweight=\"bold\" if m != \"age\" else \"normal\")\n        ax.text(0.70, 0.065 + 0.13 * (2 - k), lab, transform=ax.transAxes, ha=\"right\", va=\"bottom\", fontsize=6.0,\n                color=c)\n    ax.set_xlim(0, 1); ax.set_ylim(0, 1.005); ax.set_xticks([0, .5, 1]); ax.set_yticks([0, .5, 1])\n    ax.set_xlabel(\"false-positive rate\", fontsize=6.6); ax.set_ylabel(\"true-positive rate\", fontsize=6.6)\n    panel(ax, \"a\", \"A cohort never seen\", f\"{HOLDOUT} \u00b7 {int(A['n_AD'])} AD / {int(A['n_control'])} control\",\n          dx=-0.36)\n\n    # ---- (b) age bands ----------------------------------------------------------------------------\n    ax = fig.add_subplot(top[1])\n    x = np.arange(len(bands))\n    for m, c, mk, dx_ in ((\"pls\", PLS, \"o\", -0.2), (\"cvae_neutral\", CVAE_N, \"s\", 0.0), (\"age\", MID, \"D\", 0.2)):\n        v = bands[f\"auc_{m}_mean\"].values; e = bands[f\"auc_{m}_sd\"].values\n        ax.errorbar(x + dx_, v, yerr=e, fmt=mk, color=c, ms=4.6 if m != \"age\" else 3.8, lw=0.9, capsize=0, zorder=3)\n    ax.plot(x - 0.2, bands.auc_pls_mean, color=PLS, lw=0.6, alpha=0.5)\n    ax.plot(x, bands.auc_cvae_neutral_mean, color=CVAE_N, lw=0.6, alpha=0.5)\n    ax.axhspan(0.4, 0.55, color=PAPER_BG, lw=0, zorder=0)\n    ax.axhline(0.5, color=INK3, lw=0.5, ls=(0, (3, 2)))\n    ax.text(len(bands) - 0.55, 0.43, \"chance\", ha=\"right\", fontsize=5.6, color=INK3, style=\"italic\")\n    ax.set_xticks(x); ax.set_xticklabels([f\"{b.band} y\\n{b.n_AD}/{b.n_control}\" for b in bands.itertuples()],\n                                         fontsize=5.9)\n    ax.tick_params(axis=\"x\", length=0); ax.set_xlim(-0.5, len(bands) - 0.5)\n    ax.set_ylim(0.4, 1.0); ax.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0]); ax.set_ylabel(\"ROC-AUC\", fontsize=6.6)\n    ygrid(ax)\n    panel(ax, \"b\", \"Not an age effect\", \"AUC inside each age band\", dx=-0.42)\n\n    # ---- (c) genes both models chose: before and after cell composition --------------------------\n    ax = fig.add_subplot(top[2])\n    G = G.sort_values(\"pooled_adj\").reset_index(drop=True)\n    keep = G.q_adj < 0.05\n    for r, ok in zip(G.itertuples(), keep):\n        c = ACCENT if ok else \"#b7b7c2\"\n        ax.plot([0, 1], [r.pooled_raw, r.pooled_adj], color=c, lw=1.8 if ok else 0.8, zorder=3 if ok else 2,\n                solid_capstyle=\"round\")\n        ax.scatter([0], [r.pooled_raw], s=14 if ok else 8, color=c, zorder=4, lw=0)\n        ax.scatter([1], [r.pooled_adj], s=20 if ok else 8, color=c, zorder=4, lw=0)\n    ys = G.pooled_adj.values; order = np.argsort(ys)\n    gap = 0.075; pos = []\n    for v in ys[order]:\n        pos.append(max(v, pos[-1] + gap) if pos else v)\n    over = pos[-1] - 1.0\n    if over > 0:\n        pos = [p_ - over for p_ in pos]\n    for p_, i in zip(pos, order):\n        r = G.iloc[i]; ok = r.q_adj < 0.05\n        qs = \"q<0.001\" if r.q_adj < 0.001 else f\"q {r.q_adj:.2g}\"\n        ax.annotate(f\"{r.gene}\", xy=(1, r.pooled_adj), xytext=(1.12, p_), textcoords=\"data\",\n                    fontsize=6.0 if ok else 5.4, fontstyle=\"italic\", va=\"center\", ha=\"left\",\n                    color=ACCENT if ok else INK3, fontweight=\"bold\" if ok else \"normal\", annotation_clip=False,\n                    arrowprops=dict(arrowstyle=\"-\", color=RULE, lw=0.35, shrinkA=0, shrinkB=2))\n        ax.text(1.62, p_, qs, fontsize=5.2 if not ok else 5.6, va=\"center\", ha=\"left\",\n                color=ACCENT if ok else INK3, fontweight=\"bold\" if ok else \"normal\")\n    ax.axhline(0, color=INK3, lw=0.5, ls=(0, (3, 2)))\n    ax.set_xlim(-0.12, 2.05); ax.set_ylim(-0.15, 1.0)\n    ax.set_xticks([0, 1]); ax.set_xticklabels([\"age, sex\", \"+ 6 cell\\ntypes\"], fontsize=6.0)\n    ax.tick_params(axis=\"x\", length=0)\n    ax.set_ylabel(\"pooled effect, trained direction (SD)\", fontsize=6.4); ygrid(ax)\n    ax.spines[\"bottom\"].set_visible(False)\n    panel(ax, \"c\", f\"{int(keep.sum())} of {len(G)} shared genes survive cell mix\",\n          \"meta-analysis, five cohorts not used to define them\", dx=-0.25)\n\n    # ---- (d) the evidence map ---------------------------------------------------------------------\n    ax = fig.add_subplot(outer[1])\n    ROWS = [(\"held out\", AD, [HOLDOUT]), (\"independent microarray\", ACCENT, VALID),\n            (\"RNA-seq\", CVAE_N, [RNASEQ]), (\"pathology only\", MID, SEVERITY)]\n    COLS = [\"Separates AD\\nfrom control\", \"Beats matched\\nrandom genes\", \"Beats plain\\ntop-N DE\",\n            \"Adds beyond\\ncell mix\", \"Tracks Braak\\nstage\", \"Tracks plaques\\nor MMSE\"]\n    cx = [6.75 + 1.55 * k for k in range(len(COLS))]\n    R = 0.30\n    y = 0.0; nrow = 0\n    tally = {s: [[0, 0] for _ in COLS] for s, _ in SETS}\n\n    def sig_row(c, s):\n        if c == RNASEQ:\n            r = rs[rs.set == s].iloc[0]\n            return r.signature_auc, r.p_vs_matched, r.topDE_auc\n        r = sv[(sv.set == s) & (sv.cohort == c)]\n        if not len(r):\n            return None\n        r = r.iloc[0]\n        return r.sig_auc, r.p_auc, r.topDE_auc\n\n    for gname, gcol, cohorts in ROWS:\n        y0 = y\n        for c in cohorts:\n            nrow += 1\n            if nrow % 2 == 1:\n                ax.add_patch(Rectangle((0.0, y - 0.68), cx[-1] + 0.95, 1.30, facecolor=PAPER_BG, lw=0, zorder=0))\n            if c == RNASEQ:\n                n_ad, n_ct, plat = rna[1], rna[2], \"GPL16791\"\n            else:\n                n_ad, n_ct, plat = int(vc.loc[c, \"AD\"]), int(vc.loc[c, \"control\"]), vc.loc[c, \"platform\"]\n            ax.text(0.55, y + 0.20, c, fontsize=6.8, color=INK, va=\"center\", fontweight=\"bold\")\n            ax.text(0.55, y - 0.20, f\"{PLATFORM.get(plat, plat)} \u00b7 {REGION.get(c, '')}\", fontsize=5.0, color=INK2,\n                    va=\"center\")\n            if n_ad + n_ct > 0:\n                w = 0.80\n                ax.add_patch(Rectangle((4.75, y - 0.09), w * n_ad / 400, 0.18, facecolor=AD, lw=0))\n                ax.add_patch(Rectangle((4.75 + w * n_ad / 400, y - 0.09), w * n_ct / 400, 0.18, facecolor=CTRL, lw=0))\n                ax.text(4.75 + w * (n_ad + n_ct) / 400 + 0.08, y, f\"{n_ad}/{n_ct}\", fontsize=5.2, color=INK2,\n                        va=\"center\")\n            else:\n                ax.text(4.75, y, f\"{int(vc.loc[c, 'donors'])} graded\", fontsize=5.2, color=INK3, va=\"center\",\n                        style=\"italic\")\n            vals = {}\n            for s, _ in SETS:\n                sr = sig_row(c, s)\n                b = bc[(bc.set == s) & (bc.cohort == c)]\n                d_b = D[(D.cohort == c) & (D.set == s) & (D.measure == \"braak\")]\n                d_o = D[(D.cohort == c) & (D.set == s) & (D.measure.isin([\"plaque\", \"mmse\"]))]\n                vals[s] = [\n                    (sr[0], sr[1] < 0.05) if sr else None,\n                    (sr[1], sr[1] < 0.05) if sr else None,\n                    (sr[0] - sr[2], sr[0] > sr[2]) if sr else None,\n                    (b.delta_auc.iloc[0], b.p_vs_matched_dauc.iloc[0] < 0.05) if len(b) else None,\n                    (d_b.rho_age_cells_adjusted.iloc[0] * d_b.expected_sign.iloc[0],\n                     d_b.p_vs_matched_cells_adjusted.iloc[0] < 0.05) if len(d_b) else None,\n                    (d_o.rho_age_cells_adjusted.iloc[0] * d_o.expected_sign.iloc[0],\n                     d_o.p_vs_matched_cells_adjusted.iloc[0] < 0.05) if len(d_o) else None]\n            for k in range(len(COLS)):\n                pl, cv_ = vals[\"PLS-DA\"][k], vals[\"CVAE\"][k]\n                if pl is None and cv_ is None:\n                    ax.plot([cx[k] - 0.12, cx[k] + 0.12], [y, y], color=RULE, lw=0.8)\n                    continue\n                half_disc(ax, cx[k], y, R, [True, True] if k == 0 else\n                          [pl[1] if pl else None, cv_[1] if cv_ else None])\n                for s_i, (s, col) in enumerate(SETS):\n                    vv = vals[s][k]\n                    if vv is None:\n                        continue\n                    if k > 0:\n                        tally[s][k][1] += 1; tally[s][k][0] += int(vv[1])\n                    if k == 0:\n                        txt = f\"{vv[0]:.2f}\"\n                    elif k == 1:\n                        txt = \"p<.001\" if vv[0] < 0.001 else f\"p {vv[0]:.2f}\".replace(\"0.\", \".\")\n                    else:\n                        txt = f\"{vv[0]:+.2f}\".replace(\"-0.00\", \"0.00\").replace(\"+0.00\", \"0.00\")\n                    ax.text(cx[k] + (-0.05 if s_i == 0 else 0.05), y - R - 0.17, txt,\n                            ha=\"right\" if s_i == 0 else \"left\", va=\"center\", fontsize=5.0, color=col)\n            y -= 1.36\n        ax.plot([0.25, 0.25], [y0 + 0.5, y + 1.36 - 0.6], color=gcol, lw=2.6, solid_capstyle=\"butt\")\n        ax.text(0.10, (y0 + y + 1.36) / 2 - 0.05, gname, rotation=90, ha=\"right\", va=\"center\", fontsize=5.8, color=gcol)\n        y -= 0.35\n    ytop = 1.15\n    for k, lab in enumerate(COLS):\n        ax.text(cx[k], ytop, lab, ha=\"center\", va=\"bottom\", fontsize=5.9, color=INK, linespacing=1.15)\n        if k > 0:\n            t_p, t_c = tally[\"PLS-DA\"][k], tally[\"CVAE\"][k]\n            ax.text(cx[k] - 0.08, y + 0.55, f\"{t_p[0]}/{t_p[1]}\", ha=\"right\", va=\"center\", fontsize=7.0, color=PLS,\n                    fontweight=\"bold\")\n            ax.text(cx[k] + 0.08, y + 0.55, f\"{t_c[0]}/{t_c[1]}\", ha=\"left\", va=\"center\", fontsize=7.0, color=CVAE_N,\n                    fontweight=\"bold\")\n    ax.text(0.55, y + 0.55, \"tests passed (PLS-DA | CVAE)\", fontsize=6.4, color=INK, va=\"center\", fontweight=\"bold\")\n    ax.plot([0.0, cx[-1] + 0.95], [y + 1.05, y + 1.05], color=INK2, lw=0.6)\n    ax.text(0.55, ytop + 0.05, \"Cohort\", fontsize=6.3, color=INK, va=\"bottom\")\n    ax.text(4.75, ytop + 0.05, \"AD/ctrl\", fontsize=6.3, color=INK, va=\"bottom\")\n    kx, ky = 0.85, y - 0.30\n    half_disc(ax, kx, ky, 0.26, [True, True])\n    ax.text(kx + 0.42, ky, \"left half PLS-DA, right half CVAE \u00b7 solid = test passed ($p<0.05$, or above the comparison \"\n            \"for plain DE) \u00b7 pale = not passed\\nvalues below each disc: AUC, $p$ against matched random genes, \"\n            \"AUC minus plain DE, \u0394AUC over cell mix, partial $\\\\rho$ with pathology\", fontsize=5.6, color=INK2, va=\"center\", linespacing=1.4)\n    ax.set_xlim(-0.35, cx[-1] + 1.0); ax.set_ylim(ky - 0.45, ytop + 0.95)\n    ax.set_aspect(\"equal\"); ax.axis(\"off\")\n    ax.annotate(\"d\", xy=(0.0, 1.0), xycoords=\"axes fraction\", xytext=(-2, 4), textcoords=\"offset points\",\n                fontsize=9.5, fontweight=\"bold\", family=\"DejaVu Sans\", color=INK, va=\"bottom\", ha=\"right\",\n                annotation_clip=False)\n    ax.annotate(\"Evidence map: frozen gene signatures in every validation cohort, no model refitted\",\n                xy=(0.0, 1.0), xycoords=\"axes fraction\", xytext=(4, 4), textcoords=\"offset points\",\n                fontsize=8.4, color=INK, va=\"bottom\", annotation_clip=False)\n    save(fig, \"figV_core_validation\")\n\n\nfig_core()\n"
open('/kaggle/working/scripts/core_validation_figure.py', 'w').write(SRC_CORE_VALIDATION_FIGURE)
print('wrote core_validation_figure.py', len(SRC_CORE_VALIDATION_FIGURE), 'chars')


## `scripts/gene_sides_figure.py`


In [ ]:
SRC_GENE_SIDES_FIGURE = "\"\"\"Figure of paper3: why the two models select different sides of AD (V16ADgenesides).\n\n(a) every gene's AD effect in bulk tissue against its effect after accounting for cell\ncomposition, with the selected genes of each model; (b) how each list's AD effects\ndivide into direct, compositional, masked and undetected; (c) the size of the AD\nshift; (d) the share of that shift that remains beyond cell composition; (e)\nhigh-specificity cell-type marker genes; (f) what each importance score follows\ngenome-wide; (g) co-expression within each list against effect-matched genes.\nReads data/validation/v16adgenesides only and reuses the style of\nvalidation_figures.py (its header is executed, not its figures).\"\"\"\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nexec((HERE / \"validation_figures.py\").read_text().split(\"\\ndef fig1_design\")[0])\nfrom matplotlib.patches import Rectangle\n\nV16 = \"v16adgenesides\"\nLISTS = [(\"PLS-DA only\", \"PLS-DA only\", PLS, \"o\"),\n         (\"shared\", \"both models\", ACCENT, \"D\"),\n         (\"CVAE only\", \"CVAE only\", CVAE_N, \"s\")]\nCLASS = [(\"direct\", \"direct\", \"#2e4b63\"),\n         (\"compositional\", \"compositional\", \"#9aa7b8\"),\n         (\"masked\", \"masked\", \"#b0868f\"),\n         (\"neither\", \"not detected\", \"#e4e4ea\")]\nPANEL_BG = \"#f6f6f9\"\n\n\ndef fig_gene_sides():\n    G = rd(V16, \"gene_sides_per_gene.csv\").set_index(\"gene\")\n    EC = rd(V16, \"gene_sides_effect_classes.csv\")\n    SM = rd(V16, \"gene_sides_summary.csv\")\n    CO = rd(V16, \"gene_sides_coherence.csv\")\n    CN = rd(V16, \"gene_sides_coherence_null.csv\")\n    MK = rd(V16, \"gene_sides_celltypes.csv\")\n    AT = rd(V16, \"gene_sides_attribution.csv\")\n    FD = rd(V16, \"gene_sides_fisher.csv\").iloc[0]\n\n    fig = plt.figure(figsize=(TW, 9.15))\n    outer = fig.add_gridspec(3, 1, height_ratios=[1.32, 0.80, 0.86], hspace=0.52,\n                             left=0.075, right=0.985, top=0.905, bottom=0.045)\n    r1 = outer[0].subgridspec(1, 2, width_ratios=[1.32, 1.0], wspace=0.30)\n    r2 = outer[1].subgridspec(1, 3, width_ratios=[1.0, 1.0, 1.05], wspace=0.48)\n    r3 = outer[2].subgridspec(1, 2, width_ratios=[1.55, 1.0], wspace=0.42)\n\n    fig.text(0.075, 0.992, \"Two attribution rules, two sides of the same disease\",\n             fontsize=10.2, color=INK, va=\"top\", fontweight=\"bold\")\n    fig.text(0.075, 0.970, \"donor-level properties of the genes each model selected, against unselected genes \"\n             \"matched on the size of their AD shift\", fontsize=6.8, color=INK3, va=\"top\", style=\"italic\")\n\n    # ---- (a) AD effect in bulk and beyond cell composition ----------------------------\n    ax = fig.add_subplot(r1[0])\n    thr_raw = G.loc[G.q_raw < 0.05, \"t_raw\"].abs().min()\n    thr_adj = G.loc[G.q_adj < 0.05, \"t_adj\"].abs().min()\n    un = G[G.group == \"unselected\"]\n    ax.hexbin(un.t_raw, un.t_adj, gridsize=58, cmap=LinearSegmentedColormap.from_list(\n        \"g\", [\"#ffffff\", \"#e3e3ea\", \"#c4c4cf\", \"#9d9dab\"]), mincnt=1, bins=\"log\", linewidths=0, zorder=1)\n    lim_x, lim_y = (-17, 17), (-11, 11)\n    ax.axvspan(-thr_raw, thr_raw, color=PANEL_BG, zorder=0, lw=0)\n    ax.axhspan(-thr_adj, thr_adj, color=PANEL_BG, zorder=0, lw=0)\n    for v in (-thr_raw, thr_raw):\n        ax.axvline(v, color=RULE, lw=0.6, ls=(0, (3, 2)), zorder=1)\n    for v in (-thr_adj, thr_adj):\n        ax.axhline(v, color=RULE, lw=0.6, ls=(0, (3, 2)), zorder=1)\n    ax.plot([-17, 17], [-17, 17], color=INK3, lw=0.6, ls=(0, (1, 2)), zorder=1)\n    for key, lab, col, mk in LISTS:\n        s = G[G.group == key]\n        ax.scatter(s.t_raw, s.t_adj, s=19 if mk != \"D\" else 17, marker=mk, color=col,\n                   edgecolor=SURFACE, linewidth=0.45, zorder=4, label=f\"{lab} ({len(s)})\")\n    lab_genes = [\"VGF\", \"DUSP6\", \"SPRY4\", \"RAF1\", \"ELK1\", \"KDM2A\", \"BDNF\", \"NPTX2\", \"GFAP\", \"RPH3A\",\n                 \"CLDN5\", \"IFITM1\", \"FCGR2C\", \"CD52\", \"LAMA2\", \"VEGFA\", \"PPM1A\", \"ASF1B\", \"GPR17\", \"ACSBG1\"]\n    lg = G.loc[lab_genes]\n    ax.set_xlim(*lim_x); ax.set_ylim(*lim_y)\n    REG = ((\"direct\", thr_raw + 0.5, 10.45, \"left\"), (\"direct\", -thr_raw - 0.5, -10.45, \"right\"),\n           (\"compositional\", 16.6, -0.0, \"right\"), (\"masked\", 0.0, -10.45, \"center\"))\n    for txt, x, y, ha in REG:\n        ax.text(x, y, txt, fontsize=6.4, color=INK3, ha=ha, va=\"center\", style=\"italic\", zorder=2)\n    ax.set_xlabel(\"AD effect in bulk tissue ($t$, donor level)\")\n    ax.set_ylabel(\"AD effect after accounting for cell mix ($t$)\")\n    ax.legend(loc=\"lower right\", fontsize=6.2, handletextpad=0.3, borderaxespad=0.4, markerscale=1.1)\n    sel_ = G[G.group != \"unselected\"]\n    # every selected marker and every region label is an obstacle for the gene labels\n    ox = np.r_[sel_.t_raw.values, [thr_raw + 1.5 + k for k in range(0, 5)], [-thr_raw - 1.5 - k for k in range(0, 5)],\n               [12.5, 13.5, 14.5, 15.5, 16.3], [-0.8, 0.0, 0.8]]\n    oy = np.r_[sel_.t_adj.values, [10.45] * 5, [-10.45] * 5, [0.0] * 5, [-10.45] * 3]\n    repel_avoiding(ax, lg.t_raw.values, lg.t_adj.values, list(lg.index), ox, oy, fontsize=5.6, clearance=1.2)\n    panel(ax, \"a\", \"AD effect in bulk tissue and beyond the cell mix\",\n          \"shaded: not significant ($q\\\\geq0.05$); dotted: no change on adjustment\", dx=-0.115)\n\n    # ---- (b) effect classes per list ------------------------------------------------\n    ax = fig.add_subplot(r1[1])\n    rows = [(\"PLS-DA only\", \"PLS-DA only\", PLS), (\"shared\", \"both models\", ACCENT),\n            (\"CVAE only\", \"CVAE only\", CVAE_N), (\"all genes\", \"all genes\", INK3)]\n    for i, (key, lab, col) in enumerate(rows):\n        e = EC[EC.group == key].set_index(\"effect_class\")\n        left = 0.0\n        for cls, clab, ccol in CLASS:\n            w = 100 * e.at[cls, \"fraction\"]\n            ax.barh(i, w, left=left, height=0.62, color=ccol, edgecolor=SURFACE, linewidth=0.8, zorder=2)\n            if w >= 7:\n                txt = f\"{int(e.at[cls, 'n']):,}\" if key == \"all genes\" else f\"{int(e.at[cls, 'n'])}\"\n                ax.text(left + w / 2, i, txt, ha=\"center\", va=\"center\", fontsize=6.3,\n                        color=SURFACE if cls in (\"direct\", \"masked\") else INK, zorder=3)\n            left += w\n        if key != \"all genes\":\n            exp = 100 * e.at[\"direct\", \"null_matched_fraction\"]\n            ax.plot([exp, exp], [i - 0.40, i + 0.40], color=INK, lw=1.3, zorder=4, solid_capstyle=\"butt\")\n            p = e.at[\"direct\", \"p_vs_matched\"]\n            ax.text(101.5, i, f\"{100 * e.at['direct', 'fraction']:.0f}% direct\\n\"\n                    f\"exp. {exp:.0f}%{' ' + star(p) if star(p) else ''}\",\n                    fontsize=5.9, color=col, va=\"center\", ha=\"left\", linespacing=1.1)\n    ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[1] for r in rows], fontsize=7)\n    for t, (_, _, col) in zip(ax.get_yticklabels(), rows):\n        t.set_color(col)\n    ax.set_ylim(len(rows) - 0.45, -0.6); ax.set_xlim(0, 100)\n    ax.set_xlabel(\"% of genes in the list\")\n    ax.spines[\"left\"].set_visible(False); ax.tick_params(axis=\"y\", length=0)\n    hl = [Rectangle((0, 0), 1, 1, color=c) for _, _, c in CLASS]\n    ax.legend(hl, [l for _, l, _ in CLASS], ncol=4, loc=\"upper left\", bbox_to_anchor=(-0.02, -0.17),\n              fontsize=5.9, handlelength=0.9, columnspacing=0.8, handletextpad=0.35)\n    panel(ax, \"b\", \"How each list's AD effects relate to the cell mix\",\n          \"black tick: direct share expected for genes of the same AD shift\", dx=-0.30)\n\n    # ---- (c) size of the AD shift, (d) share beyond cell mix ------------------------\n    def strip(ax, metric, scale, ylab, clip=None, nullband=True):\n        for i, (key, lab, col, mk) in enumerate(LISTS):\n            v = G.loc[G.group == key, metric].values\n            if clip is not None:\n                v = np.clip(v, *clip)\n            v = v * scale\n            rs = np.random.RandomState(3 + i)\n            ax.scatter(i + rs.uniform(-0.17, 0.17, len(v)), v, s=9, marker=mk, color=col, alpha=0.55,\n                       linewidth=0, zorder=3)\n            med = np.median(v)\n            ax.plot([i - 0.28, i + 0.28], [med, med], color=col, lw=2.0, zorder=4, solid_capstyle=\"butt\")\n            r = SM[(SM.group == key) & (SM.metric == metric)].iloc[0]\n            if nullband:\n                ax.add_patch(Rectangle((i - 0.33, r.null_lo * scale), 0.66, (r.null_hi - r.null_lo) * scale,\n                                       facecolor=tint(INK3, 0.16), edgecolor=\"none\", zorder=1))\n                ax.plot([i - 0.33, i + 0.33], [r.null_matched_median * scale] * 2, color=INK3, lw=0.8,\n                        ls=(0, (2, 1.5)), zorder=2)\n            ax.text(i, 0.985, f\"{med:.2f}\" if scale == 1 else f\"{med:.0f}%\", transform=ax.get_xaxis_transform(),\n                    ha=\"center\", va=\"top\", fontsize=6.4, color=col, fontweight=\"bold\", zorder=6,\n                    bbox=dict(boxstyle=\"round,pad=0.12\", facecolor=SURFACE, edgecolor=\"none\"))\n        ax.set_xticks(range(3)); ax.set_xticklabels([l[1].replace(\" \", \"\\n\", 1) for l in LISTS], fontsize=6.4)\n        for t, l in zip(ax.get_xticklabels(), LISTS):\n            t.set_color(l[2])\n        ax.set_ylabel(ylab); ygrid(ax); ax.set_xlim(-0.55, 2.55)\n\n    ax = fig.add_subplot(r2[0])\n    ax.set_ylim(0, 1.72)\n    strip(ax, \"abs_d\", 1, \"|Cohen's $d$|, AD vs control\", nullband=False)\n    gen_med = G.abs_d.median()\n    ax.axhline(gen_med, color=INK3, lw=0.7, ls=(0, (3, 2)), zorder=1)\n    p = SM[(SM.group == \"PLS-DA only vs CVAE only\") & (SM.metric == \"abs_d\")].p_vs_matched.iloc[0]\n    panel(ax, \"c\", \"Size of the AD shift\", f\"dashed: all genes ({gen_med:.2f}); {pmath(p)}\", dx=-0.32, dy=1.07)\n\n    ax = fig.add_subplot(r2[1])\n    ax.set_ylim(-50, 162)\n    strip(ax, \"effect_retained\", 100, \"% of the AD effect kept\\nafter accounting for cell mix\", clip=(-0.45, 1.45))\n    ax.axhline(100, color=RULE, lw=0.6, zorder=1); ax.axhline(0, color=RULE, lw=0.6, zorder=1)\n    p = SM[(SM.group == \"PLS-DA only vs CVAE only\") & (SM.metric == \"effect_retained\")].p_vs_matched.iloc[0]\n    panel(ax, \"d\", \"Effect beyond the cell mix\",\n          f\"grey: effect-matched genes; {pmath(p)}\", dx=-0.32, dy=1.07)\n\n    # ---- (e) cell-type marker genes ---------------------------------------------------\n    ax = fig.add_subplot(r2[2])\n    CT = [(\"astrocytes\", \"astrocyte\", \"#6d8aa6\"), (\"endothelial\", \"endothelial\", \"#a0566b\"),\n          (\"microglia\", \"microglia\", \"#7a6d9e\"), (\"OPCs\", \"OPC\", \"#5f8f8a\"),\n          (\"neurons\", \"neuron\", \"#3d4f66\"), (\"oligodendrocytes\", \"oligodendrocyte\", \"#b59a6a\")]\n    for i, (key, lab, col, mk) in enumerate(LISTS):\n        r = MK[MK.group == key].iloc[0]\n        n = r.n_genes; left = 0.0\n        for ct, clab, ccol in CT:\n            w = 100 * r[ct] / n\n            if w > 0:\n                ax.bar(i, w, bottom=left, width=0.55, color=ccol, edgecolor=SURFACE, linewidth=0.6, zorder=3)\n                left += w\n        exp = 100 * r.null_matched_fraction\n        ax.plot([i - 0.36, i + 0.36], [exp, exp], color=INK, lw=1.1, zorder=4, solid_capstyle=\"butt\")\n        ax.text(i, left + 0.8, f\"{int(r.n_markers)}/{int(n)}{' ' + star(r.p_vs_matched) if star(r.p_vs_matched) else ''}\",\n                ha=\"center\", va=\"bottom\", fontsize=6.2, color=col, fontweight=\"bold\")\n    ax.set_ylim(0, 33); ax.set_xlim(-0.55, 2.55)\n    ax.set_xticks(range(3)); ax.set_xticklabels([l[1].replace(\" \", \"\\n\", 1) for l in LISTS], fontsize=6.4)\n    for t, l in zip(ax.get_xticklabels(), LISTS):\n        t.set_color(l[2])\n    ax.set_ylabel(\"% of the list that are\\ncell-type marker genes\"); ygrid(ax)\n    used = [c for c in CT if any(MK[MK.group.isin([\"PLS-DA only\", \"shared\", \"CVAE only\"])][c[0]] > 0)]\n    ax.legend([Rectangle((0, 0), 1, 1, color=c) for _, _, c in used], [l for _, l, _ in used],\n              loc=\"upper right\", fontsize=5.6, ncol=2, handlelength=0.8, columnspacing=0.6,\n              handletextpad=0.3, borderaxespad=0.1)\n    panel(ax, \"e\", \"Cell-type marker genes\", \"top-200 published markers; tick: expected\", dx=-0.30, dy=1.07)\n\n    # ---- (f) what each importance score follows genome-wide --------------------------\n    ax = fig.add_subplot(r3[0])\n    PROPS = [(\"abs_d\", \"size of the AD shift, |$d$|\", \"spearman\"),\n             (\"abs_t_adj\", \"AD shift beyond the cell mix\", \"partial_spearman_given_abs_d\"),\n             (\"connectivity\", \"co-expression with the genome\", \"partial_spearman_given_abs_d\"),\n             (\"R2_cells\", \"variance explained by the cell mix\", \"partial_spearman_given_abs_d\"),\n             (\"lnVR\", \"variance ratio, AD / control\", \"partial_spearman_given_abs_d\")]\n    for i, (prop, lab, col_) in enumerate(PROPS):\n        v = AT[(AT.score == \"vip\") & (AT.property == prop)][col_].iloc[0]\n        w = AT[(AT.score == \"ig\") & (AT.property == prop)][col_].iloc[0]\n        ax.plot([v, w], [i, i], color=RULE, lw=1.6, zorder=1, solid_capstyle=\"round\")\n        ax.scatter([v], [i], s=34, marker=\"o\", color=PLS, zorder=3, edgecolor=SURFACE, linewidth=0.6)\n        ax.scatter([w], [i], s=32, marker=\"s\", color=CVAE_N, zorder=3, edgecolor=SURFACE, linewidth=0.6)\n        for val, col, side in ((v, PLS, 1 if v >= w else -1), (w, CVAE_N, 1 if w > v else -1)):\n            ax.text(val + side * 0.035, i, f\"{val:+.2f}\", fontsize=6.1, color=col, va=\"center\",\n                    ha=\"left\" if side > 0 else \"right\", fontweight=\"bold\")\n    ax.axvline(0, color=INK3, lw=0.7, zorder=0)\n    ax.set_yticks(range(len(PROPS))); ax.set_yticklabels([p[1] for p in PROPS], fontsize=6.7)\n    ax.set_ylim(len(PROPS) - 0.5, -0.65); ax.set_xlim(-0.42, 0.62)\n    ax.set_xlabel(\"Spearman $\\\\rho$ with the importance score, all 11,214 genes\")\n    ax.tick_params(axis=\"y\", length=0); ax.spines[\"left\"].set_visible(False); xgrid(ax)\n    ax.legend(handles=[Line2D([], [], marker=\"o\", ls=\"\", color=PLS, markersize=5, label=\"PLS-DA VIP\"),\n                       Line2D([], [], marker=\"s\", ls=\"\", color=CVAE_N, markersize=5, label=\"CVAE $|$IG$|$\")],\n              loc=\"lower right\", fontsize=6.2, ncol=2, columnspacing=0.8)\n    panel(ax, \"f\", \"Genome-wide: what each importance score follows\",\n          \"rows 2--5 partial $\\\\rho$ holding |$d$| fixed\", dx=-0.47, dy=1.03)\n\n    # ---- (g) co-expression within each list ------------------------------------------\n    ax = fig.add_subplot(r3[1])\n    for i, (key, lab, col, mk) in enumerate(LISTS):\n        nv = CN[CN.group == key].coherence.values\n        vp = ax.violinplot([nv], positions=[i], widths=0.62, showextrema=False)\n        for b in vp[\"bodies\"]:\n            b.set_facecolor(tint(INK3, 0.22)); b.set_edgecolor(\"none\"); b.set_alpha(1)\n        obs = CO[CO.group == key].coherence.iloc[0]\n        ax.scatter([i], [obs], s=46, marker=mk, color=col, zorder=4, edgecolor=SURFACE, linewidth=0.7)\n        ax.text(i + 0.12, obs, f\"{obs:.2f}\", fontsize=6.3, color=col, va=\"center\", ha=\"left\", fontweight=\"bold\")\n    p = rd(V16, \"gene_sides_coherence_diff.csv\").permutation_p.iloc[0]\n    ax.set_xticks(range(3)); ax.set_xticklabels([l[1].replace(\" \", \"\\n\", 1) for l in LISTS], fontsize=6.4)\n    for t, l in zip(ax.get_xticklabels(), LISTS):\n        t.set_color(l[2])\n    ax.set_ylabel(\"mean |$r$| between genes,\\ndiagnosis effect removed\"); ygrid(ax)\n    ax.set_xlim(-0.55, 2.6)\n    panel(ax, \"g\", \"Co-expression within each list\",\n          f\"grey: effect-matched sets; PLS-DA vs CVAE only {pmath(p)}\", dx=-0.30, dy=1.03)\n\n    save(fig, \"figV16_gene_sides\")\n\n\nfig_gene_sides()\n"
open('/kaggle/working/scripts/gene_sides_figure.py', 'w').write(SRC_GENE_SIDES_FIGURE)
print('wrote gene_sides_figure.py', len(SRC_GENE_SIDES_FIGURE), 'chars')


In [ ]:
import runpy


## Main figures (merged cohorts)


In [ ]:
runpy.run_path('/kaggle/working/scripts/merged_figures.py', run_name='__main__')


## External validation at a glance (core figure)


In [ ]:
runpy.run_path('/kaggle/working/scripts/core_validation_figure.py', run_name='__main__')


## Supplementary external-validation figures


In [ ]:
runpy.run_path('/kaggle/working/scripts/validation_figures.py', run_name='__main__')


## Robustness to age (V15ADage)


In [ ]:
runpy.run_path('/kaggle/working/scripts/age_figures.py', run_name='__main__')


## Why the two models see different sides of AD (V16ADgenesides)


In [ ]:
runpy.run_path('/kaggle/working/scripts/gene_sides_figure.py', run_name='__main__')


## Output manifest


In [ ]:
import pandas as pd
rows=[(f, os.path.getsize(os.path.join('/kaggle/working/figures', f))) for f in sorted(os.listdir('/kaggle/working/figures'))]
pd.DataFrame(rows, columns=['file','bytes']).to_csv('/kaggle/working/figures_manifest.csv', index=False)
print(len(rows), 'files'); pd.DataFrame(rows, columns=['file','bytes'])
